<a href="https://colab.research.google.com/github/jkshakthi999/-DeOldify-Pro-Enhanced-Colorization-main-containing-all-the-tasks/blob/main/FinalAIProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os, sys, torch

print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Runtime > Change runtime type > T4 GPU is strongly recommended.")

!pip install -q gradio transformers scikit-image ffmpeg-python yt-dlp

# clone DeOldify (fresh copy)
!rm -rf /content/DeOldify
%cd /content
!git clone -q https://github.com/jantic/DeOldify.git DeOldify
%cd /content/DeOldify

# DeOldify's pinned requirements may fail to build on new Colab images (Pillow); the preinstalled packages are used then
!pip install -q -r requirements.txt || echo "(requirements install failed - continuing with Colab's preinstalled packages)"

sys.path.append("/content/DeOldify")
import deoldify
print("DeOldify imported successfully")

PyTorch: 2.14.1+cu130
GPU available: True
GPU: Tesla T4
/content
/content/DeOldify
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
ERROR: Failed to build 'Pillow' when getting requirements to build wheel
(requirements install failed - continuing with Colab's preinstalled packages)
DeOldify imported successfully


In [2]:
from huggingface_hub import hf_hub_download

MODELS_DIR = "/content/DeOldify/models"
os.makedirs(MODELS_DIR, exist_ok=True)

def download_model(filename: str) -> None:
    dest = os.path.join(MODELS_DIR, filename)

    if os.path.exists(dest) and os.path.getsize(dest) > 100_000_000:
        print(f"{filename} already cached ({os.path.getsize(dest)/1e6:.0f} MB)")
        return

    for repo_id in [
        "thookham/DeOldify",
        "databuzzword/deoldify-stable" if "Stable" in filename
        else "databuzzword/deoldify-artistic"
    ]:
        try:
            hf_hub_download(
                repo_id=repo_id,
                filename=filename,
                local_dir=MODELS_DIR
            )

            print(f"Downloaded {filename} from {repo_id}")
            return

        except Exception as exc:
            print(f"Failed to download from {repo_id}: {exc}")

    print(f"Could not download {filename}")


download_model("ColorizeArtistic_gen.pth")
download_model("ColorizeStable_gen.pth")

print("\nVerification")

for name in ["ColorizeArtistic_gen.pth", "ColorizeStable_gen.pth"]:
    path = os.path.join(MODELS_DIR, name)
    size = os.path.getsize(path) / 1e6 if os.path.exists(path) else 0
    status = "Downloaded" if size > 100 else "Missing"
    print(f"{status} {name}: {size:.1f} MB")

INFO:httpx:HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"


HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: HEAD https://huggingface.co/thookham/DeOldify/resolve/main/ColorizeArtistic_gen.pth "HTTP/1.1 401 Unauthorized"


HTTP Request: HEAD https://huggingface.co/thookham/DeOldify/resolve/main/ColorizeArtistic_gen.pth "HTTP/1.1 401 Unauthorized"
Failed to download from thookham/DeOldify: 401 Client Error. (Request ID: Root=1-6b001ea2-3c306544483af34a31249a27;fe3f64ef-b04d-41eb-be9a-99faefa508b4)

Repository Not Found for url: https://huggingface.co/thookham/DeOldify/resolve/main/ColorizeArtistic_gen.pth.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated and your token has the required permissions.
For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Invalid username or password.


INFO:httpx:HTTP Request: HEAD https://huggingface.co/databuzzword/deoldify-artistic/resolve/main/ColorizeArtistic_gen.pth "HTTP/1.1 302 Found"


HTTP Request: HEAD https://huggingface.co/databuzzword/deoldify-artistic/resolve/main/ColorizeArtistic_gen.pth "HTTP/1.1 302 Found"


... (9 outputs left out)

HTTP Request: GET https://huggingface.co/api/models/databuzzword/deoldify-stable/xet-read-token/f59bfa448a30d1e2b1b43b284428b720aa1d371f "HTTP/1.1 200 OK"


ColorizeStable_gen.pth: reconstructing file:   0%|          |  0.00B /  874MB            

ColorizeStable_gen.pth: downloading bytes:           |  0.00B            

Downloaded ColorizeStable_gen.pth from databuzzword/deoldify-stable

Verification
Downloaded ColorizeArtistic_gen.pth: 255.1 MB
Downloaded ColorizeStable_gen.pth: 874.1 MB


In [3]:
import sys
sys.path.insert(0, "/content/DeOldify")
import os, time, traceback
import cv2
import torch
import functools
import numpy as np
import gradio as gr
import warnings
warnings. filterwarnings("ignore")
from pathlib import Path
from PIL import Image, ImageEnhance, ImageFilter, ImageOps
from scipy.ndimage import gaussian_filter
from skimage.metrics import structural_similarity as ssim
from skimage import exposure

if torch. cuda. is_available():
  print(f" GPU available: {torch.cuda.get_device_name(0)}")
else:
  print("No GPU detected. Please change runtime type to T4 GPU.")
print("import completed successfully")

INFO:numexpr.utils:NumExpr defaulting to 2 threads.


NumExpr defaulting to 2 threads.
 GPU available: Tesla T4
import completed successfully


In [4]:
import torch
import functools

from deoldify import device
from deoldify.device_id import DeviceId
from deoldify.visualize import get_image_colorizer

device.set(device=DeviceId.GPU0)

_orig_load = torch.load

torch.load = functools.partial(
    torch.load,
    weights_only=False
)

try:
    artistic_colorizer = get_image_colorizer(
        artistic=True
    )

    stable_colorizer = get_image_colorizer(
        artistic=False
    )

    print("Both colorizers loaded successfully!")

except Exception as e:
    print(f"Error loading colorizer: {e}")

finally:
    torch.load = _orig_load

Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:01<00:00, 84.5MB/s]


Downloading: "https://download.pytorch.org/models/resnet101-63fe2227.pth" to /root/.cache/torch/hub/checkpoints/resnet101-63fe2227.pth


100%|██████████| 171M/171M [00:01<00:00, 178MB/s]


Both colorizers loaded successfully!


In [5]:
class ImageProcessor:
    """Low-level image processing primitives."""

    # --------------------------------------------------
    # PREPROCESSING
    # --------------------------------------------------

    @staticmethod
    def denoise(img_np: np.ndarray) -> np.ndarray:
        """Non-local means denoising before colorization."""

        return cv2.fastNlMeansDenoisingColored(
            cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR),
            None,
            h=6,
            hColor=6,
            templateWindowSize=7,
            searchWindowSize=21
        )[:, :, ::-1]

    @staticmethod
    def adaptive_clahe(
        gray: np.ndarray,
        clip: float = 2.0,
        grid: int = 8
    ) -> np.ndarray:
        """
        Apply CLAHE only to the luminance channel (LAB)
        to boost micro-contrast without blowing out highlights.
        """

        lab = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
        lab = cv2.cvtColor(lab, cv2.COLOR_BGR2LAB)

        l, a, b = cv2.split(lab)

        clahe = cv2.createCLAHE(
            clipLimit=clip,
            tileGridSize=(grid, grid)
        )

        l = clahe.apply(l)

        enhanced = cv2.merge([l, a, b])

        return cv2.cvtColor(
            enhanced,
            cv2.COLOR_LAB2BGR
        )[:, :, ::-1]

    # --------------------------------------------------
    # LAB BLEND
    # --------------------------------------------------

    @staticmethod
    def lab_blend(
        original_rgb: np.ndarray,
        colorized_rgb: np.ndarray,
        luminance_weight: float = 0.92
    ) -> np.ndarray:
        """
        Replace colorized luminance with the original luminance
        while preserving AI-generated color information.
        """

        orig_lab = cv2.cvtColor(
            original_rgb,
            cv2.COLOR_RGB2LAB
        ).astype(np.float32)

        color_lab = cv2.cvtColor(
            colorized_rgb,
            cv2.COLOR_RGB2LAB
        ).astype(np.float32)

        l_blended = (
            luminance_weight * orig_lab[:, :, 0]
            + (1.0 - luminance_weight) * color_lab[:, :, 0]
        )

        merged = np.stack(
            [
                l_blended,
                color_lab[:, :, 1],
                color_lab[:, :, 2]
            ],
            axis=2
        )

        merged = np.clip(
            merged,
            0,
            255
        ).astype(np.uint8)

        return cv2.cvtColor(
            merged,
            cv2.COLOR_LAB2RGB
        )

    # --------------------------------------------------
    # ENSEMBLE BLEND
    # --------------------------------------------------

    @staticmethod
    def ensemble_blend(
        artistic_rgb: np.ndarray,
        stable_rgb: np.ndarray,
        artistic_weight: float = 0.55
    ) -> np.ndarray:
        """
        Blend artistic and stable outputs in LAB space.
        """

        art_lab = cv2.cvtColor(
            artistic_rgb,
            cv2.COLOR_RGB2LAB
        ).astype(np.float32)

        stb_lab = cv2.cvtColor(
            stable_rgb,
            cv2.COLOR_RGB2LAB
        ).astype(np.float32)

        w = artistic_weight

        blended_lab = (
            w * art_lab +
            (1 - w) * stb_lab
        ).clip(0, 255).astype(np.uint8)

        return cv2.cvtColor(
            blended_lab,
            cv2.COLOR_LAB2RGB
        )

    # --------------------------------------------------
    # FACE DETECTION
    # --------------------------------------------------

    @staticmethod
    def detect_faces(rgb_img: np.ndarray):
        """
        Returns list of (x, y, w, h) face bounding boxes.
        """

        gray = cv2.cvtColor(
            rgb_img,
            cv2.COLOR_RGB2GRAY
        )

        cascade_path = (
            cv2.data.haarcascades +
            "haarcascade_frontalface_default.xml"
        )

        detector = cv2.CascadeClassifier(cascade_path)

        if detector.empty():
            return []

        faces = detector.detectMultiScale(
            gray,
            scaleFactor=1.1,
            minNeighbors=5,
            minSize=(30, 30)
        )

        return faces if len(faces) else []

    # --------------------------------------------------
    # SKIN TONE CORRECTION
    # --------------------------------------------------

    @staticmethod
    def correct_skin_tones(
        rgb_img: np.ndarray,
        faces: list,
        hue_shift: float = -3.0,
        sat_scale: float = 0.92
    ) -> np.ndarray:
        """
        Targeted YCrCb-space correction on skin locus pixels
        within detected face regions.
        """

        result = rgb_img.copy()

        h_img, w_img = rgb_img.shape[:2]

        # Build skin mask

        ycrcb = cv2.cvtColor(
            rgb_img,
            cv2.COLOR_RGB2YCrCb
        )

        cr = ycrcb[:, :, 1]
        cb = ycrcb[:, :, 2]

        skin_mask = (
            (cr >= 133) & (cr <= 173) &
            (cb >= 77) & (cb <= 127)
        ).astype(np.uint8)

        # If faces are detected, restrict to expanded face ROIs

        if len(faces) > 0:

            face_region = np.zeros(
                (h_img, w_img),
                dtype=np.uint8
            )

            for (x, y, w, fh) in faces:

                pad = int(
                    0.25 * max(w, fh)
                )

                x0 = max(0, x - pad)
                y0 = max(0, y - pad)

                x1 = min(
                    w_img,
                    x + w + pad
                )

                y1 = min(
                    h_img,
                    y + fh + pad
                )

                face_region[y0:y1, x0:x1] = 1

            skin_mask = (
                skin_mask & face_region
            ).astype(np.uint8)

        if skin_mask.sum() == 0:
            return result

        hsv = cv2.cvtColor(
            rgb_img,
            cv2.COLOR_RGB2HSV
        ).astype(np.float32)

        hsv[:, :, 0] = np.where(
            skin_mask == 1,
            np.clip(
                hsv[:, :, 0] + hue_shift,
                0,
                179
            ),
            hsv[:, :, 0]
        )

        hsv[:, :, 1] = np.where(
            skin_mask == 1,
            np.clip(
                hsv[:, :, 1] * sat_scale,
                0,
                255
            ),
            hsv[:, :, 1]
        )

        corrected = cv2.cvtColor(
            hsv.clip(0, 255).astype(np.uint8),
            cv2.COLOR_HSV2RGB
        )

        # Feather the correction for smooth transitions

        mask_f = cv2.GaussianBlur(
            skin_mask.astype(np.float32) * 255,
            (21, 21),
            0
        )[:, :, np.newaxis] / 255.0

        return (
            corrected * mask_f +
            result * (1 - mask_f)
        ).astype(np.uint8)

    # --------------------------------------------------
    # COLOR GRADING
    # --------------------------------------------------

    @staticmethod
    def color_grade(
        rgb_img: np.ndarray,
        temperature: float = 0.0,
        vibrance: float = 1.0,
        shadows_lift: float = 0.0
    ) -> np.ndarray:
        """
        temperature:
            positive -> warmer (add yellow-red)
            negative -> cooler

        vibrance:
            boost muted colours more than already-saturated ones

        shadows_lift:
            gently lift blacks to recover crushed shadow detail
        """

        img = rgb_img.astype(np.float32)

        # Temperature (push R up / B down for warm; reverse for cool)

        img[:, :, 0] = np.clip(
            img[:, :, 0] + temperature * 8,
            0,
            255
        )

        img[:, :, 2] = np.clip(
            img[:, :, 2] - temperature * 8,
            0,
            255
        )

        # Vibrance: boost saturation selectively on low-saturation pixels

        if vibrance != 1.0:

            hsv = cv2.cvtColor(
                img.clip(0, 255).astype(np.uint8),
                cv2.COLOR_RGB2HSV
            ).astype(np.float32)

            sat = hsv[:, :, 1] / 255.0

            boost = (
                1.0 + (1.0 - sat) * (vibrance - 1.0)
            )

            hsv[:, :, 1] = np.clip(
                hsv[:, :, 1] * boost,
                0,
                255
            )

            img = cv2.cvtColor(
                hsv.astype(np.uint8),
                cv2.COLOR_HSV2RGB
            ).astype(np.float32)

        # Shadow lift (S-curve toe)

        if shadows_lift != 0.0:

            lut = np.arange(
                256,
                dtype=np.float32
            )

            lut = (
                lut +
                shadows_lift * 30 *
                (1.0 - lut / 255.0) ** 2
            )

            lut = np.clip(
                lut,
                0,
                255
            ).astype(np.uint8)

            img = cv2.LUT(
                img.clip(0, 255).astype(np.uint8),
                lut
            ).astype(np.float32)

        return img.clip(
            0,
            255
        ).astype(np.uint8)

    # --------------------------------------------------
    # DETAIL ENHANCEMENT
    # --------------------------------------------------

    @staticmethod
    def unsharp_mask(
        rgb_img: np.ndarray,
        sigma: float = 1.0,
        strength: float = 0.5
    ) -> np.ndarray:
        """
        Gaussian-based sharpening: enhances edges without halos.
        Only applied on luminance to avoid colour fringing.
        """

        lab = cv2.cvtColor(
            rgb_img,
            cv2.COLOR_RGB2LAB
        ).astype(np.float32)

        l = lab[:, :, 0]

        blurred = gaussian_filter(
            l,
            sigma=sigma
        )

        sharpened = l + strength * (
            l - blurred
        )

        lab[:, :, 0] = np.clip(
            sharpened,
            0,
            255
        )

        return cv2.cvtColor(
            lab.astype(np.uint8),
            cv2.COLOR_LAB2RGB
        )

    # --------------------------------------------------
    # UPSCALE INPUT FOR BETTER DETAIL RECOVERY
    # --------------------------------------------------

    @staticmethod
    def upscale_for_colorization(
        pil_img: Image.Image,
        max_side: int = 768
    ) -> Image.Image:
        """
        Lanczos-upscale small images so DeOldify works
        on more pixels, recovering finer colour transitions.
        """

        w, h = pil_img.size

        if max(w, h) >= max_side:
            return pil_img

        scale = max_side / max(w, h)

        return pil_img.resize(
            (
                int(w * scale),
                int(h * scale)
            ),
            Image.LANCZOS
        )

In [6]:
class ColorQualityMetrics:

    @staticmethod
    def saturation_score(
        img: np.ndarray
    ) -> float:
        """
        Mean HSV saturation, 0–1.
        """

        hsv = cv2.cvtColor(
            img,
            cv2.COLOR_RGB2HSV
        ).astype(np.float32)

        return float(
            hsv[:, :, 1].mean() / 255.0
        )

    @staticmethod
    def colorfulness_score(
        img: np.ndarray
    ) -> float:
        """
        Hasler & Süsstrunk (2003) colorfulness metric.
        """

        R, G, B = [
            img[:, :, i].astype(np.float32)
            for i in range(3)
        ]

        rg = R - G

        yb = (
            0.5 * (R + G)
            - B
        )

        return float(
            np.sqrt(
                rg.std() ** 2 +
                yb.std() ** 2
            )
            +
            0.3 * np.sqrt(
                rg.mean() ** 2 +
                yb.mean() ** 2
            )
        )

    @staticmethod
    def ssim_score(
        original: np.ndarray,
        colorized: np.ndarray
    ) -> float:
        """
        Structural similarity
        (luminance consistency).
        """

        if original.shape != colorized.shape:

            colorized = cv2.resize(
                colorized,
                (
                    original.shape[1],
                    original.shape[0]
                )
            )

        return float(
            ssim(
                original,
                colorized,
                channel_axis=2,
                data_range=255
            )
        )

    @staticmethod
    def edge_preservation_score(
        gray: np.ndarray,
        colorized: np.ndarray
    ) -> float:
        """
        Canny edge IoU between grayscale input and colorized output.
        """

        gray_edges = cv2.Canny(
            gray,
            80,
            180
        )

        color_gray = cv2.cvtColor(
            colorized,
            cv2.COLOR_RGB2GRAY
        )

        color_edges = cv2.Canny(
            color_gray,
            80,
            180
        )

        inter = np.logical_and(
            gray_edges > 0,
            color_edges > 0
        ).sum()

        union = np.logical_or(
            gray_edges > 0,
            color_edges > 0
        ).sum()

        return float(
            inter / (union + 1e-8)
        )

    @staticmethod
    def naturalness_score(
        img: np.ndarray
    ) -> float:
        """
        Skin-naturalness heuristic: fraction of pixels whose
        Cr (YCrCb) falls in the human skin locus [133, 173].
        High values indicate warm skin tones.
        """

        ycrcb = cv2.cvtColor(
            img,
            cv2.COLOR_RGB2YCrCb
        )

        cr = ycrcb[:, :, 1].astype(
            np.float32
        )

        skin_mask = (
            (cr >= 133) &
            (cr <= 173)
        )

        return float(
            skin_mask.mean()
        )

    @classmethod
    def full_report(
        cls,
        original_np: np.ndarray,
        colorized_np: np.ndarray
    ) -> dict:
        """
        Generate a complete image quality metrics report.
        """

        gray_1ch = cv2.cvtColor(
            original_np,
            cv2.COLOR_RGB2GRAY
        )

        return {
            "Colorfulness":
                round(
                    cls.colorfulness_score(
                        colorized_np
                    ),
                    2
                ),

            "Saturation":
                round(
                    cls.saturation_score(
                        colorized_np
                    ),
                    4
                ),

            "SSIM":
                round(
                    cls.ssim_score(
                        original_np,
                        colorized_np
                    ),
                    4
                ),

            "Edge Preservation":
                round(
                    cls.edge_preservation_score(
                        gray_1ch,
                        colorized_np
                    ),
                    4
                ),

            "Skin Naturalness":
                round(
                    cls.naturalness_score(
                        colorized_np
                    ),
                    4
                )
        }

In [7]:
def run_colorizer(
    colorizer,
    pil_img: Image.Image,
    render_factor: int
) -> np.ndarray:
    """
    Run a single DeOldify model and return RGB numpy array.
    """

    tmp = "/tmp/deoldify_input.jpg"

    pil_img.save(
        tmp,
        quality=95
    )

    result = colorizer.get_transformed_image(
        tmp,
        render_factor=render_factor,
        watermarked=False
    )

    return np.array(
        result.resize(
            pil_img.size,
            Image.LANCZOS
        )
    )


def colorize_image(
    input_image: np.ndarray,
    model_mode: str,
    render_factor: int,
    temperature: float,
    vibrance: float,
    shadows_lift: float,
    do_denoise: bool,
    do_sharpen: bool,
    sharpen_str: float
) -> tuple:
    """
    Full enhanced pipeline:

    1. Validate & prepare input
    2. (Optional) denoise
    3. Upscale small images
    4. Run AI model(s)
    5. Ensemble blend (if Ensemble mode)
    6. LAB luminance restoration
    7. Skin-tone face correction
    8. Color grading (temperature / vibrance / shadows)
    9. (Optional) unsharp mask
    10. Quality metrics
    """

    if input_image is None:
        return None, None, "❌ Please upload an image."

    proc = ImageProcessor()
    metrics_engine = ColorQualityMetrics()

    try:

        # --- 1. Prepare

        if input_image.ndim == 2:
            input_image = cv2.cvtColor(
                input_image,
                cv2.COLOR_GRAY2RGB
            )

        orig_np = input_image.astype("uint8")

        # --- 2. Optional Denoising

        prep_np = (
            proc.denoise(orig_np)
            if do_denoise
            else orig_np
        )

        orig_pil = Image.fromarray(
            prep_np
        ).convert("RGB")

        # --- 3. Upscale for better detail capture

        working_pil = proc.upscale_for_colorization(
            orig_pil,
            max_side=800
        )

        # --- 4. AI Colorization

        if model_mode == "Ensemble (Best Quality)":

            art_np = run_colorizer(
                artistic_colorizer,
                working_pil,
                render_factor
            )

            stb_np = run_colorizer(
                stable_colorizer,
                working_pil,
                render_factor
            )

            ai_np = proc.ensemble_blend(
                art_np,
                stb_np,
                artistic_weight=0.55
            )

        elif model_mode == "Artistic (Portraits & Faces)":

            ai_np = run_colorizer(
                artistic_colorizer,
                working_pil,
                render_factor
            )

        else:  # Stable

            ai_np = run_colorizer(
                stable_colorizer,
                working_pil,
                render_factor
            )

        # Resize back to original resolution

        ai_np = np.array(
            Image.fromarray(ai_np).resize(
                orig_pil.size,
                Image.LANCZOS
            )
        )

        # --- 5. LAB Luminance Restoration
        # Keeps the AI's vibrant chroma, restores original tonal range.

        fused_np = proc.lab_blend(
            orig_np,
            ai_np,
            luminance_weight=0.90
        )

        # --- 6. Face-Aware Skin Tone Correction

        faces = proc.detect_faces(
            fused_np
        )

        fused_np = proc.correct_skin_tones(
            fused_np,
            faces,
            hue_shift=-3.5 if len(faces) > 0 else 0.0,
            sat_scale=0.90 if len(faces) > 0 else 1.0
        )

        # --- 7. Color Grading

        fused_np = proc.color_grade(
            fused_np,
            temperature=temperature,
            vibrance=vibrance,
            shadows_lift=shadows_lift
        )

        # --- 8. Unsharp Mask (luminance only)

        if do_sharpen:

            fused_np = proc.unsharp_mask(
                fused_np,
                sigma=0.8,
                strength=sharpen_str
            )

        # --- 9. Quality Metrics

        metrics = metrics_engine.full_report(
            orig_np,
            fused_np
        )

        face_str = (
            f" | Faces detected: {len(faces)}"
            if len(faces)
            else ""
        )

        metrics_text = (
            f"✅ Pipeline complete{face_str}\n"
            f"────────────────────────\n"
            f"🎨 Colorfulness      : {metrics['Colorfulness']}\n"
            f"🌈 Saturation        : {metrics['Saturation']:.4f}\n"
            f"📐 SSIM              : {metrics['SSIM']:.4f}\n"
            f"🖼️ Edge Preservation : {metrics['Edge Preservation']:.4f}\n"
            f"🧑 Skin Naturalness  : {metrics['Skin Naturalness']:.4f}\n"
        )

        return fused_np, orig_np, metrics_text

    except Exception as exc:

        import traceback

        return (
            None,
            None,
            f"❌ Error:\n{traceback.format_exc()}"
        )

In [8]:
CSS = """
#header {
    text-align: center;
    margin-bottom: 8px;
}

#metrics_box {
    font-family: monospace;
    font-size: 0.85rem;
}

#compare_label {
    font-size: 0.75rem;
    color: #888;
    text-align: center;
}

.primary-btn {
    font-weight: 700;
    font-size: 1rem !important;
}
"""

def build_main_ui():

    gr.Markdown(
        """
        # 🎨 DeOldify Pro – Enhanced Colorization Pipeline

        **Ensemble blending • LAB luminance restoration • Face-aware skin correction • Color grading**
        """,
        elem_id="header"
    )

    with gr.Row():

        # LEFT PANEL

        with gr.Column(scale=1, min_width=340):

            input_image = gr.Image(
                label="🖼 Upload B&W Photo",
                type="numpy",
                height=260
            )

            model_mode = gr.Radio(
                choices=[
                    "Ensemble (Best Quality)",
                    "Artistic (Portraits & Faces)",
                    "Stable (Objects & Architecture)"
                ],
                value="Ensemble (Best Quality)",
                label="🤖 Model Mode"
            )

            render_factor = gr.Slider(
                minimum=10,
                maximum=45,
                value=35,
                step=1,
                label="🎚 Render Factor (higher = more colour detail, slower)"
            )

            gr.Markdown("### 🎨 Color Grading")

            with gr.Row():

                temperature = gr.Slider(
                    -3.0,
                    3.0,
                    value=0.0,
                    step=0.1,
                    label="🌡 Temperature (warm ↔ cool)"
                )

                vibrance = gr.Slider(
                    0.5,
                    2.0,
                    value=1.15,
                    step=0.05,
                    label="✨ Vibrance"
                )

            shadows_lift = gr.Slider(
                0.0,
                1.0,
                value=0.15,
                step=0.05,
                label="🌑 Shadow Lift (recover crushed blacks)"
            )

            gr.Markdown("### ⚙ Processing Options")

            with gr.Row():

                do_denoise = gr.Checkbox(
                    value=True,
                    label="🔇 Denoise before AI"
                )

                do_sharpen = gr.Checkbox(
                    value=True,
                    label="✨ Sharpen output"
                )

            sharpen_str = gr.Slider(
                0.1,
                1.0,
                value=0.45,
                step=0.05,
                label="🔪 Sharpening Strength"
            )

            colorize_btn = gr.Button(
                "🚀 Colorize!",
                variant="primary",
                elem_classes="primary-btn"
            )

        # RIGHT PANEL

        with gr.Column(scale=2):

            with gr.Tabs():

                with gr.Tab("🌈 Colorized Result"):

                    color_output = gr.Image(
                        label="Colorized Output",
                        type="numpy",
                        height=400
                    )

                with gr.Tab("🪞 Side-by-Side Comparison"):

                    gr.Markdown(
                        "**Original (left) | Colorized (right)**",
                        elem_id="compare_label"
                    )

                    with gr.Row():

                        orig_preview = gr.Image(
                            label="Original",
                            height=380,
                            type="numpy"
                        )

                        color_preview2 = gr.Image(
                            label="Colorized",
                            height=380,
                            type="numpy"
                        )

            status_msg = gr.Textbox(
                label="📊 Quality Metrics & Status",
                interactive=False,
                lines=9,
                elem_id="metrics_box"
            )

    # ---------------------- WIRING ----------------------

    def run_and_split(
        inp,
        mode,
        rf,
        temp,
        vib,
        shadow,
        denoise,
        sharpen,
        sharp_str
    ):

        colorized, original, metrics = colorize_image(
            inp,
            mode,
            rf,
            temp,
            vib,
            shadow,
            denoise,
            sharpen,
            sharp_str
        )

        return (
            colorized,
            original,
            colorized,
            metrics
        )

    colorize_btn.click(
        fn=run_and_split,
        inputs=[
            input_image,
            model_mode,
            render_factor,
            temperature,
            vibrance,
            shadows_lift,
            do_denoise,
            do_sharpen,
            sharpen_str
        ],
        outputs=[
            color_output,
            orig_preview,
            color_preview2,
            status_msg
        ]
    )

In [9]:
from IPython.core.magic import register_cell_magic

MODULES = {}      # task name -> its own namespace (a dict)


@register_cell_magic("ext")
def _ext(line, cell):
    # runs the cell inside the namespace of the given task (isolated from the main project and from the other tasks)
    name = line.strip() or "default"
    ns = MODULES.setdefault(name, {"__name__": "ext_" + name})
    exec(compile(cell, "<ext:" + name + ">", "exec"), ns)

print("Extension loader ready.")

Extension loader ready.


In [10]:
%%ext task1
import os, re, io, time, shutil, subprocess, tempfile, traceback
import numpy as np
import torch, torch.nn.functional as F
import cv2
from PIL import Image
from skimage import color
import matplotlib.colors as mcolors
import gradio as gr

device = "cuda" if torch.cuda.is_available() else "cpu"

# ------------------------- object groups -------------------------
GROUPS = ["Vehicles", "Trees & plants", "Grass & fields", "Buildings", "Sky",
          "Roads & ground", "Water", "People", "Mountains & rocks", "Other objects"]
OTHER = len(GROUPS) - 1
# segmentation class names (ADE20K and Cityscapes) that belong to each group; everything else = "Other objects"
GROUP_NAMES = {
    "Vehicles": {"car", "bus", "truck", "van", "boat", "ship", "airplane", "minibike", "bicycle", "motorcycle", "train", "motorbike"},
    "Trees & plants": {"tree", "plant", "palm", "flower", "vegetation"},
    "Grass & fields": {"grass", "field", "terrain"},
    "Buildings": {"building", "house", "skyscraper", "tower", "hovel", "wall", "column", "bridge"},
    "Sky": {"sky"},
    "Roads & ground": {"road", "sidewalk", "path", "earth", "land", "floor", "dirt track", "runway", "sand"},
    "Water": {"water", "sea", "river", "lake", "swimming pool", "waterfall", "fountain"},
    "People": {"person", "rider"},
    "Mountains & rocks": {"mountain", "hill", "rock"},
}
# colours used only to draw the object map
DISPLAY_COLORS = [(230, 60, 60), (30, 140, 50), (140, 220, 60), (200, 150, 90), (70, 130, 235),
                  (130, 130, 140), (40, 190, 220), (245, 160, 200), (150, 100, 60), (180, 180, 60)]

# ------------------------- colour schemes (one colour per group, same order as GROUPS) -------------------------
SCHEMES = {
    "Natural": ["#B03A3A", "#3E7B3A", "#6FAF40", "#B79B7E", "#7FB6E6", "#7C7C82", "#3B88B8", "#D8A383", "#8A7A6A", "#A89480"],
    "Vivid":   ["#E02020", "#1E9A2E", "#4CD03A", "#E0A060", "#2A8CFF", "#6A6AA0", "#10A0D8", "#F0A070", "#A06A40", "#D08050"],
    "Vintage": ["#9C5B3E", "#6F6B3E", "#8D8A4A", "#B38B5D", "#B9B8A0", "#8B7D6B", "#6D8590", "#C99572", "#85705A", "#A38A6D"],
    "Pastel":  ["#E8A0A8", "#9FD3A0", "#C5E8A0", "#F0D8B8", "#B8DCF5", "#C8C8D0", "#A0D8E8", "#F5D0C0", "#D0C0B0", "#E0D0C0"],
    "Cool night": ["#4A5BA8", "#2D5A5A", "#3A7A6A", "#6A7AA8", "#3A4A8A", "#5A6078", "#2A5A9A", "#9A8AA8", "#4A5070", "#6A7090"],
}
CUSTOM = "Custom (edit the colours)"

# ------------------------- segmentation models (downloaded automatically from Hugging Face) -------------------------
SEG_MODELS = {
    "General scenes - fast (SegFormer-B0, ADE20K)": "nvidia/segformer-b0-finetuned-ade-512-512",
    "General scenes - accurate (SegFormer-B2, ADE20K)": "nvidia/segformer-b2-finetuned-ade-512-512",
    "Street scenes (SegFormer-B0, Cityscapes)": "nvidia/segformer-b0-finetuned-cityscapes-1024-1024",
}
DEFAULT_MODEL = list(SEG_MODELS)[0]

VIEWS = ["Colorized", "Original | Colorized", "Original | Colorized | Object map"]
print("Device:", device)
print("Object groups:", ", ".join(GROUPS))

Device: cuda
Object groups: Vehicles, Trees & plants, Grass & fields, Buildings, Sky, Roads & ground, Water, People, Mountains & rocks, Other objects


In [11]:
%%ext task1
# ---------------- colour helpers ----------------
def parse_color(value):
    # colour name, hex code or rgb(...) text -> (r, g, b) integers, or None if it is not understood
    if value is None:
        return None
    s = str(value).strip().lower()
    if not s:
        return None
    m = re.match(r"rgba?\(\s*([\d.]+)\s*,\s*([\d.]+)\s*,\s*([\d.]+)", s)
    if m:
        return tuple(int(round(min(255.0, float(m.group(i))))) for i in (1, 2, 3))
    if re.fullmatch(r"[0-9a-f]{6}", s) or re.fullmatch(r"[0-9a-f]{3}", s):
        s = "#" + s
    try:
        r, g, b = mcolors.to_rgb(s.replace(" ", ""))
        return (int(round(r * 255)), int(round(g * 255)), int(round(b * 255)))
    except Exception:
        return None


def palette_to_ab(colors):
    # the 10 chosen colours -> tensor (10 x 2) of Lab (a, b) values; the brightness always comes from the video frame
    ab = []
    for g, c in zip(GROUPS, colors):
        rgb = parse_color(c)
        if rgb is None:
            raise gr.Error(f"The colour for '{g}' is not valid: {c}")
        lab = color.rgb2lab(np.array([[rgb]], dtype=np.float64) / 255.0)[0, 0]
        ab.append(lab[1:])
    return torch.tensor(np.array(ab), dtype=torch.float32, device=device)


def srgb_to_linear(c):
    return torch.where(c <= 0.04045, c / 12.92, ((c + 0.055) / 1.055).clamp(min=0.0) ** 2.4)


def rgb_to_L_t(rgb):
    # sRGB image 3 x H x W in [0, 1]  ->  Lab lightness 1 x H x W in 0..100
    lin = srgb_to_linear(rgb)
    Y = 0.212671 * lin[0] + 0.715160 * lin[1] + 0.072169 * lin[2]
    f = torch.where(Y > (6.0 / 29.0) ** 3, Y.clamp(min=1e-8) ** (1.0 / 3.0), Y / (3.0 * (6.0 / 29.0) ** 2) + 4.0 / 29.0)
    return (116.0 * f - 16.0)[None]


def lab_to_rgb_gpu(L, ab):
    # L: B x 1 x H x W in 0..100 ; ab: B x 2 x H x W in Lab units  ->  sRGB B x 3 x H x W in [0, 1]
    fy = (L + 16.0) / 116.0
    fx = fy + ab[:, 0:1] / 500.0
    fz = fy - ab[:, 1:2] / 200.0

    def finv(t):
        return torch.where(t > 6.0 / 29.0, t ** 3, 3.0 * (6.0 / 29.0) ** 2 * (t - 4.0 / 29.0))

    X, Y, Z = 0.95047 * finv(fx), 1.0 * finv(fy), 1.08883 * finv(fz)
    r = 3.2404542 * X - 1.5371385 * Y - 0.4985314 * Z
    g = -0.9692660 * X + 1.8760108 * Y + 0.0415560 * Z
    b = 0.0556434 * X - 0.2040259 * Y + 1.0572252 * Z
    lin = torch.cat([r, g, b], dim=1).clamp(0.0, 1.0)
    return torch.where(lin > 0.0031308, 1.055 * lin.clamp(min=1e-8) ** (1.0 / 2.4) - 0.055, 12.92 * lin)


def box_filter(x, r):
    return F.avg_pool2d(x, 2 * r + 1, stride=1, padding=r, count_include_pad=False)


def guided_filter_t(I, p, r, eps):
    # edge-aware smoothing of p guided by I (He et al.), on the GPU
    mean_I, mean_p = box_filter(I, r), box_filter(p, r)
    var_I = box_filter(I * I, r) - mean_I * mean_I
    cov_Ip = box_filter(I * p, r) - mean_I * mean_p
    a = cov_Ip / (var_I + eps)
    b = mean_p - a * mean_I
    return box_filter(a, r) * I + box_filter(b, r)


# ---------------- segmentation model ----------------
_SEG_CACHE = {}
_MEAN = None
_STD = None


def get_segmenter(model_key):
    # loads (once) the pretrained SegFormer and builds the class -> object-group matrix
    if model_key in _SEG_CACHE:
        return _SEG_CACHE[model_key]
    from transformers import SegformerForSemanticSegmentation
    net = SegformerForSemanticSegmentation.from_pretrained(SEG_MODELS[model_key]).to(device).eval()
    if device == "cuda":
        net = net.half()
    id2label = net.config.id2label
    n_cls = max(int(c) for c in id2label) + 1
    M = torch.zeros(len(GROUPS), n_cls)
    for cid, label in id2label.items():
        name = str(label).split(",")[0].strip().lower()
        g = OTHER
        for gi, gname in enumerate(GROUPS[:-1]):
            if name in GROUP_NAMES[gname]:
                g = gi
                break
        M[g, int(cid)] = 1.0
    seg = {"net": net, "M": M.to(device)}
    _SEG_CACHE[model_key] = seg
    return seg


def _round32(v):
    return max(64, int(round(v / 32.0)) * 32)


@torch.no_grad()
def colorize_frame(frame, seg, ab_groups, st, strength=1.2, temporal=0.6, refine=0.7, seg_size=512, want_map=False):
    # frame: H x W x 3 uint8 RGB. st: per-stream state dict (keeps the previous frame's object probabilities).
    # returns (colorized uint8 H x W x 3, object map uint8 H x W x 3 or None, coverage array over GROUPS)
    global _MEAN, _STD
    if _MEAN is None:
        _MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
        _STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
    t = torch.from_numpy(frame).to(device).permute(2, 0, 1).float() / 255.0
    H, W = t.shape[1:]
    L = rgb_to_L_t(t)[None]                                                           # 1 x 1 x H x W, keeps the real brightness
    gray = (0.299 * t[0] + 0.587 * t[1] + 0.114 * t[2])[None, None]

    # 1) semantic segmentation at reduced size
    k = float(seg_size) / max(H, W)
    sh, sw = _round32(H * k), _round32(W * k)
    x = F.interpolate(gray, size=(sh, sw), mode="bilinear", align_corners=False).repeat(1, 3, 1, 1)
    x = (x - _MEAN) / _STD
    if device == "cuda":
        x = x.half()
    probs = torch.softmax(seg["net"](pixel_values=x).logits.float(), dim=1)           # 1 x C x h x w
    gp = torch.einsum("gc,bchw->bghw", seg["M"], probs)                               # 1 x G x h x w (sums to 1)

    # 2) temporal smoothing (reset at scene cuts)
    thumb = F.adaptive_avg_pool2d(gray, (16, 16))
    prev = st.get("prev")
    if prev is not None and prev.shape == gp.shape and float((thumb - st["thumb"]).abs().mean()) < 0.15:
        gp = temporal * prev + (1.0 - temporal) * gp
    st["prev"], st["thumb"] = gp, thumb

    # 3) colour = probability-weighted mix of the scheme colours, then edge-aware refinement
    ab = torch.einsum("gk,bghw->bkhw", ab_groups, gp)
    ab = F.interpolate(ab, size=(H, W), mode="bilinear", align_corners=False)
    if refine > 0:
        r = max(2, int(round(max(H, W) / 150.0)))
        smooth = guided_filter_t(L / 100.0, ab / 128.0, r, 1e-3) * 128.0
        ab = (1.0 - refine) * ab + refine * smooth
    rgb = lab_to_rgb_gpu(L, ab * strength)[0]
    out = (rgb.clamp(0, 1) * 255.0).byte().permute(1, 2, 0).contiguous().cpu().numpy()

    seg_img = None
    if want_map:
        idx = gp.argmax(dim=1, keepdim=True).float()
        idx = F.interpolate(idx, size=(H, W), mode="nearest")[0, 0].long()
        dc = torch.tensor(DISPLAY_COLORS, dtype=torch.float32, device=device)
        mapimg = 0.55 * dc[idx] + 0.45 * (gray[0, 0] * 255.0)[..., None]
        seg_img = mapimg.clamp(0, 255).byte().cpu().numpy()
    cov = gp.mean(dim=(2, 3))[0].cpu().numpy()
    return out, seg_img, cov

print("Colour and segmentation code ready.")

Colour and segmentation code ready.


In [12]:
%%ext task1
def fit_size(w, h, max_side):
    # working size: longest side <= max_side, both sides even (needed for H.264 video)
    k = min(1.0, float(max_side) / max(w, h))
    return max(2, int(w * k) // 2 * 2), max(2, int(h * k) // 2 * 2)


def compose_view(orig, colored, seg_img, view):
    if view == VIEWS[1]:
        return np.concatenate([orig, colored], axis=1)
    if view == VIEWS[2] and seg_img is not None:
        return np.concatenate([orig, colored, seg_img], axis=1)
    return colored


def coverage_text(cov, top=3):
    order = np.argsort(-cov)[:top]
    return ", ".join(f"{GROUPS[i]} {cov[i] * 100:.0f}%" for i in order if cov[i] > 0.01)


def finalize_video(tmp_path, src_path, out_path):
    # re-encode to H.264 so every browser can play it, and copy the original audio when there is one
    ff = shutil.which("ffmpeg")
    if ff is None:
        return tmp_path
    base = [ff, "-y", "-loglevel", "error", "-i", tmp_path]
    tail = ["-c:v", "libx264", "-pix_fmt", "yuv420p", "-preset", "veryfast", out_path]
    if src_path and os.path.isfile(src_path):
        cmd = base + ["-i", src_path, "-map", "0:v:0", "-map", "1:a:0?", "-c:a", "aac", "-shortest"] + tail
        if subprocess.run(cmd).returncode == 0:
            return out_path
    if subprocess.run(base + tail).returncode == 0:
        return out_path
    return tmp_path


def process_video(video_path, url, model_key, seg_size, max_side, max_seconds, view, strength, temporal, refine, *colors):
    # generator: yields (live preview frame, status text, finished video path or None) while the video is processed
    src = (url or "").strip()
    if src:
        if not re.match(r"^(https?|rtsp|rtmp)://", src, re.I):
            raise gr.Error("The video link must start with http://, https://, rtsp:// or rtmp://")
    else:
        src = video_path
    if not src:
        raise gr.Error("Please upload a video (or paste a video link) first.")
    ab_groups = palette_to_ab(colors)
    seg_size, max_side = int(seg_size), int(max_side)
    cap = cv2.VideoCapture(src)
    if not cap.isOpened():
        raise gr.Error("Could not open this video. Try an mp4 file.")
    fps_in = cap.get(cv2.CAP_PROP_FPS)
    if not fps_in or fps_in != fps_in or fps_in < 1 or fps_in > 120:
        fps_in = 25.0
    n_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    max_frames = int(max_seconds * fps_in)
    if n_total > 0:
        max_frames = min(max_frames, n_total)

    yield None, "Loading the segmentation model (first time downloads it) ...", None
    seg = get_segmenter(model_key)
    tmp_dir = tempfile.mkdtemp()
    tmp_path = os.path.join(tmp_dir, "raw.mp4")
    out_path = os.path.join(tmp_dir, "colorized.mp4")
    writer, st = None, {}
    n, t0, last_yield, preview, cov = 0, time.time(), 0.0, None, np.zeros(len(GROUPS))
    try:
        while n < max_frames:
            ok, bgr = cap.read()
            if not ok:
                break
            h0, w0 = bgr.shape[:2]
            w, h = fit_size(w0, h0, max_side)
            if (w, h) != (w0, h0):
                bgr = cv2.resize(bgr, (w, h), interpolation=cv2.INTER_AREA)
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            colored, seg_img, cov = colorize_frame(rgb, seg, ab_groups, st, strength, temporal, refine, seg_size,
                                                   want_map=(view == VIEWS[2]))
            frame_out = compose_view(rgb, colored, seg_img, view)
            if writer is None:
                writer = cv2.VideoWriter(tmp_path, cv2.VideoWriter_fourcc(*"mp4v"), fps_in,
                                         (frame_out.shape[1], frame_out.shape[0]))
            writer.write(cv2.cvtColor(frame_out, cv2.COLOR_RGB2BGR))
            n += 1
            preview = frame_out
            now = time.time()
            if now - last_yield > 0.12:
                last_yield = now
                fps = n / max(now - t0, 1e-6)
                flag = "✅ real time" if fps >= fps_in else "⚠️ slower than real time"
                tot = f"/{max_frames}" if max_frames else ""
                yield preview, (f"Frame {n}{tot} | **{fps:.1f} FPS** (video is {fps_in:.0f} FPS: {flag}) | "
                                f"objects: {coverage_text(cov)}"), None
    finally:
        cap.release()
        if writer is not None:
            writer.release()
    if n == 0:
        raise gr.Error("No frames could be read from this video.")
    fps = n / max(time.time() - t0, 1e-6)
    yield preview, f"Finishing the video file ({n} frames, {fps:.1f} FPS average) ...", None
    final = finalize_video(tmp_path, video_path if not (url or "").strip() else None, out_path)
    flag = "✅ real time" if fps >= fps_in else "⚠️ slower than real time on this GPU/settings"
    yield preview, (f"**Done.** {n} frames in {time.time() - t0:.1f} s = **{fps:.1f} FPS** on average "
                    f"(video is {fps_in:.0f} FPS: {flag}). Download the result below."), final


def live_frame(img, st, model_key, seg_size, max_side, view, strength, temporal, refine, *colors):
    # called for every webcam frame
    if img is None:
        return None, st
    if st is None:
        st = {}
    seg = get_segmenter(model_key)
    ab_groups = palette_to_ab(colors)
    h0, w0 = img.shape[:2]
    w, h = fit_size(w0, h0, int(max_side))
    rgb = np.ascontiguousarray(img[..., :3])
    if (w, h) != (w0, h0):
        rgb = cv2.resize(rgb, (w, h), interpolation=cv2.INTER_AREA)
    colored, seg_img, cov = colorize_frame(rgb, seg, ab_groups, st, strength, temporal, refine, int(seg_size),
                                           want_map=(view == VIEWS[2]))
    return compose_view(rgb, colored, seg_img, view), st


def process_image(img, model_key, seg_size, max_side, strength, refine, *colors):
    if img is None:
        raise gr.Error("Please upload an image first.")
    ab_groups = palette_to_ab(colors)
    seg = get_segmenter(model_key)
    rgb = np.asarray(img.convert("RGB"))
    h0, w0 = rgb.shape[:2]
    w, h = fit_size(w0, h0, max(int(max_side), 1080))
    if (w, h) != (w0, h0):
        rgb = cv2.resize(rgb, (w, h), interpolation=cv2.INTER_AREA)
    colored, seg_img, cov = colorize_frame(rgb, seg, ab_groups, {}, strength, 0.0, refine, int(seg_size), want_map=True)
    return colored, seg_img, {GROUPS[i]: float(cov[i]) for i in range(len(GROUPS))}

print("Processing functions ready.")

Processing functions ready.


In [13]:
%%ext task1
try:
    seg = get_segmenter(DEFAULT_MODEL)
    test = np.zeros((360, 640, 3), np.uint8)
    test[:140] = (150, 150, 150)                 # "sky"
    test[140:260, 100:300] = (90, 90, 90)        # "building"
    test[260:] = (110, 110, 110)                 # "ground"
    ab_g = palette_to_ab(SCHEMES["Natural"])
    st = {}
    for _ in range(3):                           # warm-up
        out, mp, cov = colorize_frame(test, seg, ab_g, st, want_map=True)
    if device == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    n = 20
    for _ in range(n):
        out, mp, cov = colorize_frame(test, seg, ab_g, st, want_map=True)
    fps = n / (time.time() - t0)
    print(f"Pipeline OK. Output {out.shape}, speed on a 640x360 frame: {fps:.1f} FPS ({1000 / fps:.0f} ms per frame).")
    print("Objects found in the test frame:", coverage_text(cov, 5))
    print("ffmpeg available:", shutil.which("ffmpeg") is not None)
except Exception as e:
    traceback.print_exc()
    print("\nSELF-TEST FAILED:", type(e).__name__, e)

INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/nvidia/segformer-b0-finetuned-ade-512-512/revision/main "HTTP/1.1 200 OK"


HTTP Request: GET https://huggingface.co/api/models/nvidia/segformer-b0-finetuned-ade-512-512/revision/main "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: HEAD https://huggingface.co/nvidia/segformer-b0-finetuned-ade-512-512/resolve/489d5cd81a0b59fab9b7ea758d3548ebe99677da/adapter_config.json "HTTP/1.1 404 Not Found"


HTTP Request: HEAD https://huggingface.co/nvidia/segformer-b0-finetuned-ade-512-512/resolve/489d5cd81a0b59fab9b7ea758d3548ebe99677da/adapter_config.json "HTTP/1.1 404 Not Found"


INFO:httpx:HTTP Request: HEAD https://huggingface.co/nvidia/segformer-b0-finetuned-ade-512-512/resolve/489d5cd81a0b59fab9b7ea758d3548ebe99677da/config.json "HTTP/1.1 307 Temporary Redirect"


HTTP Request: HEAD https://huggingface.co/nvidia/segformer-b0-finetuned-ade-512-512/resolve/489d5cd81a0b59fab9b7ea758d3548ebe99677da/config.json "HTTP/1.1 307 Temporary Redirect"


... (9 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B / 15.0MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

Pipeline OK. Output (360, 640, 3), speed on a 640x360 frame: 42.7 FPS (23 ms per frame).
Objects found in the test frame: Buildings 44%, Sky 43%, Other objects 10%, Roads & ground 2%, Mountains & rocks 1%
ffmpeg available: True


In [14]:
%%ext task1
HEADER = (
    "<h1 style='text-align:center; margin-bottom:0'>🎥 Real-time Multi-Object Colorizer</h1>"
    "<p style='text-align:center; margin-top:4px'><b>Semantic segmentation</b> &bull; <b>Vehicles, trees, buildings, sky, roads ...</b> &bull; "
    "<b>Colour schemes per object</b> &bull; <b>Video, webcam and images</b></p>"
)


def apply_scheme(name, *current):
    if name in SCHEMES:
        return tuple(SCHEMES[name])
    return tuple(current)


def build_ui(demo):
    gr.HTML(HEADER)
    with gr.Row():
        # ---------------- settings (shared by all tabs) ----------------
        with gr.Column(scale=1):
            gr.Markdown("### Settings")
            model_dd = gr.Dropdown(choices=list(SEG_MODELS), value=DEFAULT_MODEL, label="Segmentation model")
            seg_dd = gr.Dropdown(choices=[320, 512, 640, 768], value=512, label="Segmentation size (smaller = faster)")
            size_dd = gr.Dropdown(choices=[480, 640, 720, 1080], value=640, label="Output size, longest side in px (smaller = faster)")
            view_rd = gr.Radio(choices=VIEWS, value=VIEWS[1], label="Output view")
            scheme_dd = gr.Dropdown(choices=list(SCHEMES) + [CUSTOM], value="Natural", label="Colour scheme")
            with gr.Accordion("Colour of each object type (edit any of them)", open=True):
                pickers = [gr.ColorPicker(value=SCHEMES["Natural"][i], label=GROUPS[i]) for i in range(len(GROUPS))]
            strength_sl = gr.Slider(0.5, 2.0, value=1.2, step=0.1, label="Colour strength")
            temporal_sl = gr.Slider(0.0, 0.9, value=0.6, step=0.05, label="Temporal smoothing (less flicker in video)")
            refine_sl = gr.Slider(0.0, 1.0, value=0.7, step=0.05, label="Edge-aware refinement")
        # ---------------- tabs ----------------
        with gr.Column(scale=3):
            with gr.Tabs():

                with gr.Tab("🎞 Video"):
                    gr.Markdown("Upload a video file **or** paste a video link, then click **Colorize video**. "
                                "The colorized frames appear live while it is processed.")
                    vid_in = gr.Video(label="Upload a video")
                    url_tb = gr.Textbox(label="Or a video link (http, https, rtsp)", placeholder="https://.../video.mp4")
                    maxsec_sl = gr.Slider(5, 120, value=20, step=5, label="Process at most this many seconds")
                    vid_btn = gr.Button("🎨 Colorize video", variant="primary")
                    vid_status = gr.Markdown("Waiting for a video.")
                    vid_live = gr.Image(label="Live output while processing", height=420)
                    vid_out = gr.Video(label="Colorized video (download)")

                with gr.Tab("📹 Webcam (live)"):
                    gr.Markdown("Live colorization of your camera. Open the public `gradio.live` link in its own browser tab, "
                                "allow camera access, then press record/start. A small, fast setup (segmentation size 320-512, output 480-640) works best.")
                    cam_in = gr.Image(sources=["webcam"], streaming=True, type="numpy", label="Camera", height=300)
                    cam_out = gr.Image(label="Colorized live", height=420)
                    cam_state = gr.State(None)

                with gr.Tab("🖼 Image"):
                    img_in = gr.Image(type="pil", label="Upload an image", height=300)
                    img_btn = gr.Button("🎨 Colorize image", variant="primary")
                    with gr.Row():
                        img_out = gr.Image(label="Colorized", height=360)
                        img_map = gr.Image(label="Object map (what the model found)", height=360)
                    img_cov = gr.Label(label="How much of the image each object type covers", num_top_classes=len(GROUPS))

    # ---------------- events ----------------
    scheme_dd.change(apply_scheme, [scheme_dd] + pickers, pickers)

    vid_btn.click(process_video,
                  [vid_in, url_tb, model_dd, seg_dd, size_dd, maxsec_sl, view_rd, strength_sl, temporal_sl, refine_sl] + pickers,
                  [vid_live, vid_status, vid_out])

    img_btn.click(process_image, [img_in, model_dd, seg_dd, size_dd, strength_sl, refine_sl] + pickers,
                  [img_out, img_map, img_cov])

    cam_inputs = [cam_in, cam_state, model_dd, seg_dd, size_dd, view_rd, strength_sl, temporal_sl, refine_sl] + pickers
    try:
        cam_in.stream(live_frame, cam_inputs, [cam_out, cam_state], time_limit=120, stream_every=0.1)
    except TypeError:
        cam_in.stream(live_frame, cam_inputs, [cam_out, cam_state])

In [15]:
%%ext task2
import os, io, re, glob, time, shutil, random, traceback
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms as T
from PIL import Image, ImageOps
from skimage import color
from skimage.segmentation import flood
from scipy.ndimage import uniform_filter, gaussian_filter
import matplotlib.colors as mcolors
import gradio as gr

# ------------------------- Google Drive -------------------------
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = "/content/drive/MyDrive/hint_colorization"      # photos + trained model are saved here
except Exception as e:
    print("Could not mount Google Drive (", e, ")\nUsing temporary Colab storage - files will be lost when the session ends.")
    DATA_ROOT = "/content/hint_colorization"

# ------------------------- settings -------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE = 256        # training size (divisible by 32)
INFER_SIDE = 256      # longer side used by the model when colorizing
MAX_SIDE = 2048       # very large uploads are shrunk to this
WORK_SIDE = 640       # size of the photo shown in the hint canvas
MIN_SAT = 0.03        # training photos with lower average saturation are treated as black & white and skipped
BATCH_SIZE = 16
LR = 3e-4
COLOR_W = 10.0        # weight of the colour loss
HINT_W = 5.0          # weight of the "follow the hints" loss
P_NO_HINT = 0.15      # share of training samples without any hint (keeps fully automatic colorization working)
P_REGION = 0.35       # share of training samples that also get "same-colour area" hints
HINT_STEP = 8         # spacing (px at model resolution) of the hint grid inside a selected area
NUM_WORKERS = 2
PRETRAINED = True     # ImageNet weights for the encoder (downloaded automatically)

EXTS = (".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp", ".webp")
PHOTO_DIR = os.path.join(DATA_ROOT, "photos")
CACHE_ROOT = "/content/hint_cache"                          # resized copies for fast training (temporary)
CKPT = os.path.join(DATA_ROOT, "hint_colorizer.pt")         # trained model

os.makedirs(PHOTO_DIR, exist_ok=True)
random.seed(0); np.random.seed(0); torch.manual_seed(0)
print("Device:", device)
print("Data folder:", DATA_ROOT)

Mounted at /content/drive
Device: cuda
Data folder: /content/drive/MyDrive/hint_colorization


In [16]:
%%ext task2
# ---------------- training photos ----------------
def prepare_photos():
    # reads the uploaded colour photos, skips bad / black & white ones, saves resized copies for fast training
    out_dir = os.path.join(CACHE_ROOT, "photos")
    os.makedirs(out_dir, exist_ok=True)
    good, bad = [], []
    for f in sorted(glob.glob(os.path.join(PHOTO_DIR, "*"))):
        if not f.lower().endswith(EXTS):
            continue
        try:
            with Image.open(f) as im:
                try:
                    im.draft("RGB", (1024, 1024))
                except Exception:
                    pass
                im = ImageOps.exif_transpose(im).convert("RGB")
            w, h = im.size
            if min(w, h) < 64:
                bad.append((os.path.basename(f), "too small"))
                continue
            sat = np.asarray(im.resize((64, 64)).convert("HSV"))[..., 1].mean() / 255.0
            if sat < MIN_SAT:
                bad.append((os.path.basename(f), "looks black & white (training photos must be colour)"))
                continue
            k = 320.0 / min(w, h)
            if k < 1.0:
                im = im.resize((max(1, int(round(w * k))), max(1, int(round(h * k)))), Image.LANCZOS)
            dst = os.path.join(out_dir, f"photo_{len(good):05d}.jpg")
            im.save(dst, quality=95)
            good.append(dst)
        except Exception as e:
            bad.append((os.path.basename(f), f"unreadable: {e}"))
    return good, bad


# ---------------- colour-hint utilities ----------------
PRESETS = {
    "Sky blue": "#87CEEB", "Deep blue": "#1F4E9C", "Grass green": "#5FAE3C", "Forest green": "#2E6B34",
    "Skin tone (light)": "#E3B79A", "Skin tone (dark)": "#8D5B3F", "Brown / wood": "#8B5A2B",
    "Sand / beige": "#D9C39A", "Red": "#D62F2F", "Yellow": "#F2D02F", "Orange": "#F28C28", "Pink": "#F08FB0",
    "Purple": "#7D4FB5", "Water teal": "#2A9DA8", "Stone gray": "#8A8A8A", "White / cloud": "#F2F2F2",
}


def parse_color(value):
    # colour name, hex code or rgb(...) text -> (r, g, b) integers, or None if it is not understood
    if value is None:
        return None
    s = str(value).strip().lower()
    if not s:
        return None
    m = re.match(r"rgba?\(\s*([\d.]+)\s*,\s*([\d.]+)\s*,\s*([\d.]+)", s)
    if m:
        return tuple(int(round(min(255.0, float(m.group(i))))) for i in (1, 2, 3))
    for name, hx in PRESETS.items():
        if s == name.lower():
            s = hx.lower()
            break
    if re.fullmatch(r"[0-9a-f]{6}", s) or re.fullmatch(r"[0-9a-f]{3}", s):
        s = "#" + s
    try:
        r, g, b = mcolors.to_rgb(s.replace(" ", ""))
        return (int(round(r * 255)), int(round(g * 255)), int(round(b * 255)))
    except Exception:
        return None


def resolve_color(picker_value, text_value):
    # typed colour (if any) wins over the colour picker
    if text_value is not None and str(text_value).strip():
        c = parse_color(text_value)
        if c is None:
            raise gr.Error(f"I do not know the colour '{text_value}'. Use a name like 'sky blue' or a code like #87ceeb.")
        return c
    c = parse_color(picker_value)
    if c is None:
        raise gr.Error("Please pick a colour first.")
    return c


def rgb_to_ab(rgb):
    # the colour part (a, b) of an RGB colour in Lab space, scaled to ~[-1, 1] (the lightness comes from the photo)
    lab = color.rgb2lab(np.array([[rgb]], dtype=np.float64) / 255.0)[0, 0]
    return (lab[1:] / 128.0).astype(np.float32)


def sample_hints(ab, rng, YY, XX):
    # simulates user hints from a real colour image. ab: H x W x 2 in [-1, 1]  ->  H x W x 3 = [hint_a, hint_b, mask]
    H, W = ab.shape[:2]
    hint = np.zeros((H, W, 3), np.float32)
    if rng.random() < P_NO_HINT:
        return hint
    for _ in range(min(int(rng.geometric(1.0 / 8.0)), 40)):                    # random colour spots
        cy, cx = int(rng.integers(0, H)), int(rng.integers(0, W))
        rad = int(rng.integers(1, 9))
        y0, y1, x0, x1 = max(0, cy - rad), min(H, cy + rad + 1), max(0, cx - rad), min(W, cx + rad + 1)
        sub = ((YY[y0:y1, x0:x1] - cy) ** 2 + (XX[y0:y1, x0:x1] - cx) ** 2) <= rad * rad
        col = ab[y0:y1, x0:x1][sub].mean(0) + rng.normal(0, 0.03, 2)
        view = hint[y0:y1, x0:x1]
        view[sub, 0:2] = col
        view[sub, 2] = 1.0
    if rng.random() < P_REGION:                                                # "this whole area has this colour"
        for _ in range(int(rng.integers(1, 4))):
            cy, cx = int(rng.integers(0, H)), int(rng.integers(0, W))
            R = int(rng.integers(20, 71))
            y0, y1, x0, x1 = max(0, cy - R), min(H, cy + R), max(0, cx - R), min(W, cx + R)
            c0 = ab[cy, cx]
            close = np.linalg.norm(ab[y0:y1, x0:x1] - c0, axis=-1) < rng.uniform(0.04, 0.10)
            step = int(rng.choice([6, 8, 10, 12]))
            col = c0 + rng.normal(0, 0.03, 2)
            for py in range(y0 + step // 2, y1, step):
                for px in range(x0 + step // 2, x1, step):
                    if close[py - y0, px - x0]:
                        ya, xa = max(0, py - 2), max(0, px - 2)
                        hint[ya:py + 3, xa:px + 3, 0:2] = col
                        hint[ya:py + 3, xa:px + 3, 2] = 1.0
    return hint


def build_hint_array(hints, h, w):
    # the user's hints -> H x W x 3 array [hint_a, hint_b, mask] at the model resolution
    hint = np.zeros((h, w, 3), np.float32)
    yy, xx = np.mgrid[0:h, 0:w]
    for k in hints:
        ab = rgb_to_ab(k["rgb"])
        if k["kind"] == "point":
            cx, cy = k["x"] * (w - 1), k["y"] * (h - 1)
            rad = max(1.0, float(k["size"]))
            disc = (yy - cy) ** 2 + (xx - cx) ** 2 <= rad * rad
            hint[disc, 0:2] = ab
            hint[disc, 2] = 1.0
        else:
            m = np.asarray(Image.fromarray(k["mask"].astype(np.uint8) * 255).resize((w, h), Image.NEAREST)) > 127
            grid = np.zeros((h, w), bool)
            grid[HINT_STEP // 2::HINT_STEP, HINT_STEP // 2::HINT_STEP] = True
            pts = np.argwhere(m & grid)
            if len(pts) == 0 and m.any():                                       # tiny area: use a few of its pixels
                pts = np.argwhere(m)[::max(1, int(m.sum()) // 3)][:3]
            for py, px in pts:
                ya, xa = max(0, py - 2), max(0, px - 2)
                hint[ya:py + 3, xa:px + 3, 0:2] = ab
                hint[ya:py + 3, xa:px + 3, 2] = 1.0
    return hint


def enforce_hints(ab, hints, enforce):
    # blends the requested colours into the chosen areas (ab: H x W x 2 in ~[-1, 1])
    if enforce <= 0 or not hints:
        return ab
    H, W = ab.shape[:2]
    ab = np.ascontiguousarray(ab, dtype=np.float32).copy()
    for k in hints:
        c = rgb_to_ab(k["rgb"])
        if k["kind"] == "point":
            cx, cy = k["x"] * (W - 1), k["y"] * (H - 1)
            rad = max(2.0, k["size"] / float(INFER_SIDE) * max(H, W))
            pad = int(3 * rad) + 2
            y0, y1 = max(0, int(cy) - pad), min(H, int(cy) + pad + 1)
            x0, x1 = max(0, int(cx) - pad), min(W, int(cx) + pad + 1)
            yy, xx = np.mgrid[y0:y1, x0:x1]
            m = ((yy - cy) ** 2 + (xx - cx) ** 2 <= rad * rad).astype(np.float32)
            m = gaussian_filter(m, max(1.0, 0.35 * rad))[..., None] * enforce
            sub = ab[y0:y1, x0:x1]
            sub[:] = sub * (1.0 - m) + c * m
        else:
            m = np.asarray(Image.fromarray(k["mask"].astype(np.uint8) * 255).resize((W, H), Image.BILINEAR)).astype(np.float32) / 255.0
            m = gaussian_filter(m, max(1.0, 0.004 * max(H, W)))[..., None] * enforce
            ab = ab * (1.0 - m) + c * m
    return ab


def guided_filter(I, p, r, eps):
    # edge-aware smoothing of p guided by I (He et al.): colours follow the brightness edges of the photo
    size = 2 * r + 1
    mean_I, mean_p = uniform_filter(I, size), uniform_filter(p, size)
    var_I = uniform_filter(I * I, size) - mean_I * mean_I
    cov_Ip = uniform_filter(I * p, size) - mean_I * mean_p
    a = cov_Ip / (var_I + eps)
    b = mean_p - a * mean_I
    return uniform_filter(a, size) * I + uniform_filter(b, size)


# ---------------- dataset ----------------
class HintDataset(Dataset):
    def __init__(self, paths, train=True):
        self.paths = paths
        self.train = train
        self.YY, self.XX = np.mgrid[0:IMG_SIZE, 0:IMG_SIZE]
        if train:
            self.tf = T.Compose([T.RandomResizedCrop(IMG_SIZE, scale=(0.3, 1.0)), T.RandomHorizontalFlip()])
        else:
            self.tf = T.Compose([T.Resize(IMG_SIZE), T.CenterCrop(IMG_SIZE)])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        with Image.open(self.paths[i]) as im:
            img = self.tf(im.convert("RGB"))
        lab = color.rgb2lab(np.asarray(img)).astype("float32")
        ab = lab[..., 1:] / 128.0
        rng = np.random.default_rng(None if self.train else 1000 + i)           # fixed hints for validation
        hint = sample_hints(ab, rng, self.YY, self.XX)
        L = torch.from_numpy(lab[..., 0:1] / 50.0 - 1.0).permute(2, 0, 1)        # lightness -> [-1, 1]
        return L, torch.from_numpy(ab).permute(2, 0, 1), torch.from_numpy(hint).permute(2, 0, 1)


# ---------------- model ----------------
class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch + skip_ch + 3, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(True))

    def forward(self, x, skip, hint):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        h = F.interpolate(hint, size=skip.shape[-2:], mode="area")              # hints are fed to every decoder stage
        return self.conv(torch.cat([x, skip, h], dim=1))


class HintColorizer(nn.Module):
    # grayscale (L) + colour hints (hint_a, hint_b, mask)  ->  colour (ab)
    def __init__(self, pretrained=True):
        super().__init__()
        r = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None)
        conv1 = nn.Conv2d(4, 64, 7, 2, 3, bias=False)                           # input: L + 3 hint channels
        with torch.no_grad():
            conv1.weight.zero_()
            conv1.weight[:, 0:1] = r.conv1.weight.sum(dim=1, keepdim=True)      # L channel = pretrained RGB weights summed
        self.stem = nn.Sequential(conv1, r.bn1, r.relu)
        self.pool = r.maxpool
        self.l1, self.l2, self.l3, self.l4 = r.layer1, r.layer2, r.layer3, r.layer4
        self.up3 = UpBlock(512, 256, 256)
        self.up2 = UpBlock(256, 128, 128)
        self.up1 = UpBlock(128, 64, 64)
        self.up0 = UpBlock(64, 64, 64)
        self.head = nn.Sequential(nn.Conv2d(64 + 1 + 3, 32, 3, padding=1), nn.ReLU(True),
                                  nn.Conv2d(32, 2, 1), nn.Tanh())

    def forward(self, L, hint):
        s0 = self.stem(torch.cat([L, hint], dim=1))
        s1 = self.l1(self.pool(s0))
        s2 = self.l2(s1)
        s3 = self.l3(s2)
        s4 = self.l4(s3)
        d = self.up3(s4, s3, hint)
        d = self.up2(d, s2, hint)
        d = self.up1(d, s1, hint)
        d = self.up0(d, s0, hint)
        d = F.interpolate(d, size=L.shape[-2:], mode="bilinear", align_corners=False)
        return self.head(torch.cat([d, L, hint], dim=1))                        # ab in [-1, 1]

print("Photo, hint and model code ready.")

Photo, hint and model code ready.


In [17]:
%%ext task2
model = None            # trained model (filled by training, or loaded from Drive in Cell 5)


def count_photos():
    if not os.path.isdir(PHOTO_DIR):
        return 0
    return len([f for f in os.listdir(PHOTO_DIR) if f.lower().endswith(EXTS)])


def dataset_status():
    n = count_photos()
    note = "" if n >= 100 else ("  (more is better: 100+ recommended, 20 minimum)" if n > 0 else "  (none yet)")
    return f"**Training photos saved in Drive:** {n}{note}"


def save_uploads(uploaded):
    if not uploaded:
        return "No photos selected. Click the upload box, choose photos, then click Save.", dataset_status(), None
    os.makedirs(PHOTO_DIR, exist_ok=True)
    ok = bad = 0
    stamp = int(time.time())
    for i, f in enumerate(uploaded):
        path = f if isinstance(f, str) else getattr(f, "name", None)
        if path is None or os.path.splitext(path)[1].lower() not in EXTS:
            bad += 1
            continue
        try:
            with Image.open(path) as im:
                im.verify()
        except Exception:
            bad += 1
            continue
        shutil.copy(path, os.path.join(PHOTO_DIR, f"up_{stamp}_{i:04d}{os.path.splitext(path)[1].lower()}"))
        ok += 1
    msg = f"Saved **{ok}** photo(s)." + (f" Skipped {bad} (not valid images)." if bad else "")
    return msg + " Upload more, or go to the Train Model tab.", dataset_status(), None


def masked_l1(pred, hint):
    # how far the prediction is from the hinted colours, only at hinted pixels
    m = hint[:, 2:3]
    return ((pred - hint[:, 0:2]).abs() * m).sum() / (m.sum() * 2.0 + 1e-6)


def train_model(epochs):
    global model
    epochs = int(epochs)
    log = []

    def emit(msg):
        log.append(msg)
        return "\n".join(log)

    yield emit("Preparing dataset ...")
    if os.path.exists(CACHE_ROOT):
        shutil.rmtree(CACHE_ROOT)
    paths, bad = prepare_photos()
    yield emit(f"  usable colour photos: {len(paths)}   skipped: {len(bad)}")
    for name, why in bad[:3]:
        yield emit(f"      skipped {name}: {why}")
    if len(paths) < 8:
        raise gr.Error("Fewer than 8 usable colour photos. Upload more colour photos in the first tab.")
    if len(paths) < 20:
        yield emit("    WARNING: fewer than 20 photos; results will be weak. 100+ is recommended.")

    random.shuffle(paths)
    n_val = max(2, int(0.1 * len(paths)))
    train_ds = HintDataset(paths[n_val:], train=True)
    val_ds = HintDataset(paths[:n_val], train=False)
    bs = min(BATCH_SIZE, len(train_ds))
    train_dl = DataLoader(train_ds, batch_size=bs, shuffle=True, num_workers=NUM_WORKERS, drop_last=len(train_ds) > bs)
    val_dl = DataLoader(val_ds, batch_size=bs, shuffle=False, num_workers=NUM_WORKERS)
    yield emit(f"Training photos: {len(train_ds)} | validation photos: {len(val_ds)} | batch size: {bs}")

    yield emit("Building model (downloads pretrained weights the first time) ...")
    net = HintColorizer(pretrained=PRETRAINED).to(device)
    use_amp = (device == "cuda")
    opt = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, epochs=epochs, steps_per_epoch=len(train_dl))
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    color_loss = nn.SmoothL1Loss()
    best_val = float("inf")
    t0 = time.time()
    yield emit(f"Training for {epochs} epochs on {device} ...")

    for epoch in range(epochs):
        net.train()
        tr = 0.0
        for L, ab, hint in train_dl:
            L, ab, hint = L.to(device), ab.to(device), hint.to(device)
            with torch.autocast(device_type=device, enabled=use_amp):
                pred = net(L, hint)
            pred = pred.float()
            loss = COLOR_W * color_loss(pred, ab) + HINT_W * masked_l1(pred, hint)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            tr += loss.item()
        tr /= len(train_dl)

        net.eval()
        v_h = v_0 = 0.0
        with torch.no_grad():
            for L, ab, hint in val_dl:
                L, ab, hint = L.to(device), ab.to(device), hint.to(device)
                v_h += color_loss(net(L, hint).float(), ab).item()
                v_0 += color_loss(net(L, torch.zeros_like(hint)).float(), ab).item()
        v_h /= len(val_dl)
        v_0 /= len(val_dl)
        va = COLOR_W * (v_h + v_0) / 2.0

        tag = ""
        if va < best_val:
            best_val = va
            torch.save({"model": net.state_dict()}, CKPT)
            tag = "   <- best so far, saved"
        yield emit(f"epoch {epoch + 1:02d}/{epochs}   train {tr:.3f}   val error with hints {v_h:.4f}   "
                   f"without hints {v_0:.4f}   ({time.time() - t0:.0f}s){tag}")

    ck = torch.load(CKPT, map_location=device)
    net.load_state_dict(ck["model"])
    net.eval()

    e_h = e_0 = 0.0
    n = 0
    with torch.no_grad():
        for L, ab, hint in val_dl:
            L, ab, hint = L.to(device), ab.to(device), hint.to(device)
            e_h += (net(L, hint).float() - ab).abs().mean().item() * len(L)
            e_0 += (net(L, torch.zeros_like(hint)).float() - ab).abs().mean().item() * len(L)
            n += len(L)
    e_h /= max(1, n)
    e_0 /= max(1, n)
    yield emit(f"\nAverage colour error on validation photos: without hints {e_0:.4f}  ->  with simulated hints {e_h:.4f} "
               f"({100.0 * (e_0 - e_h) / max(e_0, 1e-9):.0f}% lower). Lower is better.")

    model = net
    yield emit("\nDONE. Model saved to Drive. Go to the 'Colorize with your colour hints' tab.")

print("Upload and training functions ready.")

Upload and training functions ready.


In [18]:
%%ext task2
def _round32(v):
    return max(64, int(round(v / 32.0)) * 32)


def _prep(img):
    # PIL image -> model input tensor + full-resolution lightness
    img = ImageOps.exif_transpose(img).convert("RGB")
    W, H = img.size
    if max(W, H) > MAX_SIDE:
        k = MAX_SIDE / float(max(W, H))
        W, H = max(1, int(W * k)), max(1, int(H * k))
        img = img.resize((W, H), Image.LANCZOS)
    L_full = color.rgb2lab(np.asarray(img))[..., 0].astype(np.float32)        # 0..100 at full resolution
    k = INFER_SIDE / float(max(W, H))
    small = img.resize((_round32(W * k), _round32(H * k)), Image.BICUBIC)
    L = color.rgb2lab(np.asarray(small))[..., 0] / 50.0 - 1.0
    x = torch.from_numpy(L).float()[None, None].to(device)
    return x, L_full, H, W


def _check_ready():
    if model is None:
        raise gr.Error("No trained model yet. Go to the 'Train Model' tab and click Start training first.")


@torch.no_grad()
def colorize_with_hints(img, hints, saturation=1.2, enforce=0.7, refine=0.7):
    # hints: list of the user's colour hints (empty list = fully automatic colorization)
    # enforce: 0..1 how strongly the requested colours are forced into the chosen areas
    # refine:  0..1 edge-aware colour refinement
    _check_ready()
    model.eval()
    x, L_full, H, W = _prep(img)
    h, w = x.shape[-2:]
    hint = torch.from_numpy(build_hint_array(hints, h, w)).permute(2, 0, 1)[None].to(device)
    ab = model(x, hint).float().cpu()                                           # 1 x 2 x h x w in [-1, 1]
    ab = F.interpolate(ab, size=(H, W), mode="bilinear", align_corners=False)[0].permute(1, 2, 0).numpy()
    ab = np.ascontiguousarray(ab, dtype=np.float32) * saturation               # colour strength
    ab = enforce_hints(ab, hints, enforce)                                      # the requested colours in the chosen areas
    if refine > 0:                                                              # snap colours to the photo's brightness edges
        guide = (L_full / 100.0).astype(np.float32)
        r = max(2, int(round(max(H, W) / 150.0)))
        smooth = np.stack([guided_filter(guide, np.ascontiguousarray(ab[..., k]), r, 1e-3) for k in range(2)], axis=-1)
        ab = (1.0 - refine) * ab + refine * smooth
    ab = (ab * 128.0).clip(-128, 127)
    lab = np.concatenate([L_full[..., None], ab], axis=-1)
    rgb = (color.lab2rgb(lab).clip(0, 1) * 255).astype(np.uint8)
    return Image.fromarray(rgb)


# load a previously trained model from Drive, if there is one
if os.path.exists(CKPT):
    try:
        ck = torch.load(CKPT, map_location=device)
        model = HintColorizer(pretrained=False).to(device)
        model.load_state_dict(ck["model"])
        model.eval()
        print("Loaded saved model from Drive.")
    except Exception as e:
        model = None
        print("Could not load the saved model (", e, "). Please train again in the app.")
else:
    print("No saved model yet. Upload photos and train inside the app (next cell).")

Loaded saved model from Drive.


In [19]:
%%ext task2
MODE_DOT, MODE_WAND, MODE_RECT = "dot", "wand", "rect"


# ---------------- hint canvas helpers ----------------
def hints_html(state):
    hs = state.get("hints", []) if state else []
    if not hs:
        return "<i>No colour hints yet. Without hints the model colorizes automatically.</i>"
    rows = []
    for i, h in enumerate(hs, 1):
        hexc = "#%02x%02x%02x" % tuple(h["rgb"])
        desc = (f"Dot at ({h['x'] * 100:.0f}%, {h['y'] * 100:.0f}%), size {h['size']}" if h["kind"] == "point" else h["label"])
        rows.append(f"<div>{i}. <span style='display:inline-block;width:14px;height:14px;background:{hexc};"
                    f"border:1px solid #888;vertical-align:middle'></span> {hexc} - {desc}</div>")
    if state.get("pending"):
        rows.append("<div><b>Rectangle: first corner set. Click the opposite corner.</b></div>")
    return "".join(rows)


def render_overlay(state):
    # the grayscale photo with the user's hints drawn on it
    arr = np.asarray(state["work"].convert("RGB")).astype(np.float32).copy()
    Hd, Wd = arr.shape[:2]
    yy, xx = np.mgrid[0:Hd, 0:Wd]
    for h in state["hints"]:
        c = np.array(h["rgb"], np.float32)
        if h["kind"] == "region":
            m = h["mask"]
            arr[m] = 0.5 * arr[m] + 0.5 * c
        else:
            cx, cy = h["x"] * (Wd - 1), h["y"] * (Hd - 1)
            rad = max(3.0, h["size"] / float(INFER_SIDE) * max(Hd, Wd))
            d2 = (yy - cy) ** 2 + (xx - cx) ** 2
            arr[(d2 <= (rad + 2) ** 2) & (d2 > rad * rad)] = 255.0
            arr[d2 <= rad * rad] = c
    if state.get("pending"):
        px, py = int(state["pending"][0] * (Wd - 1)), int(state["pending"][1] * (Hd - 1))
        arr[max(0, py - 6):py + 7, px] = (255, 0, 255)
        arr[py, max(0, px - 6):px + 7] = (255, 0, 255)
    return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))


def add_hint(state, nx, ny, mode, size, tol, rgb):
    Wd, Hd = state["work"].size
    nx, ny = float(min(max(nx, 0.0), 1.0)), float(min(max(ny, 0.0), 1.0))
    rgb = tuple(int(v) for v in rgb)
    hexc = "#%02x%02x%02x" % rgb
    row, col = int(round(ny * (Hd - 1))), int(round(nx * (Wd - 1)))
    if mode == MODE_DOT:
        state["hints"].append({"kind": "point", "x": nx, "y": ny, "size": int(size), "rgb": rgb})
        return f"Added a {hexc} dot. Add more hints, or click Colorize."
    if mode == MODE_WAND:
        m = flood(state["gray"], (row, col), tolerance=float(tol) / 100.0)
        frac = float(m.mean())
        state["hints"].append({"kind": "region", "mask": m, "rgb": rgb,
                               "label": f"Magic-wand area ({frac * 100:.0f}% of the photo)"})
        msg = f"Selected an area covering {frac * 100:.0f}% of the photo and coloured it {hexc}."
        if frac > 0.6:
            msg += " That is very large; lower the tolerance (or undo) if it covers too much."
        elif frac < 0.002:
            msg += " That is tiny; raise the tolerance (or undo) to select more."
        return msg
    if state.get("pending") is None:                                            # rectangle: first corner
        state["pending"] = (nx, ny)
        return "First corner set (pink cross). Now click the opposite corner."
    x0, y0 = state["pending"]
    state["pending"] = None
    c0, c1 = sorted([int(round(x0 * (Wd - 1))), col])
    r0, r1 = sorted([int(round(y0 * (Hd - 1))), row])
    m = np.zeros((Hd, Wd), bool)
    m[r0:r1 + 1, c0:c1 + 1] = True
    state["hints"].append({"kind": "region", "mask": m, "rgb": rgb,
                           "label": f"Rectangle ({float(m.mean()) * 100:.0f}% of the photo)"})
    return f"Rectangle added with colour {hexc}."


# ---------------- event handlers ----------------
def on_image_change(img, state):
    if img is None:
        return None, None, hints_html(None), "Upload a photo to begin."
    base = ImageOps.exif_transpose(img).convert("RGB")
    work = base.copy()
    work.thumbnail((WORK_SIDE, WORK_SIDE), Image.LANCZOS)
    gray = work.convert("L")
    st = {"base": base, "work": Image.merge("RGB", (gray, gray, gray)),
          "gray": gaussian_filter(np.asarray(gray).astype(np.float32) / 255.0, 1.5), "hints": [], "pending": None}
    return render_overlay(st), st, hints_html(st), "Photo loaded. Choose a colour and a tool, then click on the photo on the right."


def on_click(state, mode, size, tol, cpick, ctext, evt: gr.SelectData):
    if not state or "work" not in state:
        raise gr.Error("Upload a photo first.")
    Wd, Hd = state["work"].size
    rgb = resolve_color(cpick, ctext)
    msg = add_hint(state, evt.index[0] / float(max(1, Wd - 1)), evt.index[1] / float(max(1, Hd - 1)), mode, size, tol, rgb)
    return render_overlay(state), state, hints_html(state), msg


def on_add_xy(state, xpct, ypct, mode, size, tol, cpick, ctext):
    if not state or "work" not in state:
        raise gr.Error("Upload a photo first.")
    rgb = resolve_color(cpick, ctext)
    msg = add_hint(state, xpct / 100.0, ypct / 100.0, mode, size, tol, rgb)
    return render_overlay(state), state, hints_html(state), msg


def on_undo(state):
    if not state or "work" not in state:
        return None, state, hints_html(None), "Upload a photo first."
    if state["hints"]:
        state["hints"].pop()
    state["pending"] = None
    return render_overlay(state), state, hints_html(state), "Removed the last hint."


def on_clear(state):
    if not state or "work" not in state:
        return None, state, hints_html(None), "Upload a photo first."
    state["hints"] = []
    state["pending"] = None
    return render_overlay(state), state, hints_html(state), "All hints cleared."


def run_colorize(state, saturation, enforce, refine):
    if not state or "base" not in state:
        raise gr.Error("Please upload a photo first.")
    try:
        t0 = time.time()
        base, hints = state["base"], state["hints"]
        auto = colorize_with_hints(base, [], saturation, enforce, refine)
        res = colorize_with_hints(base, hints, saturation, enforce, refine) if hints else auto
        orig = base if base.size == res.size else base.resize(res.size, Image.LANCZOS)
        note = (f"Colorized using **{len(hints)}** colour hint(s) in {time.time() - t0:.1f} seconds." if hints
                else "No hints were added, so this is the automatic colorization. Click on the photo to add colour hints.")
        return res, orig, res, auto, res, note
    except gr.Error:
        raise
    except Exception as e:
        traceback.print_exc()
        raise gr.Error(f"{type(e).__name__}: {e}")


def model_status():
    global model
    if model is None and os.path.exists(CKPT):
        try:
            ck = torch.load(CKPT, map_location=device)
            m = HintColorizer(pretrained=False).to(device)
            m.load_state_dict(ck["model"])
            m.eval()
            model = m
        except Exception as e:
            print("Could not load saved model:", e)
    if model is None:
        return "⚠️ **No trained model yet.** Go to tab **2. Train Model** and click **Start training** (upload colour photos in tab 1 first)."
    return "✅ **Model ready.**"


HEADER = (
    "<h1 style='text-align:center; margin-bottom:0'>🎨 Conditional Image Colorizer</h1>"
    "<p style='text-align:center; margin-top:4px'><b>Tell the model what colour each area should be</b> &bull; "
    "<b>Pick or type colours</b> &bull; <b>Click to mark areas</b> &bull; <b>Compare with and without hints</b></p>"
)

def build_ui(demo):
    gr.HTML(HEADER)
    state = gr.State(None)
    with gr.Tabs():

        # ---------------- Tab 1: upload training photos ----------------
        with gr.Tab("1. Upload Training Photos"):
            gr.Markdown("Upload **real colour photos** of any scenes (sky, grass, people, buildings, water ...). The model learns natural colours "
                        "and how to follow colour hints. Use varied photos. Aim for 100+ (20 minimum).")
            up_files = gr.File(label="Choose colour photos", file_count="multiple", file_types=["image"])
            up_btn = gr.Button("Save photos", variant="primary")
            up_msg = gr.Markdown()
            ds_md = gr.Markdown(dataset_status())
            up_btn.click(save_uploads, inputs=[up_files], outputs=[up_msg, ds_md, up_files])

        # ---------------- Tab 2: train ----------------
        with gr.Tab("2. Train Model"):
            gr.Markdown("Click **Start training** after uploading photos. It takes roughly 5-20 minutes on a T4 GPU. "
                        "For a quick test, use 8 epochs. The best model is saved to Google Drive automatically.")
            epochs_sl = gr.Slider(5, 60, value=25, step=1, label="Epochs")
            train_btn = gr.Button("Start training", variant="primary")
            log_box = gr.Textbox(label="Training log", lines=16, max_lines=16, interactive=False)

        # ---------------- Tab 3: colorize with hints ----------------
        with gr.Tab("3. Colorize with your colour hints"):
            model_md = gr.Markdown()
            with gr.Row():
                with gr.Column(scale=1):
                    inp = gr.Image(type="pil", label="📷 1) Upload a B&W photo", height=260)
                    gr.Markdown("**2) Choose the colour for an area**")
                    preset_dd = gr.Dropdown(choices=list(PRESETS), value="Sky blue", label="Preset colour")
                    cpick = gr.ColorPicker(value=PRESETS["Sky blue"], label="Or pick any colour")
                    ctext = gr.Textbox(label="Or type a colour name / hex code (overrides the picker)",
                                       placeholder="e.g. grass green, skyblue, #87ceeb")
                    gr.Markdown("**3) Choose a tool, then click on the photo (right) to mark the area**")
                    mode_rd = gr.Radio(choices=[("● Dot (small spot)", MODE_DOT),
                                                ("🪄 Magic wand (select the area with similar brightness)", MODE_WAND),
                                                ("▭ Rectangle (click two opposite corners)", MODE_RECT)],
                                       value=MODE_WAND, label="Hint tool")
                    size_sl = gr.Slider(1, 12, value=4, step=1, label="Dot size")
                    tol_sl = gr.Slider(2, 40, value=12, step=1, label="Magic wand tolerance (%): higher = bigger area")
                    with gr.Row():
                        undo_btn = gr.Button("Undo last hint")
                        clear_btn = gr.Button("Clear all hints")
                    gr.Markdown("**4) Colorize**")
                    enforce_sl = gr.Slider(0.0, 1.0, value=0.7, step=0.05, label="Hint strength (how strongly your colours are applied)")
                    sat = gr.Slider(0.5, 2.0, value=1.2, step=0.1, label="Overall colour strength")
                    refine_sl = gr.Slider(0.0, 1.0, value=0.7, step=0.05, label="Edge-aware colour refinement")
                    btn = gr.Button("🎨 Colorize", variant="primary")
                    with gr.Accordion("Can't click on the photo? Add a hint by position", open=False):
                        x_sl = gr.Slider(0, 100, value=50, step=1, label="X position (% from the left)")
                        y_sl = gr.Slider(0, 100, value=20, step=1, label="Y position (% from the top)")
                        add_btn = gr.Button("Add hint at this position")
                with gr.Column(scale=2):
                    status = gr.Markdown("Upload a photo, then click on it to add colour hints.")
                    canvas = gr.Image(type="pil", interactive=False, height=420, label="👆 Click on the photo to add colour hints")
                    hint_list = gr.HTML(hints_html(None))
                    with gr.Tabs():
                        with gr.Tab("🌈 Colorized Result"):
                            out_main = gr.Image(type="pil", label="Colorized", height=520)
                        with gr.Tab("Side by Side Comparison"):
                            gr.Markdown("<p style='text-align:center'><b>Original (left) | Colorized (right)</b></p>")
                            with gr.Row():
                                out_orig = gr.Image(type="pil", label="Original", height=480)
                                out_col = gr.Image(type="pil", label="Colorized", height=480)
                        with gr.Tab("With vs Without Hints"):
                            gr.Markdown("<p style='text-align:center'><b>Automatic (no hints) (left) | With your colour hints (right)</b></p>")
                            with gr.Row():
                                out_auto = gr.Image(type="pil", label="Automatic (no hints)", height=480)
                                out_hint = gr.Image(type="pil", label="With your hints", height=480)

    # ---------------- events ----------------
    train_btn.click(train_model, inputs=epochs_sl, outputs=log_box).then(model_status, None, model_md)
    demo.load(model_status, None, model_md)
    inp.change(on_image_change, [inp, state], [canvas, state, hint_list, status])
    preset_dd.change(lambda name: PRESETS[name], preset_dd, cpick)
    canvas.select(on_click, [state, mode_rd, size_sl, tol_sl, cpick, ctext], [canvas, state, hint_list, status])
    add_btn.click(on_add_xy, [state, x_sl, y_sl, mode_rd, size_sl, tol_sl, cpick, ctext], [canvas, state, hint_list, status])
    undo_btn.click(on_undo, [state], [canvas, state, hint_list, status])
    clear_btn.click(on_clear, [state], [canvas, state, hint_list, status])
    btn.click(run_colorize, [state, sat, enforce_sl, refine_sl], [out_main, out_orig, out_col, out_auto, out_hint, status])

In [20]:
%%ext task3
import os, io, glob, time, shutil, random, traceback
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms as T
from PIL import Image, ImageOps
from skimage import color
from scipy.ndimage import uniform_filter
import gradio as gr

# ------------------------- Google Drive -------------------------
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = "/content/drive/MyDrive/scene_colorization"      # photos + trained model are saved here
except Exception as e:
    print("Could not mount Google Drive (", e, ")\nUsing temporary Colab storage - files will be lost when the session ends.")
    DATA_ROOT = "/content/scene_colorization"

# ------------------------- scene types (folder names) -------------------------
# You may edit this list BEFORE the first training. If you change it later, train again.
SCENES = ["cityscape", "forest", "mountain", "coast", "countryside"]
S2I = {s: i for i, s in enumerate(SCENES)}
SCENE_LABELS = {"cityscape": "🏙 Cityscape", "forest": "🌲 Forest", "mountain": "⛰ Mountain",
                "coast": "🌊 Coast / water", "countryside": "🌾 Countryside"}
for s in SCENES:
    SCENE_LABELS.setdefault(s, s)

# ------------------------- settings -------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE = 256        # training size (divisible by 32). 320 gives finer detail but trains slower
INFER_SIDE = 256      # longer side used by the model when colorizing
MAX_SIDE = 2048       # very large uploads are shrunk to this
MIN_SAT = 0.03        # photos with lower average saturation are treated as black & white and skipped in training
BATCH_SIZE = 16
LR = 3e-4
COLOR_W = 10.0        # weight of the colour (ab) loss
CLS_W = 0.5           # weight of the scene-classification loss
PERC_W = 1.0          # weight of the perceptual (VGG16) loss; 0 = off (faster, no 0.5 GB download)
NUM_WORKERS = 2
PRETRAINED = True     # ImageNet weights for the encoder (downloaded automatically)

EXTS = (".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp", ".webp")
CACHE_ROOT = "/content/scene_cache"                       # resized copies for fast training (temporary)
CKPT = os.path.join(DATA_ROOT, "scene_colorizer.pt")      # trained model

for s in SCENES:
    os.makedirs(os.path.join(DATA_ROOT, s), exist_ok=True)
random.seed(0); np.random.seed(0); torch.manual_seed(0)
print("Device:", device)
print("Data folder:", DATA_ROOT)
print("Scene types:", SCENES)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
Data folder: /content/drive/MyDrive/scene_colorization
Scene types: ['cityscape', 'forest', 'mountain', 'coast', 'countryside']


In [21]:
%%ext task3
def prepare_scene(s):
    # reads photos of one scene type, skips bad / black & white ones, saves resized copies for fast training
    out_dir = os.path.join(CACHE_ROOT, s)
    os.makedirs(out_dir, exist_ok=True)
    good, bad = [], []
    for f in sorted(glob.glob(os.path.join(DATA_ROOT, s, "*"))):
        if not f.lower().endswith(EXTS):
            continue
        try:
            with Image.open(f) as im:
                try:
                    im.draft("RGB", (1024, 1024))
                except Exception:
                    pass
                im = ImageOps.exif_transpose(im).convert("RGB")
            w, h = im.size
            if min(w, h) < 64:
                bad.append((os.path.basename(f), "too small"))
                continue
            sat = np.asarray(im.resize((64, 64)).convert("HSV"))[..., 1].mean() / 255.0
            if sat < MIN_SAT:
                bad.append((os.path.basename(f), "looks black & white (training photos must be colour)"))
                continue
            k = 320.0 / min(w, h)
            if k < 1.0:
                im = im.resize((max(1, int(round(w * k))), max(1, int(round(h * k)))), Image.LANCZOS)
            dst = os.path.join(out_dir, f"{s}_{len(good):05d}.jpg")
            im.save(dst, quality=95)
            good.append(dst)
        except Exception as e:
            bad.append((os.path.basename(f), f"unreadable: {e}"))
    return good, bad


class SceneDataset(Dataset):
    def __init__(self, items, train=True):
        self.items = items            # list of (path, scene_id)
        if train:
            self.tf = T.Compose([T.RandomResizedCrop(IMG_SIZE, scale=(0.3, 1.0)), T.RandomHorizontalFlip()])
        else:
            self.tf = T.Compose([T.Resize(IMG_SIZE), T.CenterCrop(IMG_SIZE)])

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        path, sid = self.items[i]
        with Image.open(path) as im:
            img = self.tf(im.convert("RGB"))
        lab = color.rgb2lab(np.asarray(img)).astype("float32")
        L = torch.from_numpy(lab[..., 0:1] / 50.0 - 1.0).permute(2, 0, 1)     # lightness -> [-1, 1]
        ab = torch.from_numpy(lab[..., 1:] / 128.0).permute(2, 0, 1)          # colour    -> ~[-1, 1]
        return L, ab, sid


# ---------------- differentiable Lab -> sRGB (used by the perceptual loss) ----------------
def lab_to_rgb_t(L, ab):
    # L: B x 1 x H x W in [-1, 1] ; ab: B x 2 x H x W in ~[-1, 1]  ->  sRGB B x 3 x H x W in [0, 1]
    Lc = (L + 1.0) * 50.0
    a = ab[:, 0:1] * 128.0
    b = ab[:, 1:2] * 128.0
    fy = (Lc + 16.0) / 116.0
    fx = fy + a / 500.0
    fz = fy - b / 200.0

    def finv(t):
        return torch.where(t > 6.0 / 29.0, t ** 3, 3.0 * (6.0 / 29.0) ** 2 * (t - 4.0 / 29.0))

    X, Y, Z = 0.95047 * finv(fx), 1.0 * finv(fy), 1.08883 * finv(fz)
    r = 3.2404542 * X - 1.5371385 * Y - 0.4985314 * Z
    g = -0.9692660 * X + 1.8760108 * Y + 0.0415560 * Z
    bl = 0.0556434 * X - 0.2040259 * Y + 1.0572252 * Z
    lin = torch.cat([r, g, bl], dim=1).clamp(0.0, 1.0)
    return torch.where(lin > 0.0031308, 1.055 * lin.clamp(min=1e-8) ** (1.0 / 2.4) - 0.055, 12.92 * lin)


# ---------------- model ----------------
def pos_enc_2d(h, w, dim, device):
    # sinusoidal 2-D positional encoding (dim divisible by 4) -> (h*w) x dim
    d = dim // 2

    def enc1d(n):
        pos = torch.arange(n, device=device, dtype=torch.float32)[:, None]
        i = torch.arange(0, d, 2, device=device, dtype=torch.float32)[None, :]
        ang = pos / (10000.0 ** (i / d))
        e = torch.zeros(n, d, device=device)
        e[:, 0::2] = torch.sin(ang)
        e[:, 1::2] = torch.cos(ang)
        return e

    ey = enc1d(h)[:, None, :].expand(h, w, d)
    ex = enc1d(w)[None, :, :].expand(h, w, d)
    return torch.cat([ey, ex], dim=-1).reshape(h * w, dim)


class ContextBlock(nn.Module):
    # global self-attention: every location can look at every other location of the scene
    def __init__(self, dim, heads=8):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, dim * 2), nn.GELU(), nn.Linear(dim * 2, dim))
        self.gamma = nn.Parameter(torch.zeros(1))        # starts as identity, learns how much context to mix in

    def forward(self, x):
        B, C, H, W = x.shape
        t = x.flatten(2).transpose(1, 2)                                  # B x HW x C
        t = t + pos_enc_2d(H, W, C, x.device).to(t.dtype)[None]
        h = self.norm1(t)
        a, _ = self.attn(h, h, h, need_weights=False)
        delta = a + self.mlp(self.norm2(t + a))
        return x + self.gamma * delta.transpose(1, 2).reshape(B, C, H, W).to(x.dtype)


class FiLM(nn.Module):
    # scales and shifts feature maps depending on the scene type
    def __init__(self, emb_dim, ch):
        super().__init__()
        self.fc = nn.Linear(emb_dim, ch * 2)
        nn.init.zeros_(self.fc.weight)
        nn.init.zeros_(self.fc.bias)

    def forward(self, x, e):
        g, b = self.fc(e).chunk(2, dim=1)
        return x * (1 + g[:, :, None, None]) + b[:, :, None, None]


class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, emb_dim):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch + skip_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(True))
        self.film = FiLM(emb_dim, out_ch)

    def forward(self, x, skip, e):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = self.conv(torch.cat([x, skip], dim=1))
        return self.film(x, e)


class SceneColorizer(nn.Module):
    # grayscale (L) + scene  ->  colour (ab) ;  grayscale (L)  ->  scene logits
    def __init__(self, n_scenes, emb_dim=64, pretrained=True):
        super().__init__()
        r = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None)
        self.stem = nn.Sequential(r.conv1, r.bn1, r.relu)
        self.pool = r.maxpool
        self.l1, self.l2, self.l3, self.l4 = r.layer1, r.layer2, r.layer3, r.layer4
        self.ctx3 = ContextBlock(256)
        self.ctx4 = ContextBlock(512)
        self.emb = nn.Embedding(n_scenes, emb_dim)
        self.film4 = FiLM(emb_dim, 512)
        self.up3 = UpBlock(512, 256, 256, emb_dim)
        self.up2 = UpBlock(256, 128, 128, emb_dim)
        self.up1 = UpBlock(128, 64, 64, emb_dim)
        self.up0 = UpBlock(64, 64, 64, emb_dim)
        self.head = nn.Sequential(nn.Conv2d(64 + 1, 32, 3, padding=1), nn.ReLU(True),
                                  nn.Conv2d(32, 2, 1), nn.Tanh())
        self.cls = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(512, n_scenes))

    def encode(self, L):
        x = L.repeat(1, 3, 1, 1)
        s0 = self.stem(x)
        s1 = self.l1(self.pool(s0))
        s2 = self.l2(s1)
        s3 = self.ctx3(self.l3(s2))
        s4 = self.ctx4(self.l4(s3))
        return s0, s1, s2, s3, s4

    def scene_logits(self, L):
        return self.cls(self.encode(L)[4])

    def forward(self, L, scene):
        # scene: LongTensor of scene ids (B)  or  FloatTensor of scene probabilities (B x n_scenes) for a blended style
        s0, s1, s2, s3, s4 = self.encode(L)
        logits = self.cls(s4)
        e = self.emb(scene) if scene.dtype in (torch.long, torch.int64) else scene @ self.emb.weight
        d = self.up3(self.film4(s4, e), s3, e)
        d = self.up2(d, s2, e)
        d = self.up1(d, s1, e)
        d = self.up0(d, s0, e)
        d = F.interpolate(d, size=L.shape[-2:], mode="bilinear", align_corners=False)
        return self.head(torch.cat([d, L], dim=1)), logits


def guided_filter(I, p, r, eps):
    # edge-aware smoothing of p guided by I (He et al.): colours follow the brightness edges of the photo
    size = 2 * r + 1
    mean_I, mean_p = uniform_filter(I, size), uniform_filter(p, size)
    var_I = uniform_filter(I * I, size) - mean_I * mean_I
    cov_Ip = uniform_filter(I * p, size) - mean_I * mean_p
    a = cov_Ip / (var_I + eps)
    b = mean_p - a * mean_I
    return uniform_filter(a, size) * I + uniform_filter(b, size)

print("Dataset and model code ready.")

Dataset and model code ready.


In [22]:
%%ext task3
model = None            # trained model (filled by training, or loaded from Drive in Cell 5)
ACTIVE_SCENES = []      # scene types the model was trained on


def count_photos(s):
    folder = os.path.join(DATA_ROOT, s)
    if not os.path.isdir(folder):
        return 0
    return len([f for f in os.listdir(folder) if f.lower().endswith(EXTS)])


def dataset_status():
    rows = ["| Scene type | Photos saved in Drive |", "|---|---|"]
    for s in SCENES:
        n = count_photos(s)
        flag = "" if n >= 20 else ("  (need more, 20+ recommended)" if n > 0 else "  (none yet)")
        rows.append(f"| {SCENE_LABELS[s]} | {n}{flag} |")
    return "\n".join(rows)


def save_uploads(scene, uploaded):
    if not uploaded:
        return "No photos selected. Click the upload box, choose photos, then click Save.", dataset_status(), None
    folder = os.path.join(DATA_ROOT, scene)
    os.makedirs(folder, exist_ok=True)
    ok = bad = 0
    stamp = int(time.time())
    for i, f in enumerate(uploaded):
        path = f if isinstance(f, str) else getattr(f, "name", None)
        if path is None or os.path.splitext(path)[1].lower() not in EXTS:
            bad += 1
            continue
        try:
            with Image.open(path) as im:
                im.verify()
        except Exception:
            bad += 1
            continue
        shutil.copy(path, os.path.join(folder, f"up_{stamp}_{i:04d}{os.path.splitext(path)[1].lower()}"))
        ok += 1
    msg = f"Saved **{ok}** photo(s) to **{SCENE_LABELS[scene]}**." + (f" Skipped {bad} (not valid images)." if bad else "")
    return msg + " Choose another scene type to upload more, or go to the Train Model tab.", dataset_status(), None


def train_model(epochs):
    global model, ACTIVE_SCENES
    epochs = int(epochs)
    log = []

    def emit(msg):
        log.append(msg)
        return "\n".join(log)

    yield emit("Preparing dataset ...")
    if os.path.exists(CACHE_ROOT):
        shutil.rmtree(CACHE_ROOT)
    items, counts = [], {}
    for s in SCENES:
        good, bad = prepare_scene(s)
        counts[s] = len(good)
        yield emit(f"  {s:12s} usable photos: {len(good):4d}   skipped: {len(bad)}")
        for name, why in bad[:3]:
            yield emit(f"      skipped {name}: {why}")
        if 0 < len(good) < 20:
            yield emit(f"    WARNING: only {len(good)} photos for {s}; results for this scene type will be weak.")
        items += [(f, S2I[s]) for f in good]
    if len(items) < 8:
        raise gr.Error("Fewer than 8 usable colour photos in total. Upload more colour photos in the first tab.")
    active = [s for s in SCENES if counts[s] > 0]
    if len(active) < 2:
        yield emit("NOTE: only one scene type has photos, so scene detection cannot learn to tell scenes apart.")
    yield emit("Scene types that will be learned: " + ", ".join(active))

    random.shuffle(items)
    n_val = max(2, int(0.1 * len(items)))
    val_items, train_items = items[:n_val], items[n_val:]
    train_ds = SceneDataset(train_items, train=True)
    val_ds = SceneDataset(val_items, train=False)
    per_scene = {}
    for _, sid in train_items:
        per_scene[sid] = per_scene.get(sid, 0) + 1
    weights = [1.0 / per_scene[sid] for _, sid in train_items]         # every scene type is sampled equally often
    bs = min(BATCH_SIZE, len(train_ds))
    sampler = WeightedRandomSampler(weights, num_samples=len(train_ds), replacement=True)
    train_dl = DataLoader(train_ds, batch_size=bs, sampler=sampler, num_workers=NUM_WORKERS, drop_last=len(train_ds) > bs)
    val_dl = DataLoader(val_ds, batch_size=bs, shuffle=False, num_workers=NUM_WORKERS)
    yield emit(f"Training photos: {len(train_ds)} | validation photos: {len(val_ds)} | batch size: {bs}")

    vgg = None
    if PERC_W > 0:
        try:
            yield emit("Loading VGG16 for the perceptual loss (first time downloads about 0.5 GB) ...")
            vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1).features[:16].to(device).eval()
            for q in vgg.parameters():
                q.requires_grad_(False)
        except Exception as e:
            vgg = None
            yield emit(f"  perceptual loss disabled ({e})")
    vmean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
    vstd = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

    yield emit("Building model (downloads pretrained weights the first time) ...")
    net = SceneColorizer(len(SCENES), pretrained=PRETRAINED).to(device)
    use_amp = (device == "cuda")
    opt = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, epochs=epochs, steps_per_epoch=len(train_dl))
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    color_loss = nn.SmoothL1Loss()
    cls_loss = nn.CrossEntropyLoss()
    best_val = float("inf")
    t0 = time.time()
    yield emit(f"Training for {epochs} epochs on {device} ...")

    for epoch in range(epochs):
        net.train()
        tr = 0.0
        for L, ab, sid in train_dl:
            L, ab, sid = L.to(device), ab.to(device), sid.to(device)
            with torch.autocast(device_type=device, enabled=use_amp):
                pred, logits = net(L, sid)
            pred = pred.float()
            loss = COLOR_W * color_loss(pred, ab) + CLS_W * cls_loss(logits.float(), sid)
            if vgg is not None:
                rgb_pred = lab_to_rgb_t(L, pred)
                with torch.no_grad():
                    rgb_true = lab_to_rgb_t(L, ab)
                with torch.autocast(device_type=device, enabled=use_amp):
                    f_pred = vgg((rgb_pred - vmean) / vstd)
                    with torch.no_grad():
                        f_true = vgg((rgb_true - vmean) / vstd)
                loss = loss + PERC_W * F.l1_loss(f_pred.float(), f_true.float())
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            tr += loss.item()
        tr /= len(train_dl)

        net.eval()
        v_col = v_cls = 0.0
        correct = total = 0
        with torch.no_grad():
            for L, ab, sid in val_dl:
                L, ab, sid = L.to(device), ab.to(device), sid.to(device)
                pred, logits = net(L, sid)
                v_col += color_loss(pred.float(), ab).item()
                v_cls += cls_loss(logits.float(), sid).item()
                correct += (logits.argmax(1) == sid).sum().item()
                total += sid.numel()
        v_col /= len(val_dl)
        v_cls /= len(val_dl)
        acc = correct / max(1, total)
        va = COLOR_W * v_col + CLS_W * v_cls

        tag = ""
        if va < best_val:
            best_val = va
            torch.save({"model": net.state_dict(), "scenes": SCENES, "active": active}, CKPT)
            tag = "   <- best so far, saved"
        yield emit(f"epoch {epoch + 1:02d}/{epochs}   train {tr:.3f}   val colour {v_col:.4f}   "
                   f"scene accuracy {acc * 100:4.1f}%   ({time.time() - t0:.0f}s){tag}")

    ck = torch.load(CKPT, map_location=device)
    net.load_state_dict(ck["model"])
    net.eval()

    stats = {s: [0, 0, 0.0] for s in SCENES}      # correct, total, colour error sum
    with torch.no_grad():
        for L, ab, sid in val_dl:
            pred, logits = net(L.to(device), sid.to(device))
            err = (pred.float().cpu() - ab).abs().mean(dim=(1, 2, 3))
            guess = logits.argmax(1).cpu()
            for k in range(len(sid)):
                name = SCENES[int(sid[k])]
                stats[name][1] += 1
                stats[name][0] += int(guess[k] == sid[k])
                stats[name][2] += float(err[k])
    yield emit("\nPer-scene results on validation photos (scene detection accuracy | colour error, lower is better):")
    for s in SCENES:
        c, n, e = stats[s]
        if n > 0:
            yield emit(f"  {s:12s} {100.0 * c / n:5.1f}%   {e / n:.4f}   ({n} photos)")

    model = net
    ACTIVE_SCENES = active
    yield emit("\nDONE. Model saved to Drive. Go to the 'Colorize' tab.")

print("Upload and training functions ready.")

Upload and training functions ready.


In [23]:
%%ext task3
AUTO = "auto"


def _round32(v):
    return max(64, int(round(v / 32.0)) * 32)


def _prep(img):
    # PIL image -> model input tensor + full-resolution lightness
    img = ImageOps.exif_transpose(img).convert("RGB")
    W, H = img.size
    if max(W, H) > MAX_SIDE:
        k = MAX_SIDE / float(max(W, H))
        W, H = max(1, int(W * k)), max(1, int(H * k))
        img = img.resize((W, H), Image.LANCZOS)
    L_full = color.rgb2lab(np.asarray(img))[..., 0].astype(np.float32)        # 0..100 at full resolution
    k = INFER_SIDE / float(max(W, H))
    small = img.resize((_round32(W * k), _round32(H * k)), Image.BICUBIC)
    L = color.rgb2lab(np.asarray(small))[..., 0] / 50.0 - 1.0
    x = torch.from_numpy(L).float()[None, None].to(device)
    return x, L_full, H, W


def _check_ready():
    if model is None or not ACTIVE_SCENES:
        raise gr.Error("No trained model yet. Go to the 'Train Model' tab and click Start training first.")


def _scene_probs(x):
    # tensor 1 x n_scenes of probabilities (only over the scene types the model was trained on)
    lg = (model.scene_logits(x) + model.scene_logits(torch.flip(x, dims=[3]))) / 2.0      # flip test-time augmentation
    mask = torch.full_like(lg, -1e9)
    for s in ACTIVE_SCENES:
        mask[0, S2I[s]] = 0.0
    return torch.softmax(lg.float() + mask, dim=1)


@torch.no_grad()
def detect_scene(img):
    # returns {scene: probability} over the trained scene types
    _check_ready()
    model.eval()
    x, _, _, _ = _prep(img)
    p = _scene_probs(x)[0].cpu().numpy()
    return {s: float(p[S2I[s]]) for s in ACTIVE_SCENES}


@torch.no_grad()
def colorize(img, scene=AUTO, saturation=1.0, refine=0.7):
    # scene = AUTO (blend of detected scenes) or one scene name. refine: 0..1 strength of edge-aware colour refinement
    _check_ready()
    model.eval()
    x, L_full, H, W = _prep(img)
    if scene == AUTO or scene is None:
        cond = _scene_probs(x)
    else:
        if scene not in ACTIVE_SCENES:
            raise gr.Error(f"The model has no training photos for '{scene}'. Available: {', '.join(ACTIVE_SCENES)}")
        cond = torch.tensor([S2I[scene]], device=device)
    ab = model(x, cond)[0].float().cpu()                                        # 1 x 2 x h x w in [-1, 1]
    ab = F.interpolate(ab, size=(H, W), mode="bilinear", align_corners=False)[0].permute(1, 2, 0).numpy()
    if refine > 0:                                                              # snap colours to the photo's brightness edges
        guide = (L_full / 100.0).astype(np.float32)
        r = max(2, int(round(max(H, W) / 150.0)))
        smooth = np.stack([guided_filter(guide, np.ascontiguousarray(ab[..., k]).astype(np.float32), r, 1e-3)
                           for k in range(2)], axis=-1)
        ab = (1.0 - refine) * ab + refine * smooth
    ab = (ab * 128.0 * saturation).clip(-128, 127)
    lab = np.concatenate([L_full[..., None], ab], axis=-1)
    rgb = (color.lab2rgb(lab).clip(0, 1) * 255).astype(np.uint8)
    return Image.fromarray(rgb)


# load a previously trained model from Drive, if there is one
if os.path.exists(CKPT):
    try:
        ck = torch.load(CKPT, map_location=device)
        if ck["scenes"] == SCENES:
            model = SceneColorizer(len(SCENES), pretrained=False).to(device)
            model.load_state_dict(ck["model"])
            model.eval()
            ACTIVE_SCENES = ck.get("active", SCENES)
            print("Loaded saved model from Drive. Scene types:", ACTIVE_SCENES)
        else:
            print("Saved model uses a different scene list; please train again in the app.")
    except Exception as e:
        print("Could not load the saved model (", e, "). Please train again in the app.")
else:
    print("No saved model yet. Upload photos and train inside the app (next cell).")

Loaded saved model from Drive. Scene types: ['cityscape', 'forest', 'mountain', 'coast', 'countryside']


In [24]:
%%ext task3
AUTO_LABEL = "🔎 Auto-detect scene (blends detected scenes)"


def scene_choices(names):
    return [(AUTO_LABEL, AUTO)] + [(SCENE_LABELS[s], s) for s in names]


def refresh_state():
    # shows the model status and the scene choices: Auto-detect + the scene types the model was trained on
    global model, ACTIVE_SCENES
    if model is None and os.path.exists(CKPT):
        try:
            ck = torch.load(CKPT, map_location=device)
            if ck["scenes"] == SCENES:
                m = SceneColorizer(len(SCENES), pretrained=False).to(device)
                m.load_state_dict(ck["model"])
                m.eval()
                model = m
                ACTIVE_SCENES = ck.get("active", SCENES)
        except Exception as e:
            print("Could not load saved model:", e)
    if model is None or not ACTIVE_SCENES:
        return ("⚠️ **No trained model yet.** Go to tab **2. Train Model** and click **Start training** "
                "(upload colour photos in tab 1 first).",
                gr.Radio(choices=scene_choices(SCENES), value=AUTO, label="Scene type"))
    return (f"✅ **Model ready.** Scene types: {', '.join(ACTIVE_SCENES)}",
            gr.Radio(choices=scene_choices(ACTIVE_SCENES), value=AUTO, label="Scene type (auto-detected, or choose manually)"))


def run_colorize(img, scene_choice, saturation, refine):
    if img is None:
        raise gr.Error("Please upload a photo first.")
    try:
        t0 = time.time()
        probs = detect_scene(img)
        top = max(probs, key=probs.get)
        if scene_choice == AUTO:
            note = (f"Scene understanding: mostly **{SCENE_LABELS[top]}** ({probs[top] * 100:.0f}%). "
                    f"Colours were chosen from a blend of the detected scenes.")
        else:
            note = (f"Manual scene: **{SCENE_LABELS[scene_choice]}**. "
                    f"(The model's own guess was {SCENE_LABELS[top]}, {probs[top] * 100:.0f}%.)")
        res = colorize(img, scene_choice, saturation, refine)
        orig = ImageOps.exif_transpose(img).convert("RGB")
        if orig.size != res.size:
            orig = orig.resize(res.size, Image.LANCZOS)
        label_probs = {SCENE_LABELS[s]: p for s, p in probs.items()}
        return res, orig, res, label_probs, note + f"  ({time.time() - t0:.1f}s)"
    except gr.Error:
        raise
    except Exception as e:
        traceback.print_exc()
        raise gr.Error(f"{type(e).__name__}: {e}")


def compare_styles(img, saturation, refine):
    if img is None:
        raise gr.Error("Please upload a photo first.")
    try:
        probs = detect_scene(img)
        return [(colorize(img, s, saturation, refine), f"{SCENE_LABELS[s]}  ({probs[s] * 100:.0f}% match)")
                for s in ACTIVE_SCENES]
    except gr.Error:
        raise
    except Exception as e:
        traceback.print_exc()
        raise gr.Error(f"{type(e).__name__}: {e}")


HEADER = (
    "<h1 style='text-align:center; margin-bottom:0'>🏙 Context-Aware Scene Colorizer</h1>"
    "<p style='text-align:center; margin-top:4px'><b>Understands the whole scene</b> &bull; <b>Global context attention</b> &bull; "
    "<b>Edge-aware colours</b> &bull; <b>Cityscapes, forests, mountains, coasts</b></p>"
)

def build_ui(demo):
    gr.HTML(HEADER)
    with gr.Tabs():

        # ---------------- Tab 1: upload training photos ----------------
        with gr.Tab("1. Upload Training Photos"):
            gr.Markdown("Upload **real colour photos** of one scene type at a time. Select the scene type, choose the photos, "
                        "click **Save photos**, and repeat for each type. The model learns the natural colours of each kind of scene "
                        "(and how shadows, reflections and neighbouring objects relate). Use varied, sharp photos of complex scenes. "
                        "Aim for 100+ per scene type (20 minimum), and at least 2 scene types.")
            with gr.Row():
                with gr.Column():
                    up_scene = gr.Radio(choices=[(SCENE_LABELS[s], s) for s in SCENES], value=SCENES[0], label="Scene type of these photos")
                    up_files = gr.File(label="Choose colour photos", file_count="multiple", file_types=["image"])
                    up_btn = gr.Button("Save photos", variant="primary")
                    up_msg = gr.Markdown()
                with gr.Column():
                    ds_md = gr.Markdown(dataset_status())
                    ds_btn = gr.Button("Refresh counts")
            up_btn.click(save_uploads, inputs=[up_scene, up_files], outputs=[up_msg, ds_md, up_files])
            ds_btn.click(dataset_status, outputs=ds_md)

        # ---------------- Tab 2: train ----------------
        with gr.Tab("2. Train Model"):
            gr.Markdown("Click **Start training** after uploading photos. It takes roughly 10-30 minutes on a T4 GPU "
                        "(longer than the simpler projects because of the context blocks and the perceptual loss). "
                        "For a quick test, use 8 epochs. The best model is saved to Google Drive automatically.")
            epochs_sl = gr.Slider(5, 60, value=25, step=1, label="Epochs")
            train_btn = gr.Button("Start training", variant="primary")
            log_box = gr.Textbox(label="Training log", lines=16, max_lines=16, interactive=False)

        # ---------------- Tab 3: colorize ----------------
        with gr.Tab("3. Colorize"):
            model_md = gr.Markdown()
            with gr.Row():
                with gr.Column(scale=1):
                    inp = gr.Image(type="pil", label="📷 Upload B&W Photo", height=380)
                    scene_dd = gr.Radio(choices=scene_choices(SCENES), value=AUTO, label="Scene type (auto-detected, or choose manually)")
                    sat = gr.Slider(0.5, 2.0, value=1.2, step=0.1, label="Colour strength")
                    refine_sl = gr.Slider(0.0, 1.0, value=0.7, step=0.05, label="Edge-aware colour refinement")
                    btn = gr.Button("🎨 Colorize", variant="primary")
                    btn_all = gr.Button("Compare scene styles")
                    refresh_btn = gr.Button("Refresh model status")
                with gr.Column(scale=2):
                    status = gr.Markdown("Upload a photo and click **Colorize**. Leave *Auto-detect* on, or pick a scene type to override it.")
                    with gr.Tabs():
                        with gr.Tab("🌈 Colorized Result"):
                            out_main = gr.Image(type="pil", label="Colorized", height=520)
                        with gr.Tab("Side by Side Comparison"):
                            gr.Markdown("<p style='text-align:center'><b>Original (left) | Colorized (right)</b></p>")
                            with gr.Row():
                                out_orig = gr.Image(type="pil", label="Original", height=480)
                                out_col = gr.Image(type="pil", label="Colorized", height=480)
                        with gr.Tab("Scene Understanding"):
                            out_probs = gr.Label(label="How much the photo matches each scene type", num_top_classes=len(SCENES))
                        with gr.Tab("Compare Scene Styles"):
                            gallery = gr.Gallery(label="Same photo with each scene style", columns=3)

    train_btn.click(train_model, inputs=epochs_sl, outputs=log_box).then(refresh_state, None, [model_md, scene_dd])
    refresh_btn.click(refresh_state, None, [model_md, scene_dd])
    btn.click(run_colorize, inputs=[inp, scene_dd, sat, refine_sl], outputs=[out_main, out_orig, out_col, out_probs, status])
    btn_all.click(compare_styles, inputs=[inp, sat, refine_sl], outputs=gallery)
    demo.load(refresh_state, None, [model_md, scene_dd])

In [25]:
%%ext task4
import os, io, glob, time, shutil, random, traceback
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms as T
from PIL import Image, ImageOps
from skimage import color
import gradio as gr

# ------------------------- Google Drive -------------------------
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = "/content/drive/MyDrive/era_colorization"      # photos + trained model are saved here
except Exception as e:
    print("Could not mount Google Drive (", e, ")\nUsing temporary Colab storage - files will be lost when the session ends.")
    DATA_ROOT = "/content/era_colorization"

# ------------------------- eras (folder names) -------------------------
# You may edit this list BEFORE the first training. If you change it later, train again.
PERIODS = ["1900s-1910s", "1920s", "1930s", "1940s", "1950s"]
P2I = {p: i for i, p in enumerate(PERIODS)}

# ------------------------- settings -------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE = 256        # training size (divisible by 32)
INFER_SIDE = 256      # longer side used by the model when colorizing
MAX_SIDE = 2048       # very large uploads are shrunk to this
MIN_SAT = 0.03        # photos with lower average saturation are treated as black & white and skipped in training
BATCH_SIZE = 16
LR = 3e-4
COLOR_W = 10.0        # weight of the colour loss
CLS_W = 1.0           # weight of the era-classification loss
NUM_WORKERS = 2
PRETRAINED = True     # ImageNet weights for the encoder (downloaded automatically)

EXTS = (".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp", ".webp")
CACHE_ROOT = "/content/era_cache"                       # resized copies for fast training (temporary)
CKPT = os.path.join(DATA_ROOT, "era_colorizer.pt")       # trained model

for p in PERIODS:
    os.makedirs(os.path.join(DATA_ROOT, p), exist_ok=True)
random.seed(0); np.random.seed(0); torch.manual_seed(0)
print("Device:", device)
print("Data folder:", DATA_ROOT)
print("Eras:", PERIODS)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
Data folder: /content/drive/MyDrive/era_colorization
Eras: ['1900s-1910s', '1920s', '1930s', '1940s', '1950s']


In [26]:
%%ext task4
def prepare_period(p):
    # reads photos of one era, skips bad / black & white ones, saves resized copies for fast training
    out_dir = os.path.join(CACHE_ROOT, p)
    os.makedirs(out_dir, exist_ok=True)
    good, bad = [], []
    for f in sorted(glob.glob(os.path.join(DATA_ROOT, p, "*"))):
        if not f.lower().endswith(EXTS):
            continue
        try:
            with Image.open(f) as im:
                try:
                    im.draft("RGB", (1024, 1024))
                except Exception:
                    pass
                im = ImageOps.exif_transpose(im).convert("RGB")
            w, h = im.size
            if min(w, h) < 64:
                bad.append((os.path.basename(f), "too small"))
                continue
            sat = np.asarray(im.resize((64, 64)).convert("HSV"))[..., 1].mean() / 255.0
            if sat < MIN_SAT:
                bad.append((os.path.basename(f), "looks black & white (training photos must be colour)"))
                continue
            s = 320.0 / min(w, h)
            if s < 1.0:
                im = im.resize((max(1, int(round(w * s))), max(1, int(round(h * s)))), Image.LANCZOS)
            dst = os.path.join(out_dir, f"{p}_{len(good):05d}.jpg")
            im.save(dst, quality=95)
            good.append(dst)
        except Exception as e:
            bad.append((os.path.basename(f), f"unreadable: {e}"))
    return good, bad


class EraDataset(Dataset):
    def __init__(self, items, train=True):
        self.items = items            # list of (path, era_id)
        self.train = train
        if train:
            self.tf = T.Compose([T.RandomResizedCrop(IMG_SIZE, scale=(0.4, 1.0)), T.RandomHorizontalFlip()])
        else:
            self.tf = T.Compose([T.Resize(IMG_SIZE), T.CenterCrop(IMG_SIZE)])

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        path, pid = self.items[i]
        with Image.open(path) as im:
            img = self.tf(im.convert("RGB"))
        lab = color.rgb2lab(np.asarray(img)).astype("float32")
        L = torch.from_numpy(lab[..., 0:1] / 50.0 - 1.0).permute(2, 0, 1)     # lightness -> [-1, 1]
        ab = torch.from_numpy(lab[..., 1:] / 128.0).permute(2, 0, 1)          # colour    -> ~[-1, 1]
        if self.train:
            # make the grayscale input look like old photographs: contrast / brightness changes and film grain
            L = L * random.uniform(0.75, 1.25) + random.uniform(-0.15, 0.15)
            if random.random() < 0.5:
                L = L + torch.randn_like(L) * random.uniform(0.01, 0.05)
            L = L.clamp(-1.0, 1.0)
        return L, ab, pid


class FiLM(nn.Module):
    # scales and shifts feature maps depending on the selected era
    def __init__(self, emb_dim, ch):
        super().__init__()
        self.fc = nn.Linear(emb_dim, ch * 2)
        nn.init.zeros_(self.fc.weight)
        nn.init.zeros_(self.fc.bias)

    def forward(self, x, e):
        g, b = self.fc(e).chunk(2, dim=1)
        return x * (1 + g[:, :, None, None]) + b[:, :, None, None]


class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, emb_dim):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch + skip_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(True))
        self.film = FiLM(emb_dim, out_ch)

    def forward(self, x, skip, e):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = self.conv(torch.cat([x, skip], dim=1))
        return self.film(x, e)


class EraColorizer(nn.Module):
    # grayscale (L) + era  ->  colour (ab) ;  grayscale (L)  ->  era logits
    def __init__(self, n_periods, emb_dim=64, pretrained=True):
        super().__init__()
        r = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None)
        self.stem = nn.Sequential(r.conv1, r.bn1, r.relu)
        self.pool = r.maxpool
        self.l1, self.l2, self.l3, self.l4 = r.layer1, r.layer2, r.layer3, r.layer4
        self.emb = nn.Embedding(n_periods, emb_dim)
        self.film4 = FiLM(emb_dim, 512)
        self.up3 = UpBlock(512, 256, 256, emb_dim)
        self.up2 = UpBlock(256, 128, 128, emb_dim)
        self.up1 = UpBlock(128, 64, 64, emb_dim)
        self.up0 = UpBlock(64, 64, 64, emb_dim)
        self.head = nn.Sequential(nn.Conv2d(64, 32, 3, padding=1), nn.ReLU(True),
                                  nn.Conv2d(32, 2, 1), nn.Tanh())
        # era classifier on the deepest encoder features
        self.cls = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(512, n_periods))

    def encode(self, L):
        x = L.repeat(1, 3, 1, 1)
        s0 = self.stem(x)
        s1 = self.l1(self.pool(s0))
        s2 = self.l2(s1)
        s3 = self.l3(s2)
        s4 = self.l4(s3)
        return s0, s1, s2, s3, s4

    def era_logits(self, L):
        return self.cls(self.encode(L)[4])

    def forward(self, L, period):
        s0, s1, s2, s3, s4 = self.encode(L)
        logits = self.cls(s4)                      # what era does this photo look like?
        e = self.emb(period)                       # which era's palette should be used?
        d = self.up3(self.film4(s4, e), s3, e)
        d = self.up2(d, s2, e)
        d = self.up1(d, s1, e)
        d = self.up0(d, s0, e)
        d = F.interpolate(d, size=L.shape[-2:], mode="bilinear", align_corners=False)
        return self.head(d), logits                # ab in [-1, 1], era logits

print("Dataset and model code ready.")

Dataset and model code ready.


In [27]:
%%ext task4
model = None            # trained model (filled by training, or loaded from Drive in Cell 5)
ACTIVE_PERIODS = []     # eras the model was trained on


def count_photos(p):
    folder = os.path.join(DATA_ROOT, p)
    if not os.path.isdir(folder):
        return 0
    return len([f for f in os.listdir(folder) if f.lower().endswith(EXTS)])


def dataset_status():
    rows = ["| Era | Photos saved in Drive |", "|---|---|"]
    for p in PERIODS:
        n = count_photos(p)
        flag = "" if n >= 20 else ("  (need more, 20+ recommended)" if n > 0 else "  (none yet)")
        rows.append(f"| {p} | {n}{flag} |")
    return "\n".join(rows)


def save_uploads(period, uploaded):
    if not uploaded:
        return "No photos selected. Click the upload box, choose photos, then click Save.", dataset_status(), None
    folder = os.path.join(DATA_ROOT, period)
    os.makedirs(folder, exist_ok=True)
    ok = bad = 0
    stamp = int(time.time())
    for i, f in enumerate(uploaded):
        path = f if isinstance(f, str) else getattr(f, "name", None)
        if path is None or os.path.splitext(path)[1].lower() not in EXTS:
            bad += 1
            continue
        try:
            with Image.open(path) as im:
                im.verify()
        except Exception:
            bad += 1
            continue
        shutil.copy(path, os.path.join(folder, f"up_{stamp}_{i:04d}{os.path.splitext(path)[1].lower()}"))
        ok += 1
    msg = f"Saved **{ok}** photo(s) to **{period}**." + (f" Skipped {bad} (not valid images)." if bad else "")
    return msg + " Choose another era to upload more, or go to the Train Model tab.", dataset_status(), None


def train_model(epochs):
    global model, ACTIVE_PERIODS
    epochs = int(epochs)
    log = []

    def emit(msg):
        log.append(msg)
        return "\n".join(log)

    yield emit("Preparing dataset ...")
    if os.path.exists(CACHE_ROOT):
        shutil.rmtree(CACHE_ROOT)
    items, counts = [], {}
    for p in PERIODS:
        good, bad = prepare_period(p)
        counts[p] = len(good)
        yield emit(f"  {p:12s} usable photos: {len(good):4d}   skipped: {len(bad)}")
        for name, why in bad[:3]:
            yield emit(f"      skipped {name}: {why}")
        if 0 < len(good) < 20:
            yield emit(f"    WARNING: only {len(good)} photos for {p}; results for this era will be weak.")
        items += [(f, P2I[p]) for f in good]
    if len(items) < 8:
        raise gr.Error("Fewer than 8 usable colour photos in total. Upload more colour photos in the first tab.")
    active = [p for p in PERIODS if counts[p] > 0]
    if len(active) < 2:
        yield emit("NOTE: only one era has photos, so era detection cannot learn to tell eras apart. Upload at least 2 eras.")
    yield emit("Eras that will be learned: " + ", ".join(active))

    random.shuffle(items)
    n_val = max(2, int(0.1 * len(items)))
    val_items, train_items = items[:n_val], items[n_val:]
    train_ds = EraDataset(train_items, train=True)
    val_ds = EraDataset(val_items, train=False)
    per_era = {}
    for _, pid in train_items:
        per_era[pid] = per_era.get(pid, 0) + 1
    weights = [1.0 / per_era[pid] for _, pid in train_items]          # every era is sampled equally often
    bs = min(BATCH_SIZE, len(train_ds))
    sampler = WeightedRandomSampler(weights, num_samples=len(train_ds), replacement=True)
    train_dl = DataLoader(train_ds, batch_size=bs, sampler=sampler, num_workers=NUM_WORKERS, drop_last=len(train_ds) > bs)
    val_dl = DataLoader(val_ds, batch_size=bs, shuffle=False, num_workers=NUM_WORKERS)
    yield emit(f"Training photos: {len(train_ds)} | validation photos: {len(val_ds)} | batch size: {bs}")

    yield emit("Building model (downloads pretrained weights the first time) ...")
    net = EraColorizer(len(PERIODS), pretrained=PRETRAINED).to(device)
    use_amp = (device == "cuda")
    opt = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, epochs=epochs, steps_per_epoch=len(train_dl))
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    color_loss = nn.SmoothL1Loss()
    cls_loss = nn.CrossEntropyLoss()
    best_val = float("inf")
    t0 = time.time()
    yield emit(f"Training for {epochs} epochs on {device} ...")

    for epoch in range(epochs):
        net.train()
        tr = 0.0
        for L, ab, pid in train_dl:
            L, ab, pid = L.to(device), ab.to(device), pid.to(device)
            with torch.autocast(device_type=device, enabled=use_amp):
                pred, logits = net(L, pid)
            loss = COLOR_W * color_loss(pred.float(), ab) + CLS_W * cls_loss(logits.float(), pid)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            tr += loss.item()
        tr /= len(train_dl)

        net.eval()
        v_col = v_cls = 0.0
        correct = total = 0
        with torch.no_grad():
            for L, ab, pid in val_dl:
                L, ab, pid = L.to(device), ab.to(device), pid.to(device)
                pred, logits = net(L, pid)
                v_col += color_loss(pred.float(), ab).item()
                v_cls += cls_loss(logits.float(), pid).item()
                correct += (logits.argmax(1) == pid).sum().item()
                total += pid.numel()
        v_col /= len(val_dl)
        v_cls /= len(val_dl)
        acc = correct / max(1, total)
        va = COLOR_W * v_col + CLS_W * v_cls

        tag = ""
        if va < best_val:
            best_val = va
            torch.save({"model": net.state_dict(), "periods": PERIODS, "active": active}, CKPT)
            tag = "   <- best so far, saved"
        yield emit(f"epoch {epoch + 1:02d}/{epochs}   train {tr:.3f}   val colour {v_col:.4f}   "
                   f"era accuracy {acc * 100:4.1f}%   ({time.time() - t0:.0f}s){tag}")

    ck = torch.load(CKPT, map_location=device)
    net.load_state_dict(ck["model"])
    net.eval()

    # per-era validation results of the best model
    stats = {p: [0, 0, 0.0] for p in PERIODS}      # correct, total, colour error sum
    with torch.no_grad():
        for L, ab, pid in val_dl:
            pred, logits = net(L.to(device), pid.to(device))
            pred, pid_c = pred.float().cpu(), pid
            err = (pred - ab).abs().mean(dim=(1, 2, 3))
            guess = logits.argmax(1).cpu()
            for k in range(len(pid_c)):
                name = PERIODS[int(pid_c[k])]
                stats[name][1] += 1
                stats[name][0] += int(guess[k] == pid_c[k])
                stats[name][2] += float(err[k])
    yield emit("\nPer-era results on validation photos (era detection accuracy | colour error, lower is better):")
    for p in PERIODS:
        c, n, e = stats[p]
        if n > 0:
            yield emit(f"  {p:12s} {100.0 * c / n:5.1f}%   {e / n:.4f}   ({n} photos)")
    yield emit("Note: era detection from grayscale photos is hard. If accuracy is low, add more varied photos per era "
               "(different places, people, photographers) and use the manual era option.")

    model = net
    ACTIVE_PERIODS = active
    yield emit(f"\nDONE. Model saved to Drive. Go to the 'Colorize' tab.")

print("Upload and training functions ready.")

Upload and training functions ready.


In [28]:
%%ext task4
def _round32(v):
    return max(64, int(round(v / 32.0)) * 32)


def _prep(img):
    # PIL image -> model input tensor + full-resolution lightness
    img = ImageOps.exif_transpose(img).convert("RGB")
    W, H = img.size
    if max(W, H) > MAX_SIDE:
        s = MAX_SIDE / float(max(W, H))
        W, H = max(1, int(W * s)), max(1, int(H * s))
        img = img.resize((W, H), Image.LANCZOS)
    L_full = color.rgb2lab(np.asarray(img))[..., 0]                    # 0..100 at full resolution
    s = INFER_SIDE / float(max(W, H))
    small = img.resize((_round32(W * s), _round32(H * s)), Image.BICUBIC)
    L = color.rgb2lab(np.asarray(small))[..., 0] / 50.0 - 1.0
    x = torch.from_numpy(L).float()[None, None].to(device)
    return x, L_full, H, W


def _check_ready():
    if model is None or not ACTIVE_PERIODS:
        raise gr.Error("No trained model yet. Go to the 'Train Model' tab and click Start training first.")


@torch.no_grad()
def detect_era(img):
    # returns {era: probability} over the eras the model was trained on
    _check_ready()
    model.eval()
    x, _, _, _ = _prep(img)
    lg = (model.era_logits(x) + model.era_logits(torch.flip(x, dims=[3]))) / 2.0     # flip test-time augmentation
    lg = lg.float().cpu()[0]
    mask = torch.full_like(lg, -1e9)
    for p in ACTIVE_PERIODS:
        mask[P2I[p]] = 0.0
    probs = torch.softmax(lg + mask, dim=0).numpy()
    return {p: float(probs[P2I[p]]) for p in ACTIVE_PERIODS}


@torch.no_grad()
def colorize(img, period, saturation=1.0):
    # colorizes with the palette of the given era; keeps the original brightness (Lab colour space)
    _check_ready()
    if period not in ACTIVE_PERIODS:
        raise gr.Error(f"The model has no training photos for '{period}'. Available: {', '.join(ACTIVE_PERIODS)}")
    model.eval()
    x, L_full, H, W = _prep(img)
    pid = torch.tensor([P2I[period]], device=device)
    ab = model(x, pid)[0].float().cpu()
    ab = F.interpolate(ab, size=(H, W), mode="bilinear", align_corners=False)[0]
    ab = (ab.permute(1, 2, 0).numpy() * 128.0 * saturation).clip(-128, 127)
    lab = np.concatenate([L_full[..., None], ab], axis=-1)
    rgb = (color.lab2rgb(lab).clip(0, 1) * 255).astype(np.uint8)
    return Image.fromarray(rgb)


# load a previously trained model from Drive, if there is one
if os.path.exists(CKPT):
    try:
        ck = torch.load(CKPT, map_location=device)
        if ck["periods"] == PERIODS:
            model = EraColorizer(len(PERIODS), pretrained=False).to(device)
            model.load_state_dict(ck["model"])
            model.eval()
            ACTIVE_PERIODS = ck.get("active", PERIODS)
            print("Loaded saved model from Drive. Eras:", ACTIVE_PERIODS)
        else:
            print("Saved model uses a different era list; please train again in the app.")
    except Exception as e:
        print("Could not load the saved model (", e, "). Please train again in the app.")
else:
    print("No saved model yet. Upload photos and train inside the app (next cell).")

Loaded saved model from Drive. Eras: ['1900s-1910s', '1920s', '1930s', '1940s', '1950s']


In [29]:
%%ext task4
AUTO = "auto"


def refresh_state():
    # shows the model status and the era choices: Auto-detect + the eras the model was trained on
    global model, ACTIVE_PERIODS
    if model is None and os.path.exists(CKPT):
        try:
            ck = torch.load(CKPT, map_location=device)
            if ck["periods"] == PERIODS:
                m = EraColorizer(len(PERIODS), pretrained=False).to(device)
                m.load_state_dict(ck["model"])
                m.eval()
                model = m
                ACTIVE_PERIODS = ck.get("active", PERIODS)
        except Exception as e:
            print("Could not load saved model:", e)
    if model is None or not ACTIVE_PERIODS:
        return ("⚠️ **No trained model yet.** Go to tab **2. Train Model** and click **Start training** "
                "(upload colour photos in tab 1 first).",
                gr.Radio(choices=[("🔎 Auto-detect era (recommended)", AUTO)] + [(p, p) for p in PERIODS],
                         value=AUTO, label="Era"))
    return (f"✅ **Model ready.** Eras: {', '.join(ACTIVE_PERIODS)}",
            gr.Radio(choices=[("🔎 Auto-detect era (recommended)", AUTO)] + [(p, p) for p in ACTIVE_PERIODS],
                     value=AUTO, label="Era (auto-detected, or choose manually)"))


def run_colorize(img, era_choice, saturation):
    if img is None:
        raise gr.Error("Please upload a photo first.")
    try:
        t0 = time.time()
        probs = detect_era(img)
        top = max(probs, key=probs.get)
        if era_choice == AUTO:
            used = top
            note = f"Auto-detected era: **{top}** ({probs[top] * 100:.0f}% confidence). Palette used: **{used}**."
            if probs[top] < 0.4:
                note += " ⚠️ Low confidence - consider choosing the era manually."
        else:
            used = era_choice
            note = (f"Manual era: **{used}**. (The model's own guess was {top}, {probs[top] * 100:.0f}%.)")
        res = colorize(img, used, saturation)
        orig = ImageOps.exif_transpose(img).convert("RGB")
        if orig.size != res.size:
            orig = orig.resize(res.size, Image.LANCZOS)
        return res, orig, res, probs, note + f"  ({time.time() - t0:.1f}s)"
    except gr.Error:
        raise
    except Exception as e:
        traceback.print_exc()
        raise gr.Error(f"{type(e).__name__}: {e}")


def compare_all(img, saturation):
    if img is None:
        raise gr.Error("Please upload a photo first.")
    try:
        probs = detect_era(img)
        return [(colorize(img, p, saturation), f"{p}  ({probs[p] * 100:.0f}% match)") for p in ACTIVE_PERIODS]
    except gr.Error:
        raise
    except Exception as e:
        traceback.print_exc()
        raise gr.Error(f"{type(e).__name__}: {e}")


HEADER = (
    "<h1 style='text-align:center; margin-bottom:0'>🕰 Time-Based Historical Photo Colorizer</h1>"
    "<p style='text-align:center; margin-top:4px'><b>Detects the era</b> &bull; <b>Uses that era's colour palette</b> &bull; "
    "<b>Change the era manually</b> &bull; <b>Compare all eras</b></p>"
)

def build_ui(demo):
    gr.HTML(HEADER)
    with gr.Tabs():

        # ---------------- Tab 1: upload training photos ----------------
        with gr.Tab("1. Upload Training Photos"):
            gr.Markdown("Upload **real colour photos** from one era at a time (autochromes, Kodachrome, colour scans). "
                        "Select the era, choose the photos, click **Save photos**. Repeat for each era. "
                        "The model learns each era's palette, and learns to recognize the era from the photo's look. "
                        "Use varied photos (different places, people, photographers). Aim for 100+ per era (20 minimum), "
                        "and at least 2 eras so era detection can work.")
            with gr.Row():
                with gr.Column():
                    up_period = gr.Radio(choices=PERIODS, value=PERIODS[0], label="🕰 Era of these photos")
                    up_files = gr.File(label="Choose colour photos", file_count="multiple", file_types=["image"])
                    up_btn = gr.Button("Save photos", variant="primary")
                    up_msg = gr.Markdown()
                with gr.Column():
                    ds_md = gr.Markdown(dataset_status())
                    ds_btn = gr.Button("Refresh counts")
            up_btn.click(save_uploads, inputs=[up_period, up_files], outputs=[up_msg, ds_md, up_files])
            ds_btn.click(dataset_status, outputs=ds_md)

        # ---------------- Tab 2: train ----------------
        with gr.Tab("2. Train Model"):
            gr.Markdown("Click **Start training** after uploading photos. It takes roughly 5-20 minutes on a T4 GPU. "
                        "For a quick test, use 8 epochs. The best model is saved to Google Drive automatically.")
            epochs_sl = gr.Slider(5, 60, value=25, step=1, label="Epochs")
            train_btn = gr.Button("Start training", variant="primary")
            log_box = gr.Textbox(label="Training log", lines=16, max_lines=16, interactive=False)

        # ---------------- Tab 3: colorize ----------------
        with gr.Tab("3. Colorize"):
            model_md = gr.Markdown()
            with gr.Row():
                with gr.Column(scale=1):
                    inp = gr.Image(type="pil", label="📷 Upload B&W Photo", height=380)
                    era_dd = gr.Radio(choices=[("🔎 Auto-detect era (recommended)", AUTO)] + [(p, p) for p in PERIODS],
                                      value=AUTO, label="Era (auto-detected, or choose manually)")
                    sat = gr.Slider(0.5, 2.0, value=1.2, step=0.1, label="Colour strength")
                    btn = gr.Button("🎨 Colorize", variant="primary")
                    btn_all = gr.Button("Compare all eras")
                    refresh_btn = gr.Button("Refresh model status")
                with gr.Column(scale=2):
                    status = gr.Markdown("Upload a photo and click **Colorize**. Leave *Auto-detect* on, or pick an era to override it.")
                    with gr.Tabs():
                        with gr.Tab("🌈 Colorized Result"):
                            out_main = gr.Image(type="pil", label="Colorized", height=520)
                        with gr.Tab("Side by Side Comparison"):
                            gr.Markdown("<p style='text-align:center'><b>Original (left) | Colorized (right)</b></p>")
                            with gr.Row():
                                out_orig = gr.Image(type="pil", label="Original", height=480)
                                out_col = gr.Image(type="pil", label="Colorized", height=480)
                        with gr.Tab("Era Detection"):
                            out_probs = gr.Label(label="How much the photo matches each era", num_top_classes=len(PERIODS))
                        with gr.Tab("Compare All Eras"):
                            gallery = gr.Gallery(label="Same photo in every era", columns=3)

    train_btn.click(train_model, inputs=epochs_sl, outputs=log_box).then(refresh_state, None, [model_md, era_dd])
    refresh_btn.click(refresh_state, None, [model_md, era_dd])
    btn.click(run_colorize, inputs=[inp, era_dd, sat], outputs=[out_main, out_orig, out_col, out_probs, status])
    btn_all.click(compare_all, inputs=[inp, sat], outputs=gallery)
    demo.load(refresh_state, None, [model_md, era_dd])

In [30]:
%%ext task5
import os, io, glob, time, shutil, random, traceback
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms as T
import torchvision.transforms.functional as TF
from PIL import Image, ImageOps, ImageFilter
from skimage import color
import gradio as gr

# ------------------------- Google Drive -------------------------
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = "/content/drive/MyDrive/cross_domain_colorization"      # data + trained model are saved here
except Exception as e:
    print("Could not mount Google Drive (", e, ")\nUsing temporary Colab storage - files will be lost when the session ends.")
    DATA_ROOT = "/content/cross_domain_colorization"

# ------------------------- domains -------------------------
DOMAINS = ["photo", "sketch", "infrared"]
D2I = {d: i for i, d in enumerate(DOMAINS)}
DOMAIN_LABELS = {"photo": "📷 Photo (black & white photograph)",
                 "sketch": "✏️ Sketch (black & white line art)",
                 "infrared": "🛰 Infrared satellite image"}

# ------------------------- settings -------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE = 256        # training size (divisible by 32)
INFER_SIDE = 256      # longer side used by the model when colorizing
MAX_SIDE = 2048       # very large uploads are shrunk to this
MIN_SAT = 0.03        # target images with lower average saturation are treated as black & white and skipped
BATCH_SIZE = 16
LR = 3e-4
NUM_WORKERS = 2
PRETRAINED = True     # ImageNet weights for the encoder (downloaded automatically)

EXTS = (".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp", ".webp")
CACHE_ROOT = "/content/xd_cache"                       # resized copies for fast training (temporary)
CKPT = os.path.join(DATA_ROOT, "xdomain_colorizer.pt")  # trained model

for d in DOMAINS:
    os.makedirs(os.path.join(DATA_ROOT, d, "target"), exist_ok=True)
    os.makedirs(os.path.join(DATA_ROOT, d, "input"), exist_ok=True)
random.seed(0); np.random.seed(0); torch.manual_seed(0)
print("Device:", device)
print("Data folder:", DATA_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
Data folder: /content/drive/MyDrive/cross_domain_colorization


In [31]:
%%ext task5
# ---------------- domain-specific image handling ----------------
def to_uint8_gray(im):
    # any PIL image (including 16-bit infrared TIFF/PNG) -> 8-bit grayscale PIL image
    if im.mode in ("I;16", "I;16B", "I;16L", "I", "F"):
        a = np.asarray(im).astype(np.float32)
        lo, hi = np.percentile(a, (1, 99))
        a = np.clip((a - lo) / (hi - lo + 1e-6), 0, 1) * 255.0
        return Image.fromarray(a.astype(np.uint8))
    return im.convert("L")


def normalize_input(domain, g8):
    # domain-specific preparation of the 1-channel input. g8: uint8 HxW array -> float32 HxW in [0, 1]
    g = g8.astype(np.float32) / 255.0
    if domain == "sketch":
        if g.mean() < 0.5:                       # white lines on black -> black lines on white
            g = 1.0 - g
    elif domain == "infrared":
        lo, hi = np.percentile(g, (2, 98))       # contrast stretch
        g = np.clip((g - lo) / (hi - lo + 1e-6), 0, 1)
    return g.astype(np.float32)


def make_sketch(img):
    # pencil-sketch effect from a colour image (colour-dodge of the gray image with its blurred negative)
    g = np.asarray(img.convert("L")).astype(np.float32)
    inv = Image.fromarray((255.0 - g).astype(np.uint8))
    blur = np.asarray(inv.filter(ImageFilter.GaussianBlur(radius=random.uniform(4, 12)))).astype(np.float32)
    s = np.clip(g * 255.0 / np.maximum(255.0 - blur, 1.0), 0, 255) / 255.0
    s = s ** random.uniform(1.0, 2.0)            # darker, more pencil-like lines
    return (np.clip(s, 0, 1) * 255).astype(np.uint8)


def make_pseudo_ir(img):
    # rough simulation of a near-infrared band from a colour satellite image:
    # vegetation bright, water dark. (Real paired infrared images give much better results.)
    a = np.asarray(img).astype(np.float32) / 255.0
    R, G, B = a[..., 0], a[..., 1], a[..., 2]
    lum = 0.299 * R + 0.587 * G + 0.114 * B
    green = np.clip(2 * G - R - B, 0, 1)
    water = np.clip(B - R, 0, 1) * (1 - lum)
    nir = 0.6 * lum + 1.2 * green - 0.8 * water
    nir = nir + np.random.default_rng().normal(0, 0.02, nir.shape)
    return (np.clip(nir, 0, 1) * 255).astype(np.uint8)


def synth_input(domain, img):
    # makes the model input from a colour target image when no real input image was uploaded
    if domain == "photo":
        g = np.asarray(img.convert("L")).astype(np.float32)
        c, b = random.uniform(0.7, 1.2), random.uniform(-20, 20)
        return np.clip((g - 128) * c + 128 + b, 0, 255).astype(np.uint8)
    if domain == "sketch":
        return make_sketch(img)
    return make_pseudo_ir(img)


# ---------------- dataset preparation ----------------
def prepare_domain(d):
    # reads colour targets (+ optional paired inputs, matched by file name) of one domain and caches resized copies
    tdir = os.path.join(DATA_ROOT, d, "target")
    idir = os.path.join(DATA_ROOT, d, "input")
    out_dir = os.path.join(CACHE_ROOT, d)
    os.makedirs(out_dir, exist_ok=True)
    inputs = {}
    if os.path.isdir(idir):
        for f in os.listdir(idir):
            if f.lower().endswith(EXTS):
                inputs[os.path.splitext(f)[0]] = os.path.join(idir, f)
    pairs, bad, n_paired = [], [], 0
    names = sorted(os.listdir(tdir)) if os.path.isdir(tdir) else []
    for f in names:
        if not f.lower().endswith(EXTS):
            continue
        stem = os.path.splitext(f)[0]
        try:
            with Image.open(os.path.join(tdir, f)) as im:
                try:
                    im.draft("RGB", (1024, 1024))
                except Exception:
                    pass
                im = ImageOps.exif_transpose(im).convert("RGB")
            w, h = im.size
            if min(w, h) < 64:
                bad.append((f, "too small"))
                continue
            sat = np.asarray(im.resize((64, 64)).convert("HSV"))[..., 1].mean() / 255.0
            if sat < MIN_SAT:
                bad.append((f, "looks black & white (target images must be colour)"))
                continue
            s = 320.0 / min(w, h)
            if s < 1.0:
                im = im.resize((max(1, int(round(w * s))), max(1, int(round(h * s)))), Image.LANCZOS)
            idx = len(pairs)
            tdst = os.path.join(out_dir, f"{d}_{idx:05d}.jpg")
            im.save(tdst, quality=95)
            idst = None
            if stem in inputs:
                try:
                    with Image.open(inputs[stem]) as gi:
                        g = to_uint8_gray(ImageOps.exif_transpose(gi))
                    g = g.resize(im.size, Image.BILINEAR)
                    idst = os.path.join(out_dir, f"{d}_{idx:05d}_in.png")
                    g.save(idst)
                    n_paired += 1
                except Exception:
                    idst = None
            pairs.append((tdst, idst))
        except Exception as e:
            bad.append((f, f"unreadable: {e}"))
    return pairs, bad, n_paired


class DomainDataset(Dataset):
    def __init__(self, items, train=True):
        self.items = items            # list of (target_path, input_path_or_None, domain_id)
        self.train = train

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        tpath, ipath, did = self.items[i]
        domain = DOMAINS[did]
        with Image.open(tpath) as im:
            tgt = im.convert("RGB")
        inp = None
        if ipath:
            with Image.open(ipath) as im:
                inp = im.convert("L")
        if self.train:
            i0, j0, h0, w0 = T.RandomResizedCrop.get_params(tgt, scale=[0.4, 1.0], ratio=[3 / 4, 4 / 3])
            tgt = TF.resized_crop(tgt, i0, j0, h0, w0, [IMG_SIZE, IMG_SIZE])
            if inp is not None:
                inp = TF.resized_crop(inp, i0, j0, h0, w0, [IMG_SIZE, IMG_SIZE])
            if random.random() < 0.5:
                tgt = TF.hflip(tgt)
                if inp is not None:
                    inp = TF.hflip(inp)
        else:
            tgt = TF.center_crop(TF.resize(tgt, IMG_SIZE), IMG_SIZE)
            if inp is not None:
                inp = TF.center_crop(TF.resize(inp, IMG_SIZE), IMG_SIZE)
        g8 = np.asarray(inp) if inp is not None else synth_input(domain, tgt)
        g = normalize_input(domain, g8)
        x = torch.from_numpy(g * 2.0 - 1.0).float()[None]                                   # 1 x H x W in [-1, 1]
        y = torch.from_numpy(np.asarray(tgt).astype(np.float32) / 127.5 - 1.0).permute(2, 0, 1)   # 3 x H x W
        return x, y, did


# ---------------- model ----------------
class FiLM(nn.Module):
    # scales and shifts feature maps depending on the selected domain
    def __init__(self, emb_dim, ch):
        super().__init__()
        self.fc = nn.Linear(emb_dim, ch * 2)
        nn.init.zeros_(self.fc.weight)
        nn.init.zeros_(self.fc.bias)

    def forward(self, x, e):
        g, b = self.fc(e).chunk(2, dim=1)
        return x * (1 + g[:, :, None, None]) + b[:, :, None, None]


class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, emb_dim):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch + skip_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(True))
        self.film = FiLM(emb_dim, out_ch)

    def forward(self, x, skip, e):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = self.conv(torch.cat([x, skip], dim=1))
        return self.film(x, e)


class DomainColorizer(nn.Module):
    # 1-channel input (gray photo / sketch / infrared) + domain id  ->  3-channel RGB in [-1, 1]
    def __init__(self, n_domains, emb_dim=64, pretrained=True):
        super().__init__()
        r = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None)
        self.stem = nn.Sequential(r.conv1, r.bn1, r.relu)
        self.pool = r.maxpool
        self.l1, self.l2, self.l3, self.l4 = r.layer1, r.layer2, r.layer3, r.layer4
        self.emb = nn.Embedding(n_domains, emb_dim)
        self.film4 = FiLM(emb_dim, 512)
        self.up3 = UpBlock(512, 256, 256, emb_dim)
        self.up2 = UpBlock(256, 128, 128, emb_dim)
        self.up1 = UpBlock(128, 64, 64, emb_dim)
        self.up0 = UpBlock(64, 64, 64, emb_dim)
        self.head = nn.Sequential(nn.Conv2d(64 + 1, 32, 3, padding=1), nn.ReLU(True),
                                  nn.Conv2d(32, 3, 1), nn.Tanh())

    def forward(self, x, domain):
        e = self.emb(domain)
        s0 = self.stem(x.repeat(1, 3, 1, 1))
        s1 = self.l1(self.pool(s0))
        s2 = self.l2(s1)
        s3 = self.l3(s2)
        s4 = self.film4(self.l4(s3), e)
        d = self.up3(s4, s3, e)
        d = self.up2(d, s2, e)
        d = self.up1(d, s1, e)
        d = self.up0(d, s0, e)
        d = F.interpolate(d, size=x.shape[-2:], mode="bilinear", align_corners=False)
        return self.head(torch.cat([d, x], dim=1))

print("Data and model code ready.")

Data and model code ready.


In [32]:
%%ext task5
model = None            # trained model (filled by training, or loaded from Drive in Cell 5)
ACTIVE_DOMAINS = []     # domains the model was trained on


def count_files(folder):
    if not os.path.isdir(folder):
        return 0
    return len([f for f in os.listdir(folder) if f.lower().endswith(EXTS)])


def dataset_status():
    rows = ["| Domain | Colour target images | Matching input images (optional) |", "|---|---|---|"]
    for d in DOMAINS:
        nt = count_files(os.path.join(DATA_ROOT, d, "target"))
        ni = count_files(os.path.join(DATA_ROOT, d, "input"))
        note = "" if nt >= 20 else ("  (need more, 20+ recommended)" if nt > 0 else "  (none yet)")
        rows.append(f"| {DOMAIN_LABELS[d]} | {nt}{note} | {ni} |")
    return "\n".join(rows)


def _save_list(files_, folder):
    os.makedirs(folder, exist_ok=True)
    ok = bad = 0
    for f in (files_ or []):
        path = f if isinstance(f, str) else getattr(f, "name", None)
        if path is None or os.path.splitext(path)[1].lower() not in EXTS:
            bad += 1
            continue
        try:
            with Image.open(path) as im:
                im.verify()
        except Exception:
            bad += 1
            continue
        shutil.copy(path, os.path.join(folder, os.path.basename(path)))
        ok += 1
    return ok, bad


def save_uploads(domain, targets, inputs):
    if not targets and not inputs:
        return "Nothing selected. Choose colour target images (and optionally matching inputs), then click Save.", dataset_status(), None, None
    ok_t, bad_t = _save_list(targets, os.path.join(DATA_ROOT, domain, "target"))
    ok_i, bad_i = _save_list(inputs, os.path.join(DATA_ROOT, domain, "input"))
    msg = f"Saved **{ok_t}** target image(s) and **{ok_i}** input image(s) for **{DOMAIN_LABELS[domain]}**."
    if bad_t or bad_i:
        msg += f" Skipped {bad_t + bad_i} file(s) that were not valid images."
    return msg + " Choose another domain to upload more, or go to the Train Model tab.", dataset_status(), None, None


def train_model(epochs):
    global model, ACTIVE_DOMAINS
    epochs = int(epochs)
    log = []

    def emit(msg):
        log.append(msg)
        return "\n".join(log)

    yield emit("Preparing dataset ...")
    if os.path.exists(CACHE_ROOT):
        shutil.rmtree(CACHE_ROOT)
    items, counts = [], {}
    for d in DOMAINS:
        pairs, bad, n_paired = prepare_domain(d)
        counts[d] = len(pairs)
        yield emit(f"  {d:9s} usable images: {len(pairs):4d}   with real paired input: {n_paired:4d}   skipped: {len(bad)}")
        for name, why in bad[:3]:
            yield emit(f"      skipped {name}: {why}")
        if 0 < len(pairs) < 20:
            yield emit(f"    WARNING: only {len(pairs)} images for {d}; results for this domain will be weak.")
        items += [(t, i, D2I[d]) for t, i in pairs]
    if len(items) < 8:
        raise gr.Error("Fewer than 8 usable colour images in total. Upload more in the first tab.")
    active = [d for d in DOMAINS if counts[d] > 0]
    yield emit("Domains that will be learned: " + ", ".join(active))

    random.shuffle(items)
    n_val = max(2, int(0.1 * len(items)))
    val_items, train_items = items[:n_val], items[n_val:]
    train_ds = DomainDataset(train_items, train=True)
    val_ds = DomainDataset(val_items, train=False)
    # balance the domains: every domain is sampled equally often, even if one has fewer images
    per_dom = {}
    for _, _, did in train_items:
        per_dom[did] = per_dom.get(did, 0) + 1
    weights = [1.0 / per_dom[did] for _, _, did in train_items]
    bs = min(BATCH_SIZE, len(train_ds))
    sampler = WeightedRandomSampler(weights, num_samples=len(train_ds), replacement=True)
    train_dl = DataLoader(train_ds, batch_size=bs, sampler=sampler, num_workers=NUM_WORKERS, drop_last=len(train_ds) > bs)
    val_dl = DataLoader(val_ds, batch_size=bs, shuffle=False, num_workers=NUM_WORKERS)
    yield emit(f"Training images: {len(train_ds)} | validation images: {len(val_ds)} | batch size: {bs}")

    yield emit("Building model (downloads pretrained weights the first time) ...")
    net = DomainColorizer(len(DOMAINS), pretrained=PRETRAINED).to(device)
    use_amp = (device == "cuda")
    opt = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, epochs=epochs, steps_per_epoch=len(train_dl))
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    loss_fn = nn.SmoothL1Loss()
    best_val = float("inf")
    t0 = time.time()
    yield emit(f"Training for {epochs} epochs on {device} ...")

    for epoch in range(epochs):
        net.train()
        tr = 0.0
        for x, y, did in train_dl:
            x, y, did = x.to(device), y.to(device), did.to(device)
            with torch.autocast(device_type=device, enabled=use_amp):
                pred = net(x, did)
            loss = loss_fn(pred.float(), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            tr += loss.item()
        tr /= len(train_dl)

        net.eval()
        va = 0.0
        with torch.no_grad():
            for x, y, did in val_dl:
                x, y, did = x.to(device), y.to(device), did.to(device)
                va += loss_fn(net(x, did).float(), y).item()
        va /= len(val_dl)

        tag = ""
        if va < best_val:
            best_val = va
            torch.save({"model": net.state_dict(), "domains": DOMAINS, "active": active}, CKPT)
            tag = "   <- best so far, saved"
        yield emit(f"epoch {epoch + 1:02d}/{epochs}   train {tr:.4f}   val {va:.4f}   ({time.time() - t0:.0f}s){tag}")

    ck = torch.load(CKPT, map_location=device)
    net.load_state_dict(ck["model"])
    net.eval()

    sums = {d: [0.0, 0] for d in DOMAINS}
    with torch.no_grad():
        for x, y, did in val_dl:
            out = net(x.to(device), did.to(device)).float().cpu()
            err = (out - y).abs().mean(dim=(1, 2, 3))
            for e_, i_ in zip(err, did):
                sums[DOMAINS[int(i_)]][0] += float(e_)
                sums[DOMAINS[int(i_)]][1] += 1
    yield emit("\nColour error per domain on validation images (lower is better):")
    for d in DOMAINS:
        if sums[d][1] > 0:
            yield emit(f"  {d:9s} {sums[d][0] / sums[d][1]:.4f}   ({sums[d][1]} images)")

    model = net
    ACTIVE_DOMAINS = active
    yield emit(f"\nDONE. Best validation loss {best_val:.4f}. Model saved to Drive. Go to the 'Colorize' tab.")

print("Upload and training functions ready.")

Upload and training functions ready.


In [33]:
%%ext task5
def _round32(v):
    return max(64, int(round(v / 32.0)) * 32)


def display_rgb(img):
    # PIL image -> 8-bit RGB for display (handles 16-bit infrared images)
    img = ImageOps.exif_transpose(img)
    if img.mode in ("I;16", "I;16B", "I;16L", "I", "F"):
        return to_uint8_gray(img).convert("RGB")
    return img.convert("RGB")


@torch.no_grad()
def colorize(img, domain, saturation=1.0, line_strength=0.5):
    # returns (colorized PIL image, the pre-processed model input as PIL image), both at the (possibly shrunk) input size
    if model is None:
        raise gr.Error("No trained model yet. Go to the 'Train Model' tab and click Start training first.")
    if domain not in ACTIVE_DOMAINS:
        raise gr.Error(f"The model has no training data for '{domain}'. Available: {', '.join(ACTIVE_DOMAINS)}")
    model.eval()
    img = ImageOps.exif_transpose(img)
    gimg = to_uint8_gray(img)
    if max(gimg.size) > MAX_SIDE:
        s = MAX_SIDE / float(max(gimg.size))
        gimg = gimg.resize((max(1, int(gimg.size[0] * s)), max(1, int(gimg.size[1] * s))), Image.LANCZOS)
    g = normalize_input(domain, np.asarray(gimg))                      # domain-specific preparation, HxW in [0, 1]
    H, W = g.shape

    s = INFER_SIDE / float(max(H, W))
    small = Image.fromarray((g * 255).astype(np.uint8)).resize((_round32(W * s), _round32(H * s)), Image.BICUBIC)
    x = torch.from_numpy(np.asarray(small).astype(np.float32) / 127.5 - 1.0)[None, None].to(device)
    did = torch.tensor([D2I[domain]], device=device)
    out = model(x, did).float().cpu()                                  # 1 x 3 x h x w in [-1, 1]
    out = F.interpolate(out, size=(H, W), mode="bilinear", align_corners=False)[0].permute(1, 2, 0).numpy()
    rgb = np.clip((out + 1.0) / 2.0, 0, 1)

    # domain-specific post-processing
    lab = color.rgb2lab(rgb)
    lab[..., 1:] = lab[..., 1:] * saturation                           # colour strength
    if domain == "photo":                                              # keep the original brightness exactly
        lab[..., 0] = color.rgb2lab(np.stack([g, g, g], axis=-1))[..., 0]
    res = np.clip(color.lab2rgb(lab), 0, 1)
    if domain == "sketch" and line_strength > 0:                       # keep the ink lines of the drawing
        res = res * (1.0 - line_strength * (1.0 - g))[..., None]
    return Image.fromarray((np.clip(res, 0, 1) * 255).astype(np.uint8)), Image.fromarray((g * 255).astype(np.uint8))


# load a previously trained model from Drive, if there is one
if os.path.exists(CKPT):
    try:
        ck = torch.load(CKPT, map_location=device)
        if ck["domains"] == DOMAINS:
            model = DomainColorizer(len(DOMAINS), pretrained=False).to(device)
            model.load_state_dict(ck["model"])
            model.eval()
            ACTIVE_DOMAINS = ck.get("active", DOMAINS)
            print("Loaded saved model from Drive. Domains:", ACTIVE_DOMAINS)
        else:
            print("Saved model uses a different DOMAINS list; please train again in the app.")
    except Exception as e:
        print("Could not load the saved model (", e, "). Please train again in the app.")
else:
    print("No saved model yet. Upload data and train inside the app (next cell).")

Loaded saved model from Drive. Domains: ['photo', 'sketch', 'infrared']


In [34]:
%%ext task5
DOMAIN_CHOICES = [(DOMAIN_LABELS[d], d) for d in DOMAINS]


def refresh_state():
    # shows the model status and only the domains the model was trained on
    global model, ACTIVE_DOMAINS
    if model is None and os.path.exists(CKPT):
        try:
            ck = torch.load(CKPT, map_location=device)
            if ck["domains"] == DOMAINS:
                m = DomainColorizer(len(DOMAINS), pretrained=False).to(device)
                m.load_state_dict(ck["model"])
                m.eval()
                model = m
                ACTIVE_DOMAINS = ck.get("active", DOMAINS)
        except Exception as e:
            print("Could not load saved model:", e)
    if model is None or not ACTIVE_DOMAINS:
        return ("⚠️ **No trained model yet.** Go to tab **2. Train Model** and click **Start training** "
                "(upload colour images in tab 1 first).",
                gr.Radio(choices=DOMAIN_CHOICES, value=DOMAINS[0], label="Domain of the input image"))
    return (f"✅ **Model ready.** Trained domains: {', '.join(ACTIVE_DOMAINS)}",
            gr.Radio(choices=[(DOMAIN_LABELS[d], d) for d in ACTIVE_DOMAINS], value=ACTIVE_DOMAINS[0],
                     label="Domain of the input image"))


def run_colorize(img, domain, saturation, line_strength):
    if img is None:
        raise gr.Error("Please upload an image first.")
    try:
        t0 = time.time()
        res, pre = colorize(img, domain, saturation, line_strength)
        orig = display_rgb(img)
        if orig.size != res.size:
            orig = orig.resize(res.size, Image.LANCZOS)
        return res, orig, res, pre, f"Colorized with the **{DOMAIN_LABELS[domain]}** pipeline in {time.time() - t0:.1f} seconds."
    except gr.Error:
        raise
    except Exception as e:
        traceback.print_exc()
        raise gr.Error(f"{type(e).__name__}: {e}")


HEADER = (
    "<h1 style='text-align:center; margin-bottom:0'>🎨 Cross-Domain Image Colorizer</h1>"
    "<p style='text-align:center; margin-top:4px'><b>Photos</b> &bull; <b>Sketches</b> &bull; <b>Infrared satellite images</b> "
    "&bull; <b>Choose the domain, preview the result</b></p>"
)

def build_ui(demo):
    gr.HTML(HEADER)
    with gr.Tabs():

        # ---------------- Tab 1: upload training data ----------------
        with gr.Tab("1. Upload Training Data"):
            gr.Markdown(
                "Pick a domain, then upload **colour target images** (the colourful result the model should learn). "
                "The black-and-white / sketch / infrared **input is created automatically** from them.\n\n"
                "* **Photo:** colour photographs of any scenes.\n"
                "* **Sketch:** colourful drawings, illustrations, cartoons or photos (pencil sketches are generated automatically).\n"
                "* **Infrared:** true-colour satellite or aerial images (infrared is simulated). "
                "**For much better results**, also upload the *real* infrared images in the second box, "
                "with the **same file names** as the colour images (e.g. tile_01.jpg and tile_01.png).\n\n"
                "The same applies to real hand-drawn sketches. Aim for 100+ images per domain (20 minimum).")
            with gr.Row():
                with gr.Column():
                    up_domain = gr.Radio(choices=DOMAIN_CHOICES, value=DOMAINS[0], label="Domain of these images")
                    up_t = gr.File(label="Colour target images (required)", file_count="multiple", file_types=["image"])
                    up_i = gr.File(label="Matching input images (optional, same file names)", file_count="multiple", file_types=["image"])
                    up_btn = gr.Button("Save", variant="primary")
                    up_msg = gr.Markdown()
                with gr.Column():
                    ds_md = gr.Markdown(dataset_status())
                    ds_btn = gr.Button("Refresh counts")
            up_btn.click(save_uploads, inputs=[up_domain, up_t, up_i], outputs=[up_msg, ds_md, up_t, up_i])
            ds_btn.click(dataset_status, outputs=ds_md)

        # ---------------- Tab 2: train ----------------
        with gr.Tab("2. Train Model"):
            gr.Markdown("Click **Start training** after uploading data. It takes roughly 5-20 minutes on a T4 GPU. "
                        "For a quick test, use 8 epochs. The best model is saved to Google Drive automatically.")
            epochs_sl = gr.Slider(5, 60, value=25, step=1, label="Epochs")
            train_btn = gr.Button("Start training", variant="primary")
            log_box = gr.Textbox(label="Training log", lines=16, max_lines=16, interactive=False)

        # ---------------- Tab 3: colorize ----------------
        with gr.Tab("3. Colorize"):
            model_md = gr.Markdown()
            with gr.Row():
                with gr.Column(scale=1):
                    inp = gr.Image(type="pil", label="Upload input image (B&W photo, sketch or infrared)", height=380)
                    domain_dd = gr.Radio(choices=DOMAIN_CHOICES, value=DOMAINS[0], label="Domain of the input image")
                    sat = gr.Slider(0.5, 2.0, value=1.2, step=0.1, label="Colour strength")
                    line_sl = gr.Slider(0.0, 1.0, value=0.5, step=0.05, label="Ink line strength (sketch only)")
                    btn = gr.Button("🎨 Colorize", variant="primary")
                    refresh_btn = gr.Button("Refresh model status")
                with gr.Column(scale=2):
                    status = gr.Markdown("Upload an image, choose its domain and click **Colorize**.")
                    with gr.Tabs():
                        with gr.Tab("🌈 Colorized Result"):
                            out_main = gr.Image(type="pil", label="Colorized", height=520)
                        with gr.Tab("Side by Side Comparison"):
                            gr.Markdown("<p style='text-align:center'><b>Original input (left) | Colorized (right)</b></p>")
                            with gr.Row():
                                out_orig = gr.Image(type="pil", label="Original input", height=480)
                                out_col = gr.Image(type="pil", label="Colorized", height=480)
                        with gr.Tab("Model Input (after domain pre-processing)"):
                            gr.Markdown("What the model actually sees: sketches are auto-inverted if needed, "
                                        "infrared images are contrast-stretched, photos are used as they are.")
                            out_pre = gr.Image(type="pil", label="Pre-processed input", height=480)

    train_btn.click(train_model, inputs=epochs_sl, outputs=log_box).then(refresh_state, None, [model_md, domain_dd])
    refresh_btn.click(refresh_state, None, [model_md, domain_dd])
    btn.click(run_colorize, inputs=[inp, domain_dd, sat, line_sl], outputs=[out_main, out_orig, out_col, out_pre, status])
    demo.load(refresh_state, None, [model_md, domain_dd])

In [35]:
%%ext task6
import os, io, glob, time, shutil, random
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms as T
from PIL import Image, ImageOps
from skimage import color
import gradio as gr

# ------------------------- Google Drive -------------------------
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = "/content/drive/MyDrive/historical_colors"      # photos + trained model are saved here
except Exception as e:
    print("Could not mount Google Drive (", e, ")\nUsing temporary Colab storage instead - files will be lost when the session ends.")
    DATA_ROOT = "/content/historical_colors"

# ------------------------- settings -------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
PERIODS = ["1900s-1910s", "1920s", "1930s", "WWII-1940s", "1950s"]
P2I = {p: i for i, p in enumerate(PERIODS)}

IMG_SIZE = 256        # training size (divisible by 32)
INFER_SIDE = 256      # longer side used by the model when colorizing
MAX_SIDE = 2048       # very large uploads are shrunk to this
MIN_SAT = 0.03        # photos with lower average saturation are treated as black & white and skipped in training
BATCH_SIZE = 16
LR = 3e-4
NUM_WORKERS = 2
PRETRAINED = True     # ImageNet weights for the encoder (downloaded automatically)

EXTS = (".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp", ".webp")
CACHE_ROOT = "/content/hc_cache"                       # resized copies used for fast training (temporary)
CKPT = os.path.join(DATA_ROOT, "colorizer.pt")         # trained model

for p in PERIODS:
    os.makedirs(os.path.join(DATA_ROOT, p), exist_ok=True)
random.seed(0); np.random.seed(0); torch.manual_seed(0)
print("Device:", device)
print("Data folder:", DATA_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
Data folder: /content/drive/MyDrive/historical_colors


In [36]:
%%ext task6
def prepare_period(p):
    # reads photos of one period from Drive, skips bad/black&white ones, saves resized copies for fast training
    out_dir = os.path.join(CACHE_ROOT, p)
    os.makedirs(out_dir, exist_ok=True)
    good, bad = [], []
    for f in sorted(glob.glob(os.path.join(DATA_ROOT, p, "*"))):
        if not f.lower().endswith(EXTS):
            continue
        try:
            with Image.open(f) as im:
                try:
                    im.draft("RGB", (1024, 1024))
                except Exception:
                    pass
                im = ImageOps.exif_transpose(im).convert("RGB")
            w, h = im.size
            if min(w, h) < 64:
                bad.append((os.path.basename(f), "too small"))
                continue
            sat = np.asarray(im.resize((64, 64)).convert("HSV"))[..., 1].mean() / 255.0
            if sat < MIN_SAT:
                bad.append((os.path.basename(f), "looks black & white"))
                continue
            s = 320.0 / min(w, h)
            if s < 1.0:
                im = im.resize((max(1, int(round(w * s))), max(1, int(round(h * s)))), Image.LANCZOS)
            dst = os.path.join(out_dir, f"{p}_{len(good):05d}.jpg")
            im.save(dst, quality=95)
            good.append(dst)
        except Exception as e:
            bad.append((os.path.basename(f), f"unreadable: {e}"))
    return good, bad


class PeriodDataset(Dataset):
    def __init__(self, items, train=True):
        self.items = items
        if train:
            self.tf = T.Compose([T.RandomResizedCrop(IMG_SIZE, scale=(0.4, 1.0)), T.RandomHorizontalFlip()])
        else:
            self.tf = T.Compose([T.Resize(IMG_SIZE), T.CenterCrop(IMG_SIZE)])

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        path, pid = self.items[i]
        with Image.open(path) as im:
            img = self.tf(im.convert("RGB"))
        lab = color.rgb2lab(np.asarray(img)).astype("float32")
        L = torch.from_numpy(lab[..., 0:1] / 50.0 - 1.0).permute(2, 0, 1)     # lightness -> [-1, 1]
        ab = torch.from_numpy(lab[..., 1:] / 128.0).permute(2, 0, 1)          # colour    -> ~[-1, 1]
        return L, ab, pid


class FiLM(nn.Module):
    # scales and shifts feature maps depending on the selected period
    def __init__(self, emb_dim, ch):
        super().__init__()
        self.fc = nn.Linear(emb_dim, ch * 2)
        nn.init.zeros_(self.fc.weight)
        nn.init.zeros_(self.fc.bias)

    def forward(self, x, e):
        g, b = self.fc(e).chunk(2, dim=1)
        return x * (1 + g[:, :, None, None]) + b[:, :, None, None]


class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, emb_dim):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch + skip_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(True))
        self.film = FiLM(emb_dim, out_ch)

    def forward(self, x, skip, e):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = self.conv(torch.cat([x, skip], dim=1))
        return self.film(x, e)


class PeriodColorizer(nn.Module):
    def __init__(self, n_periods, emb_dim=64, pretrained=True):
        super().__init__()
        r = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None)
        self.stem = nn.Sequential(r.conv1, r.bn1, r.relu)
        self.pool = r.maxpool
        self.l1, self.l2, self.l3, self.l4 = r.layer1, r.layer2, r.layer3, r.layer4
        self.emb = nn.Embedding(n_periods, emb_dim)
        self.film4 = FiLM(emb_dim, 512)
        self.up3 = UpBlock(512, 256, 256, emb_dim)
        self.up2 = UpBlock(256, 128, 128, emb_dim)
        self.up1 = UpBlock(128, 64, 64, emb_dim)
        self.up0 = UpBlock(64, 64, 64, emb_dim)
        self.head = nn.Sequential(nn.Conv2d(64, 32, 3, padding=1), nn.ReLU(True),
                                  nn.Conv2d(32, 2, 1), nn.Tanh())

    def forward(self, L, period):
        e = self.emb(period)
        x = L.repeat(1, 3, 1, 1)
        s0 = self.stem(x)
        s1 = self.l1(self.pool(s0))
        s2 = self.l2(s1)
        s3 = self.l3(s2)
        s4 = self.film4(self.l4(s3), e)
        d = self.up3(s4, s3, e)
        d = self.up2(d, s2, e)
        d = self.up1(d, s1, e)
        d = self.up0(d, s0, e)
        d = F.interpolate(d, size=L.shape[-2:], mode="bilinear", align_corners=False)
        return self.head(d)

print("Dataset and model code ready.")

Dataset and model code ready.


In [37]:
%%ext task6
def prepare_period(p):
    # reads photos of one period from Drive, skips bad/black&white ones, saves resized copies for fast training
    out_dir = os.path.join(CACHE_ROOT, p)
    os.makedirs(out_dir, exist_ok=True)
    good, bad = [], []
    for f in sorted(glob.glob(os.path.join(DATA_ROOT, p, "*"))):
        if not f.lower().endswith(EXTS):
            continue
        try:
            with Image.open(f) as im:
                try:
                    im.draft("RGB", (1024, 1024))
                except Exception:
                    pass
                im = ImageOps.exif_transpose(im).convert("RGB")
            w, h = im.size
            if min(w, h) < 64:
                bad.append((os.path.basename(f), "too small"))
                continue
            sat = np.asarray(im.resize((64, 64)).convert("HSV"))[..., 1].mean() / 255.0
            if sat < MIN_SAT:
                bad.append((os.path.basename(f), "looks black & white"))
                continue
            s = 320.0 / min(w, h)
            if s < 1.0:
                im = im.resize((max(1, int(round(w * s))), max(1, int(round(h * s)))), Image.LANCZOS)
            dst = os.path.join(out_dir, f"{p}_{len(good):05d}.jpg")
            im.save(dst, quality=95)
            good.append(dst)
        except Exception as e:
            bad.append((os.path.basename(f), f"unreadable: {e}"))
    return good, bad


class PeriodDataset(Dataset):
    def __init__(self, items, train=True):
        self.items = items
        if train:
            self.tf = T.Compose([T.RandomResizedCrop(IMG_SIZE, scale=(0.4, 1.0)), T.RandomHorizontalFlip()])
        else:
            self.tf = T.Compose([T.Resize(IMG_SIZE), T.CenterCrop(IMG_SIZE)])

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        path, pid = self.items[i]
        with Image.open(path) as im:
            img = self.tf(im.convert("RGB"))
        lab = color.rgb2lab(np.asarray(img)).astype("float32")
        L = torch.from_numpy(lab[..., 0:1] / 50.0 - 1.0).permute(2, 0, 1)     # lightness -> [-1, 1]
        ab = torch.from_numpy(lab[..., 1:] / 128.0).permute(2, 0, 1)          # colour    -> ~[-1, 1]
        return L, ab, pid


class FiLM(nn.Module):
    # scales and shifts feature maps depending on the selected period
    def __init__(self, emb_dim, ch):
        super().__init__()
        self.fc = nn.Linear(emb_dim, ch * 2)
        nn.init.zeros_(self.fc.weight)
        nn.init.zeros_(self.fc.bias)

    def forward(self, x, e):
        g, b = self.fc(e).chunk(2, dim=1)
        return x * (1 + g[:, :, None, None]) + b[:, :, None, None]


class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, emb_dim):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch + skip_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(True))
        self.film = FiLM(emb_dim, out_ch)

    def forward(self, x, skip, e):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = self.conv(torch.cat([x, skip], dim=1))
        return self.film(x, e)


class PeriodColorizer(nn.Module):
    def __init__(self, n_periods, emb_dim=64, pretrained=True):
        super().__init__()
        r = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None)
        self.stem = nn.Sequential(r.conv1, r.bn1, r.relu)
        self.pool = r.maxpool
        self.l1, self.l2, self.l3, self.l4 = r.layer1, r.layer2, r.layer3, r.layer4
        self.emb = nn.Embedding(n_periods, emb_dim)
        self.film4 = FiLM(emb_dim, 512)
        self.up3 = UpBlock(512, 256, 256, emb_dim)
        self.up2 = UpBlock(256, 128, 128, emb_dim)
        self.up1 = UpBlock(128, 64, 64, emb_dim)
        self.up0 = UpBlock(64, 64, 64, emb_dim)
        self.head = nn.Sequential(nn.Conv2d(64, 32, 3, padding=1), nn.ReLU(True),
                                  nn.Conv2d(32, 2, 1), nn.Tanh())

    def forward(self, L, period):
        e = self.emb(period)
        x = L.repeat(1, 3, 1, 1)
        s0 = self.stem(x)
        s1 = self.l1(self.pool(s0))
        s2 = self.l2(s1)
        s3 = self.l3(s2)
        s4 = self.film4(self.l4(s3), e)
        d = self.up3(s4, s3, e)
        d = self.up2(d, s2, e)
        d = self.up1(d, s1, e)
        d = self.up0(d, s0, e)
        d = F.interpolate(d, size=L.shape[-2:], mode="bilinear", align_corners=False)
        return self.head(d)

print("Dataset and model code ready.")

Dataset and model code ready.


In [38]:
%%ext task6
model = None            # the trained model (filled by training, or loaded from Drive in Cell 5)
ACTIVE_PERIODS = []     # periods the model was trained on


def count_photos(p):
    folder = os.path.join(DATA_ROOT, p)
    if not os.path.isdir(folder):
        return 0
    return len([f for f in os.listdir(folder) if f.lower().endswith(EXTS)])


def dataset_status():
    rows = ["| Period | Photos saved in Drive |", "|---|---|"]
    for p in PERIODS:
        n = count_photos(p)
        flag = "" if n >= 20 else ("  (need more, 20+ recommended)" if n > 0 else "  (none yet)")
        rows.append(f"| {p} | {n}{flag} |")
    return "\n".join(rows)


def save_uploads(period, uploaded):
    if not uploaded:
        return "No photos selected. Click the upload box, choose photos, then click Save.", dataset_status(), None
    folder = os.path.join(DATA_ROOT, period)
    os.makedirs(folder, exist_ok=True)
    ok = bad = 0
    stamp = int(time.time())
    for i, f in enumerate(uploaded):
        path = f if isinstance(f, str) else getattr(f, "name", None)
        if path is None or os.path.splitext(path)[1].lower() not in EXTS:
            bad += 1
            continue
        try:
            with Image.open(path) as im:
                im.verify()
        except Exception:
            bad += 1
            continue
        shutil.copy(path, os.path.join(folder, f"up_{stamp}_{i:04d}{os.path.splitext(path)[1].lower()}"))
        ok += 1
    msg = f"Saved **{ok}** photo(s) to **{period}**." + (f" Skipped {bad} (not valid images)." if bad else "")
    return msg + " Choose another period to upload more, or go to the Train Model tab.", dataset_status(), None


def train_model(epochs):
    global model, ACTIVE_PERIODS
    epochs = int(epochs)
    log = []

    def emit(msg):
        log.append(msg)
        return "\n".join(log)

    yield emit("Preparing dataset ...")
    if os.path.exists(CACHE_ROOT):
        shutil.rmtree(CACHE_ROOT)
    items, counts = [], {}
    for p in PERIODS:
        good, bad = prepare_period(p)
        counts[p] = len(good)
        yield emit(f"  {p:12s} usable photos: {len(good):4d}   skipped: {len(bad)}")
        if 0 < len(good) < 20:
            yield emit(f"    WARNING: only {len(good)} photos for {p}; results for this period will be weak.")
        items += [(f, P2I[p]) for f in good]
    if len(items) < 8:
        raise gr.Error("Fewer than 8 usable colour photos in total. Upload more colour photos in the first tab.")
    active = [p for p in PERIODS if counts[p] > 0]
    yield emit("Periods that will be learned: " + ", ".join(active))

    random.shuffle(items)
    n_val = max(2, int(0.1 * len(items)))
    train_ds = PeriodDataset(items[n_val:], train=True)
    val_ds = PeriodDataset(items[:n_val], train=False)
    bs = min(BATCH_SIZE, len(train_ds))
    train_dl = DataLoader(train_ds, batch_size=bs, shuffle=True, num_workers=NUM_WORKERS, drop_last=len(train_ds) > bs)
    val_dl = DataLoader(val_ds, batch_size=bs, shuffle=False, num_workers=NUM_WORKERS)
    yield emit(f"Training photos: {len(train_ds)} | validation photos: {len(val_ds)} | batch size: {bs}")

    yield emit("Building model (downloads pretrained weights the first time) ...")
    net = PeriodColorizer(len(PERIODS), pretrained=PRETRAINED).to(device)
    use_amp = (device == "cuda")
    opt = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, epochs=epochs, steps_per_epoch=len(train_dl))
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    loss_fn = nn.SmoothL1Loss()
    best_val = float("inf")
    t0 = time.time()
    yield emit(f"Training for {epochs} epochs on {device} ...")

    for epoch in range(epochs):
        net.train()
        tr = 0.0
        for L, ab, pid in train_dl:
            L, ab, pid = L.to(device), ab.to(device), pid.to(device)
            with torch.autocast(device_type=device, enabled=use_amp):
                pred = net(L, pid)
            loss = loss_fn(pred.float(), ab)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            tr += loss.item()
        tr /= len(train_dl)

        net.eval()
        va = 0.0
        with torch.no_grad():
            for L, ab, pid in val_dl:
                L, ab, pid = L.to(device), ab.to(device), pid.to(device)
                va += loss_fn(net(L, pid).float(), ab).item()
        va /= len(val_dl)

        tag = ""
        if va < best_val:
            best_val = va
            torch.save({"model": net.state_dict(), "periods": PERIODS, "active": active}, CKPT)
            tag = "   <- best so far, saved"
        yield emit(f"epoch {epoch + 1:02d}/{epochs}   train {tr:.4f}   val {va:.4f}   ({time.time() - t0:.0f}s){tag}")

    ck = torch.load(CKPT, map_location=device)
    net.load_state_dict(ck["model"])
    net.eval()

    # validation colour error per period (lower = closer to that period's real colours)
    sums = {p: [0.0, 0] for p in PERIODS}
    with torch.no_grad():
        for L, ab, pid in val_dl:
            out = net(L.to(device), pid.to(device)).float().cpu()
            err = (out - ab).abs().mean(dim=(1, 2, 3))
            for e_, i_ in zip(err, pid):
                sums[PERIODS[int(i_)]][0] += float(e_)
                sums[PERIODS[int(i_)]][1] += 1
    yield emit("\nColour error per period on validation photos (lower is better):")
    for p in PERIODS:
        if sums[p][1] > 0:
            yield emit(f"  {p:12s} {sums[p][0] / sums[p][1]:.4f}   ({sums[p][1]} photos)")

    model = net
    ACTIVE_PERIODS = active
    yield emit(f"\nDONE. Best validation loss {best_val:.4f}. Model saved to Drive. Go to the 'Colorize' tab.")

print("Upload and training functions ready.")

Upload and training functions ready.


In [39]:
%%ext task6
def _round32(v):
    return max(64, int(round(v / 32.0)) * 32)


@torch.no_grad()
def colorize(img, period, saturation=1.0):
    # img: PIL image (black & white or colour). Returns a colorized PIL image at the original size.
    if model is None:
        raise gr.Error("No trained model yet. Go to the 'Train Model' tab and click Start training first.")
    if period not in ACTIVE_PERIODS:
        raise gr.Error(f"The model has no training photos for '{period}'. Choose one of: {', '.join(ACTIVE_PERIODS)}")
    model.eval()
    img = ImageOps.exif_transpose(img).convert("RGB")
    W, H = img.size
    if max(W, H) > MAX_SIDE:
        s = MAX_SIDE / max(W, H)
        W, H = max(1, int(W * s)), max(1, int(H * s))
        img = img.resize((W, H), Image.LANCZOS)
    L_full = color.rgb2lab(np.asarray(img))[..., 0]                    # lightness at full resolution (0..100)
    s = INFER_SIDE / float(max(W, H))
    small = img.resize((_round32(W * s), _round32(H * s)), Image.BICUBIC)
    L = color.rgb2lab(np.asarray(small))[..., 0] / 50.0 - 1.0
    x = torch.from_numpy(L).float()[None, None].to(device)
    pid = torch.tensor([P2I[period]], device=device)
    ab = model(x, pid).float().cpu()
    ab = F.interpolate(ab, size=(H, W), mode="bilinear", align_corners=False)[0]
    ab = (ab.permute(1, 2, 0).numpy() * 128.0 * saturation).clip(-128, 127)
    lab = np.concatenate([L_full[..., None], ab], axis=-1)
    rgb = (color.lab2rgb(lab).clip(0, 1) * 255).astype(np.uint8)
    return Image.fromarray(rgb)


# load a previously trained model from Drive, if there is one
if os.path.exists(CKPT):
    try:
        ck = torch.load(CKPT, map_location=device)
        if ck["periods"] == PERIODS:
            model = PeriodColorizer(len(PERIODS), pretrained=False).to(device)
            model.load_state_dict(ck["model"])
            model.eval()
            ACTIVE_PERIODS = ck.get("active", PERIODS)
            print("Loaded saved model from Drive. Periods:", ACTIVE_PERIODS)
        else:
            print("Saved model uses a different PERIODS list; please train again in the app.")
    except Exception as e:
        print("Could not load the saved model (", e, "). Please train again in the app.")
else:
    print("No saved model yet. Upload photos and train inside the app (next cell).")

Loaded saved model from Drive. Periods: ['1900s-1910s', '1920s', '1930s', 'WWII-1940s', '1950s']


In [40]:
%%ext task6
def run_colorize(img, period, saturation):
    if img is None:
        raise gr.Error("Please upload a photo first.")
    t0 = time.time()
    res = colorize(img, period, saturation)
    orig = ImageOps.exif_transpose(img).convert("RGB")
    if orig.size != res.size:
        orig = orig.resize(res.size, Image.LANCZOS)
    return res, orig, res, f"Colorized with the **{period}** palette in {time.time() - t0:.1f} seconds."


def compare_all(img, saturation):
    if img is None:
        raise gr.Error("Please upload a photo first.")
    if not ACTIVE_PERIODS:
        raise gr.Error("No trained model yet. Go to the 'Train Model' tab first.")
    return [(colorize(img, p, saturation), p) for p in ACTIVE_PERIODS]


HEADER = (
    "<h1 style='text-align:center; margin-bottom:0'>🎨 Historical Photo Colorizer - Period-Aware Pipeline</h1>"
    "<p style='text-align:center; margin-top:4px'><b>Upload photos</b> &bull; <b>Train on real period colours</b> &bull; "
    "<b>Choose the era</b> &bull; <b>Compare all periods</b></p>"
)

def build_ui(demo):
    gr.HTML(HEADER)
    with gr.Tabs():

        # ---------------- Tab 1: upload training photos ----------------
        with gr.Tab("1. Upload Training Photos"):
            gr.Markdown("Upload **real colour photos** from one era at a time (autochromes, Kodachrome, colour scans). "
                        "Select the period, choose the photos, click **Save photos**. Repeat for each period. "
                        "Aim for 100+ photos per period (20 minimum).")
            with gr.Row():
                with gr.Column():
                    up_period = gr.Radio(choices=PERIODS, value=PERIODS[0], label="🕰 Period of these photos")
                    up_files = gr.File(label="Choose colour photos", file_count="multiple", file_types=["image"])
                    up_btn = gr.Button("Save photos", variant="primary")
                    up_msg = gr.Markdown()
                with gr.Column():
                    ds_md = gr.Markdown(dataset_status())
                    ds_btn = gr.Button("Refresh counts")
            up_btn.click(save_uploads, inputs=[up_period, up_files], outputs=[up_msg, ds_md, up_files])
            ds_btn.click(dataset_status, outputs=ds_md)

        # ---------------- Tab 2: train ----------------
        with gr.Tab("2. Train Model"):
            gr.Markdown("Click **Start training** after uploading photos. It takes roughly 5-20 minutes on a T4 GPU. "
                        "For a quick test, use 8 epochs. The best model is saved to Google Drive automatically.")
            epochs_sl = gr.Slider(5, 60, value=25, step=1, label="Epochs")
            train_btn = gr.Button("Start training", variant="primary")
            log_box = gr.Textbox(label="Training log", lines=16, max_lines=16, interactive=False)
            train_btn.click(train_model, inputs=epochs_sl, outputs=log_box)

        # ---------------- Tab 3: colorize ----------------
        with gr.Tab("3. Colorize"):
            with gr.Row():
                with gr.Column(scale=1):
                    inp = gr.Image(type="pil", label="📷 Upload B&W Photo", height=380)
                    period_dd = gr.Radio(choices=PERIODS, value="WWII-1940s", label="🕰 Historical Period")
                    sat = gr.Slider(0.5, 2.0, value=1.2, step=0.1, label="Colour strength")
                    btn = gr.Button("🎨 Colorize", variant="primary")
                    btn_all = gr.Button("Compare all periods")
                with gr.Column(scale=2):
                    status = gr.Markdown("Upload a photo, choose a period and click **Colorize**.")
                    with gr.Tabs():
                        with gr.Tab("🌈 Colorized Result"):
                            out_main = gr.Image(type="pil", label="Colorized", height=520)
                        with gr.Tab("Side by Side Comparison"):
                            gr.Markdown("<p style='text-align:center'><b>Original (left) | Colorized (right)</b></p>")
                            with gr.Row():
                                out_orig = gr.Image(type="pil", label="Original", height=480)
                                out_col = gr.Image(type="pil", label="Colorized", height=480)
                        with gr.Tab("Compare All Periods"):
                            gallery = gr.Gallery(label="Same photo in every period", columns=3)
            btn.click(run_colorize, inputs=[inp, period_dd, sat], outputs=[out_main, out_orig, out_col, status])
            btn_all.click(compare_all, inputs=[inp, sat], outputs=gallery)

In [41]:
LAB_METHODS = [
    ("DeOldify Ensemble (main project)", "main_ens"),
    ("DeOldify Artistic (main project)", "main_art"),
    ("DeOldify Stable (main project)", "main_stb"),
    ("Task 1: object-segmentation colours", "t1"),
    ("Task 2: conditional, automatic (no hints)", "t2"),
    ("Task 3: context-aware scene", "t3"),
    ("Task 4: time-based era (auto-detected)", "t4"),
    ("Task 5: cross-domain (photo)", "t5"),
    ("Task 6: period-aware (chosen period)", "t6"),
]
LAB_LABELS = {v: k for k, v in LAB_METHODS}
LAB_HEADERS = ["Method", "Colorfulness", "Saturation", "SSIM", "Edge Preservation", "Skin Naturalness", "Time (s)"]


def _mod(key):
    if key not in MODULES:
        raise RuntimeError("the extension module is not loaded (run its cells first)")
    return MODULES[key]


def _lab_deoldify(mode):
    def run(img_np, pil, period):
        out, _orig, msg = colorize_image(img_np, mode, 35, 0.0, 1.15, 0.15, True, True, 0.45)
        if out is None:
            raise RuntimeError(msg.splitlines()[-1] if msg else "DeOldify failed")
        return out, ""
    return run


def _lab_t1(img_np, pil, period):
    ns = _mod("task1")
    res, _seg, _cov = ns["process_image"](pil, ns["DEFAULT_MODEL"], 512, 1080, 1.2, 0.7, *ns["SCHEMES"]["Natural"])
    return res, "Natural scheme"


def _lab_t2(img_np, pil, period):
    return _mod("task2")["colorize_with_hints"](pil, [], 1.2, 0.7, 0.7), "no hints"


def _lab_t3(img_np, pil, period):
    ns = _mod("task3")
    return ns["colorize"](pil, ns["AUTO"], 1.2, 0.7), "auto scene"


def _lab_t4(img_np, pil, period):
    ns = _mod("task4")
    probs = ns["detect_era"](pil)
    top = max(probs, key=probs.get)
    return ns["colorize"](pil, top, 1.2), f"era {top} ({probs[top] * 100:.0f}%)"


def _lab_t5(img_np, pil, period):
    res, _pre = _mod("task5")["colorize"](pil, "photo", 1.2, 0.5)
    return res, "photo domain"


def _lab_t6(img_np, pil, period):
    return _mod("task6")["colorize"](pil, period, 1.2), f"period {period}"


LAB_FUNCS = {
    "main_ens": _lab_deoldify("Ensemble (Best Quality)"),
    "main_art": _lab_deoldify("Artistic (Portraits & Faces)"),
    "main_stb": _lab_deoldify("Stable (Objects & Architecture)"),
    "t1": _lab_t1, "t2": _lab_t2, "t3": _lab_t3, "t4": _lab_t4, "t5": _lab_t5, "t6": _lab_t6,
}


def run_lab(img_np, selected, period):
    if img_np is None:
        raise gr.Error("Please upload a photo first.")
    if not selected:
        raise gr.Error("Tick at least one method.")
    arr = np.asarray(img_np)
    if arr.ndim == 2:
        arr = np.stack([arr] * 3, axis=-1)
    arr = np.ascontiguousarray(arr[..., :3]).astype("uint8")
    H, W = arr.shape[:2]
    pil = Image.fromarray(arr)
    gallery, rows, notes = [(arr, "Original")], [], []
    for key in selected:
        label = LAB_LABELS[key]
        t0 = time.time()
        try:
            out, extra = LAB_FUNCS[key](arr, pil, period)
            out = np.asarray(out)
            if out.shape[:2] != (H, W):
                out = np.asarray(Image.fromarray(out).resize((W, H), Image.LANCZOS))
            rep = ColorQualityMetrics.full_report(arr, out)
            rows.append([label, rep["Colorfulness"], rep["Saturation"], rep["SSIM"], rep["Edge Preservation"],
                         rep["Skin Naturalness"], round(time.time() - t0, 1)])
            gallery.append((out, label + (f" ({extra})" if extra else "")))
        except Exception as e:
            first = (str(e).strip().splitlines() or [type(e).__name__])[0]
            notes.append(f"- **{label}**: skipped ({first})")
    note = "\n".join(notes) if notes else "All selected methods finished."
    return gallery, rows, note


def build_lab_ui():
    gr.Markdown(
        "### 🧪 Experiment Lab\n"
        "Run one photo through several methods and compare them side by side. Every result is scored with the main project's "
        "**ColorQualityMetrics**. Methods whose model is not trained yet are skipped with a note. "
        "Metrics are heuristics (higher colorfulness = more colourful, higher SSIM and edge preservation = structure kept), "
        "best used to compare methods on the same photo, not as a ground-truth score."
    )
    with gr.Row():
        with gr.Column(scale=1):
            lab_in = gr.Image(type="numpy", label="🖼 Upload B&W Photo", height=300)
            lab_sel = gr.CheckboxGroup(choices=LAB_METHODS, value=["main_ens", "t1"], label="Methods to compare")
            periods = MODULES["task6"]["PERIODS"] if "task6" in MODULES and "PERIODS" in MODULES["task6"] else ["1950s"]
            lab_period = gr.Dropdown(choices=periods, value=("WWII-1940s" if "WWII-1940s" in periods else periods[0]),
                                     label="Period used by the Task 6 method")
            lab_btn = gr.Button("🧪 Run experiment", variant="primary")
        with gr.Column(scale=2):
            lab_msg = gr.Markdown("Upload a photo, tick the methods, and click **Run experiment**.")
            lab_gal = gr.Gallery(label="Results", columns=3, height=420)
            lab_tbl = gr.Dataframe(headers=LAB_HEADERS, label="Quality metrics", interactive=False)
    lab_btn.click(run_lab, [lab_in, lab_sel, lab_period], [lab_gal, lab_tbl, lab_msg])

print("Experiment Lab ready.")

Experiment Lab ready.


In [ ]:
EXT_TABS = [
    ("🎞 Task 1: Real-time multi-object", "task1"),
    ("🖌 Task 2: Conditional (colour hints)", "task2"),
    ("🏙 Task 3: Context-aware scenes", "task3"),
    ("🕰 Task 4: Time-based era", "task4"),
    ("🛰 Task 5: Cross-domain", "task5"),
    ("📜 Task 6: Period-aware", "task6"),
]

with gr.Blocks(title="DeOldify Pro + Internship Tasks", css=CSS) as interface:
    gr.Markdown("## 🧩 AI Colorization Project: main pipeline + internship tasks as extra features")
    with gr.Tabs():
        with gr.Tab("🎨 DeOldify Pro (main project)"):
            build_main_ui()
        with gr.Tab("🧪 Experiment Lab"):
            build_lab_ui()
        for tab_title, key in EXT_TABS:
            with gr.Tab(tab_title):
                MODULES[key]["build_ui"](interface)

interface.queue().launch(share=True, inline=True, show_error=True, debug=True)

INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-initiated-analytics "HTTP/1.1 200 OK"


HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-initiated-analytics "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"


HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: GET http://127.0.0.1:7860/gradio_api/startup-events "HTTP/1.1 200 OK"


HTTP Request: GET http://127.0.0.1:7860/gradio_api/startup-events "HTTP/1.1 200 OK"


... (6 outputs left out)

INFO:httpx:HTTP Request: HEAD https://614cc31272883ad49b.gradio.live "HTTP/1.1 200 OK"


HTTP Request: HEAD https://614cc31272883ad49b.gradio.live "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-launched-telemetry "HTTP/1.1 200 OK"


HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-launched-telemetry "HTTP/1.1 200 OK"
